# MedMNIST v2 ResNet baselines — independent replication (ReScience)

**One self-contained notebook** that produces every result and figure for the paper.
It writes our `src/` package to disk from embedded cells (no GitHub clone needed),
then runs one of three modes set in the **CONFIG** cell:

- `MODE="baselines"` — train the tiered run matrix (resumable across Kaggle's ~9h cap).
- `MODE="extensions"` — per-class analysis, bias mitigation, lightweight variant (DermaMNIST).
- `MODE="report"` — aggregate all runs → comparison table + every figure.

**Scope (compute-minimal slice of Table 3).** The cheapest route to a *solid*
partial replication is breadth at 28×28, not depth at 224. So we replicate
**ResNet-18 @ 28 across all twelve MedMNIST2D datasets** and keep the **full
four-config matrix (R18/R50 × 28/224) on DermaMNIST**:

- **Tier 1 — DermaMNIST (primary):** ResNet-18 and ResNet-50, sizes 28 and 224, seeds 0/1/2. 12 runs.
- **Tier 2 — small/medium datasets, R18 @ 28, seeds 0/1/2:** Retina, Breast, Pneumonia, Blood, OrganA, OrganC, OrganS. 21 runs.
- **Tier 3 — large datasets, R18 @ 28, seed 0 only:** Tissue, OCT, Path, Chest. 4 runs. Single-seed for compute reasons (stable at that data scale); disclosed in the writeup.

**~37 runs total ≈ 12–15 GPU-hours on a P100.** Tier 3's four large datasets at
a single seed dominate the cost; drop one or two if you're tight and note it.

**Hard rule:** nothing here is copied from `MedMNIST/experiments`. We use the
`medmnist` PyPI package only for (a) the standardized dataset loaders and
(b) `medmnist.Evaluator` as a metric oracle. Everything else is our own code.

### Kaggle workflow (because ~37 runs won't fit in one 9h session)
1. Turn **Internet ON** (Settings) so the datasets can download from Zenodo, and **GPU** on (P100).
2. Run `MODE="baselines"` with a `TIERS`/`MAX_MINUTES` you can finish. **Save Version** (commit).
3. Next session: **Add data → your previous notebook output**, set `PREV_RESULTS` to it, run again — finished runs skip, partial runs resume from `last.pth`.
4. When all needed runs exist: run `MODE="extensions"` once, then `MODE="report"` once.

## 1. Install deps + write the `src/` package

In [ ]:
# medmnist is the only missing dep on Kaggle (torch/torchvision/sklearn/matplotlib preinstalled).
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "medmnist==3.0.2"], check=False)


In [ ]:
SRC_B64 = {
    '__init__.py': 'IiIiSW5kZXBlbmRlbnQgcmVpbXBsZW1lbnRhdGlvbiBvZiB0aGUgTWVkTU5JU1QgdjIgUmVzTmV0IGJhc2VsaW5lcy4KCk5vdGhpbmcgaW4gdGhpcyBwYWNrYWdlIGlzIGltcG9ydGVkIG9yIGFkYXB0ZWQgZnJvbSBgYE1lZE1OSVNUL2V4cGVyaW1lbnRzYGAuClRoZSBvbmx5IE1lZE1OSVNUIGNvZGUgdXNlZCBpcyB0aGUgYGBtZWRtbmlzdGBgIFB5UEkgcGFja2FnZTogaXRzIGRhdGFzZXQKbG9hZGVycyAoZm9yIHRoZSBzdGFuZGFyZGl6ZWQgZGF0YSkgYW5kIGBgRXZhbHVhdG9yYGAgKGFzIGEgbWV0cmljIG9yYWNsZSkuCgpUaGUgb25lIGV4Y2VwdGlvbiBpcyA6bW9kOmBzcmMucmVwcm9kdWN0aW9uX2FybWAsIHdoaWNoIGRvZXMgbm90IGltcGxlbWVudAphbnl0aGluZyDigJQgaXQgb25seSB0YWJ1bGF0ZXMsIHdpdGggZXhwbGljaXQgcHJvdmVuYW5jZSwgdGhlIHJ1bnMgdGhhdCB3ZXJlCmV4ZWN1dGVkIHdpdGggdGhlIGF1dGhvcnMnIG93biBjb2RlIGFzIGEgc2VwYXJhdGUgKnJlcHJvZHVjdGlvbiogYXJtLgoiIiIKCl9fYWxsX18gPSBbIm1ldHJpY3MiLCAibW9kZWxzIiwgImRhdGEiLCAidHJhaW4iLCAiZXZhbHVhdGUiLCAicnVuIiwgImFnZ3JlZ2F0ZSIsCiAgICAgICAgICAgImV4dGVuc2lvbnMiLCAiZmlndXJlc19yZXBsaWNhdGlvbiIsICJwbG90dGluZyIsICJyZWZlcmVuY2UiLAogICAgICAgICAgICJmcm9tX3ByZWRpY3Rpb25zIiwgInJlcHJvZHVjdGlvbl9hcm0iXQo=',
    'aggregate.py': 'IiIiQWdncmVnYXRlIHBlci1ydW4gYGBydW4uanNvbmBgIGZpbGVzIGludG8gbWVhbsKxc3RkIHRhYmxlcyB2cyB0aGUgcGFwZXIuCgpFbWl0cyBgYHJlcG9ydC9jb21wYXJpc29uLmNzdmBgIGFuZCBgYHJlcG9ydC9jb21wYXJpc29uLm1kYGA6IG9uZSByb3cgcGVyCihkYXRhc2V0LCBtb2RlbCwgc2l6ZSkgd2l0aCBvdXIgbWVhbsKxc3RkIEFVQy9BQ0MsIHRoZSBwYXBlcidzIHJlZmVyZW5jZSB2YWx1ZSwKdGhlIHNpZ25lZCBkZWx0YSwgYW5kIGFuIG91dC1vZi10b2xlcmFuY2UgZmxhZy4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgb3MKaW1wb3J0IGdsb2IKaW1wb3J0IGpzb24KaW1wb3J0IG51bXB5IGFzIG5wCgpmcm9tIC5yZWZlcmVuY2UgaW1wb3J0IFJFRkVSRU5DRQoKIyBUb2xlcmFuY2VzIGZyb20gdGhlIHRhc2sncyBkZWZpbml0aW9uIG9mIGRvbmUuCkFVQ19UT0wgPSAwLjAyCkFDQ19UT0wgPSAwLjAzCgoKZGVmIGxvYWRfcnVucyhyZXN1bHRzX2Rpcj0icmVzdWx0cyIpOgogICAgcnVucyA9IFtdCiAgICBmb3IgcGF0aCBpbiBzb3J0ZWQoZ2xvYi5nbG9iKG9zLnBhdGguam9pbihyZXN1bHRzX2RpciwgIioiLCAicnVuLmpzb24iKSkpOgogICAgICAgIHdpdGggb3BlbihwYXRoKSBhcyBmOgogICAgICAgICAgICBydW5zLmFwcGVuZChqc29uLmxvYWQoZikpCiAgICByZXR1cm4gcnVucwoKCmRlZiBfaXNfYmFzZWxpbmUocik6CiAgICBjID0gclsiY29uZmlnIl0KICAgIHJldHVybiAoYy5nZXQoIndpZHRoX211bHQiLCAxLjApID09IDEuMCBhbmQgbm90IGMuZ2V0KCJ3ZWlnaHRlZF9zYW1wbGVyIikKICAgICAgICAgICAgYW5kIG5vdCBjLmdldCgid2VpZ2h0ZWRfbG9zcyIpIGFuZCBub3QgYy5nZXQoInRhZyIpKQoKCmRlZiBhZ2dyZWdhdGUocmVzdWx0c19kaXI9InJlc3VsdHMiLCByZXBvcnRfZGlyPSJyZXBvcnQiKToKICAgIHJ1bnMgPSBbciBmb3IgciBpbiBsb2FkX3J1bnMocmVzdWx0c19kaXIpIGlmIF9pc19iYXNlbGluZShyKV0KICAgIGdyb3VwcyA9IHt9CiAgICBmb3IgciBpbiBydW5zOgogICAgICAgIGMgPSByWyJjb25maWciXQogICAgICAgIGtleSA9IChjWyJkYXRhc2V0Il0sIGNbIm1vZGVsIl0sIGNbInNpemUiXSkKICAgICAgICBncm91cHMuc2V0ZGVmYXVsdChrZXksIFtdKS5hcHBlbmQocikKCiAgICByb3dzID0gW10KICAgIGZvciBrZXkgaW4gc29ydGVkKGdyb3Vwcyk6CiAgICAgICAgZGF0YXNldCwgbW9kZWwsIHNpemUgPSBrZXkKICAgICAgICBnID0gZ3JvdXBzW2tleV0KICAgICAgICBhdWNzID0gbnAuYXJyYXkoW3JbInRlc3RfYXVjIl0gZm9yIHIgaW4gZ10pCiAgICAgICAgYWNjcyA9IG5wLmFycmF5KFtyWyJ0ZXN0X2FjYyJdIGZvciByIGluIGddKQogICAgICAgIHJlZiA9IFJFRkVSRU5DRS5nZXQoa2V5KQogICAgICAgIHJlZl9hdWMsIHJlZl9hY2MgPSAocmVmIGlmIHJlZiBlbHNlIChOb25lLCBOb25lKSkKICAgICAgICBkX2F1YyA9IGZsb2F0KGF1Y3MubWVhbigpIC0gcmVmX2F1YykgaWYgcmVmIGVsc2UgTm9uZQogICAgICAgIGRfYWNjID0gZmxvYXQoYWNjcy5tZWFuKCkgLSByZWZfYWNjKSBpZiByZWYgZWxzZSBOb25lCiAgICAgICAgZmxhZyA9ICIiCiAgICAgICAgaWYgcmVmOgogICAgICAgICAgICBpZiBhYnMoZF9hdWMpID4gQVVDX1RPTCBvciBhYnMoZF9hY2MpID4gQUNDX1RPTDoKICAgICAgICAgICAgICAgIGZsYWcgPSAiT1VUX09GX1RPTCIKICAgICAgICByb3dzLmFwcGVuZChkaWN0KAogICAgICAgICAgICBkYXRhc2V0PWRhdGFzZXQsIG1vZGVsPW1vZGVsLCBzaXplPXNpemUsIG5fc2VlZHM9bGVuKGcpLAogICAgICAgICAgICBhdWNfbWVhbj1mbG9hdChhdWNzLm1lYW4oKSksIGF1Y19zdGQ9ZmxvYXQoYXVjcy5zdGQoZGRvZj0wKSksCiAgICAgICAgICAgIGFjY19tZWFuPWZsb2F0KGFjY3MubWVhbigpKSwgYWNjX3N0ZD1mbG9hdChhY2NzLnN0ZChkZG9mPTApKSwKICAgICAgICAgICAgcmVmX2F1Yz1yZWZfYXVjLCByZWZfYWNjPXJlZl9hY2MsCiAgICAgICAgICAgIGRlbHRhX2F1Yz1kX2F1YywgZGVsdGFfYWNjPWRfYWNjLCBmbGFnPWZsYWcsCiAgICAgICAgKSkKCiAgICBvcy5tYWtlZGlycyhyZXBvcnRfZGlyLCBleGlzdF9vaz1UcnVlKQogICAgX3dyaXRlX2Nzdihyb3dzLCBvcy5wYXRoLmpvaW4ocmVwb3J0X2RpciwgImNvbXBhcmlzb24uY3N2IikpCiAgICBfd3JpdGVfbWQocm93cywgb3MucGF0aC5qb2luKHJlcG9ydF9kaXIsICJjb21wYXJpc29uLm1kIikpCiAgICByZXR1cm4gcm93cwoKCmRlZiBfd3JpdGVfY3N2KHJvd3MsIHBhdGgpOgogICAgaW1wb3J0IGNzdgogICAgZmllbGRzID0gWyJkYXRhc2V0IiwgIm1vZGVsIiwgInNpemUiLCAibl9zZWVkcyIsICJhdWNfbWVhbiIsICJhdWNfc3RkIiwKICAgICAgICAgICAgICAiYWNjX21lYW4iLCAiYWNjX3N0ZCIsICJyZWZfYXVjIiwgInJlZl9hY2MiLCAiZGVsdGFfYXVjIiwKICAgICAgICAgICAgICAiZGVsdGFfYWNjIiwgImZsYWciXQogICAgd2l0aCBvcGVuKHBhdGgsICJ3IiwgbmV3bGluZT0iIikgYXMgZjoKICAgICAgICB3ID0gY3N2LkRpY3RXcml0ZXIoZiwgZmllbGRuYW1lcz1maWVsZHMpCiAgICAgICAgdy53cml0ZWhlYWRlcigpCiAgICAgICAgZm9yIHIgaW4gcm93czoKICAgICAgICAgICAgdy53cml0ZXJvdyhyKQoKCmRlZiBfZm10KHgsIG5kPTMpOgogICAgcmV0dXJuICIiIGlmIHggaXMgTm9uZSBlbHNlIGYie3g6LntuZH1mfSIKCgpkZWYgX3dyaXRlX21kKHJvd3MsIHBhdGgpOgogICAgbGluZXMgPSBbCiAgICAgICAgIiMgTWVkTU5JU1QgdjIgcmVwbGljYXRpb24g4oCUIGNvbXBhcmlzb24gdnMgcGFwZXIgKFRhYmxlIDMpIiwKICAgICAgICAiIiwKICAgICAgICAiQVVDIC8gQUNDIHJlcG9ydGVkIGFzIG91ciBtZWFuIMKxIHN0ZCBhY3Jvc3Mgc2VlZHM7IGRlbHRhID0gb3VycyDiiJIgcGFwZXIuIiwKICAgICAgICBmIlRvbGVyYW5jZTogfM6UQVVDfCDiiaQge0FVQ19UT0x9LCB8zpRBQ0N8IOKJpCB7QUNDX1RPTH0uIEZsYWcgbWFya3MgY29uZmlncyBvdXRzaWRlIGl0LiIsCiAgICAgICAgIiIsCiAgICAgICAgInwgRGF0YXNldCB8IE1vZGVsIHwgU2l6ZSB8IFNlZWRzIHwgT3VyIEFVQyB8IFBhcGVyIEFVQyB8IM6UQVVDIHwgT3VyIEFDQyB8IFBhcGVyIEFDQyB8IM6UQUNDIHwgRmxhZyB8IiwKICAgICAgICAifC0tLXwtLS18LS0tfC0tLXwtLS18LS0tfC0tLXwtLS18LS0tfC0tLXwtLS18IiwKICAgIF0KICAgIGZvciByIGluIHJvd3M6CiAgICAgICAgb3VyX2F1YyA9IGYie3JbJ2F1Y19tZWFuJ106LjNmfSDCsSB7clsnYXVjX3N0ZCddOi4zZn0iCiAgICAgICAgb3VyX2FjYyA9IGYie3JbJ2FjY19tZWFuJ106LjNmfSDCsSB7clsnYWNjX3N0ZCddOi4zZn0iCiAgICAgICAgbGluZXMuYXBwZW5kKAogICAgICAgICAgICBmInwge3JbJ2RhdGFzZXQnXX0gfCB7clsnbW9kZWwnXX0gfCB7clsnc2l6ZSddfSB8IHtyWyduX3NlZWRzJ119IHwgIgogICAgICAgICAgICBmIntvdXJfYXVjfSB8IHtfZm10KHJbJ3JlZl9hdWMnXSl9IHwge19mbXQoclsnZGVsdGFfYXVjJ10pfSB8ICIKICAgICAgICAgICAgZiJ7b3VyX2FjY30gfCB7X2ZtdChyWydyZWZfYWNjJ10pfSB8IHtfZm10KHJbJ2RlbHRhX2FjYyddKX0gfCAiCiAgICAgICAgICAgIGYie3JbJ2ZsYWcnXX0gfCIpCiAgICBsaW5lcy5hcHBlbmQoIiIpCiAgICB3aXRoIG9wZW4ocGF0aCwgInciKSBhcyBmOgogICAgICAgIGYud3JpdGUoIlxuIi5qb2luKGxpbmVzKSkKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgaW1wb3J0IHN5cwogICAgcmQgPSBzeXMuYXJndlsxXSBpZiBsZW4oc3lzLmFyZ3YpID4gMSBlbHNlICJyZXN1bHRzIgogICAgcm93cyA9IGFnZ3JlZ2F0ZShyZCkKICAgIGZvciByIGluIHJvd3M6CiAgICAgICAgcHJpbnQocikK',
    'data.py': 'IiIiRGF0YXNldCBsb2FkaW5nIGFuZCB0cmFuc2Zvcm0gcGlwZWxpbmVzLgoKV2UgdGFrZSBvbmx5IHRoZSBzdGFuZGFyZGl6ZWQgZGF0YXNldCBmcm9tIHRoZSBgYG1lZG1uaXN0YGAgcGFja2FnZSAoaXRzIGxvYWRlcgpjbGFzc2VzKSBwbHVzIHRoZSBsYWJlbCBhcnJheS4gRXZlcnl0aGluZyBlbHNlIGhlcmUgaXMgb3Vycy4KCktleSBwcm90b2NvbCBwb2ludHMsIGZhaXRoZnVsIHRvIE1lZE1OSVNUIHYyICgyMDIzKToKCiogQWx3YXlzIGxvYWQgdGhlICoqMjgtcGl4ZWwqKiBgYC5ucHpgYCAoYGBzaXplPTI4YGApLCBgYGFzX3JnYj1UcnVlYGAuCiogVGhlIDIyNCBjb25maWdzICpyZXNpemUgdGhlIDI4LXBpeGVsIGRhdGEgaW5zaWRlIHRoZSB0cmFuc2Zvcm0qIHVzaW5nCiAgbmVhcmVzdC1uZWlnaGJvdXIgaW50ZXJwb2xhdGlvbi4gV2UgZGVsaWJlcmF0ZWx5IGRvICoqbm90KiogbG9hZCBNZWRNTklTVCsncwogIG5hdGl2ZSAyMjQgaW1hZ2VzLCB3aGljaCBkaWQgbm90IGV4aXN0IGZvciB0aGUgMjAyMyBwYXBlciBhbmQgd291bGQgZ2l2ZQogIGRpZmZlcmVudCAoYmV0dGVyKSBudW1iZXJzLgoqIE5vcm1hbGlzYXRpb24gaXMgYGBtZWFuPVsuNV0sIHN0ZD1bLjVdYGAgYnJvYWRjYXN0IGFjcm9zcyB0aGUgMyBjaGFubmVscwogIChyb3VnaGx5IG1hcHMgdG8gWy0xLCAxXSkuIE5vIHRyYWluLXRpbWUgYXVnbWVudGF0aW9uIGluIHRoZSBiYXNlbGluZS4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgZ2xvYgppbXBvcnQgaGFzaGxpYgppbXBvcnQgb3MKaW1wb3J0IHNodXRpbAoKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCB0b3JjaAppbXBvcnQgdG9yY2h2aXNpb24udHJhbnNmb3JtcyBhcyBUCmZyb20gUElMIGltcG9ydCBJbWFnZQoKaW1wb3J0IG1lZG1uaXN0CmZyb20gbWVkbW5pc3QgaW1wb3J0IElORk8KZnJvbSBtZWRtbmlzdC5kYXRhc2V0IGltcG9ydCBERUZBVUxUX1JPT1QKCgpkZWYgX21kNShwYXRoLCBjaHVua19zaXplPTEgPDwgMjApOgogICAgaCA9IGhhc2hsaWIubWQ1KCkKICAgIHdpdGggb3BlbihwYXRoLCAicmIiKSBhcyBmOgogICAgICAgIGZvciBjaHVuayBpbiBpdGVyKGxhbWJkYTogZi5yZWFkKGNodW5rX3NpemUpLCBiIiIpOgogICAgICAgICAgICBoLnVwZGF0ZShjaHVuaykKICAgIHJldHVybiBoLmhleGRpZ2VzdCgpCgoKZGVmIF9zdGFnZV9mcm9tX21pcnJvcihkYXRhc2V0LCByb290LCBzaXplX2ZsYWc9IiIpOgogICAgIiIiQ29weSBhIHZhbGlkIGNhY2hlZC9taXJyb3JlZCBgYC5ucHpgYCBpbnRvIHBsYWNlIHNvIGBgbWVkbW5pc3RgYCBuZXZlcgogICAgaGFzIHRvIHRvdWNoIFplbm9kbyBhdCBhbGwuCgogICAgWmVub2RvICh0aGUgcGFja2FnZSdzIG9ubHkgZG93bmxvYWQgaG9zdCkgaGFzIHJlYWwgb3V0YWdlcyAtLSByZXRyeWluZyBhCiAgICBkZWFkIGhvc3QganVzdCBidXJucyBHUFUgdGltZS4gSWYgYSBzYW1lLW5hbWVkIGBgLm5wemBgIGFscmVhZHkgc2l0cwogICAgYW55d2hlcmUgdW5kZXIgYSBLYWdnbGUgaW5wdXQgbW91bnQgKGBgL2thZ2dsZS9pbnB1dC8qKmBgKSBvciBpbgogICAgYGBNRURNTklTVF9NSVJST1JfRElSU2BgIChjb2xvbi1zZXBhcmF0ZWQpLCBhbmQgaXRzIE1ENSBtYXRjaGVzIHRoZQogICAgY2hlY2tzdW0gYGBtZWRtbmlzdGBgIGl0c2VsZiB3b3VsZCB2ZXJpZnksIHN0YWdlIGl0IGludG8gYGByb290YGAgc28gdGhlCiAgICBpbnRlZ3JpdHkgY2hlY2sgaW4gYGB0b3JjaHZpc2lvbi5kYXRhc2V0cy51dGlscy5kb3dubG9hZF91cmxgYCBzaG9ydC0KICAgIGNpcmN1aXRzIGJlZm9yZSBhbnkgbmV0d29yayBjYWxsLiBBIG5vbi1tYXRjaGluZyBvciBtaXNzaW5nIG1pcnJvciBpcyBhCiAgICBzaWxlbnQgbm8tb3AgLS0gdGhlIG5vcm1hbCAocG9zc2libHkgcmV0cmllZCkgZG93bmxvYWQgc3RpbGwgcnVucy4KICAgICIiIgogICAgaW5mbyA9IElORk9bZGF0YXNldF0KICAgIGZpbGVuYW1lID0gZiJ7ZGF0YXNldH17c2l6ZV9mbGFnfS5ucHoiCiAgICBkZXN0ID0gb3MucGF0aC5qb2luKHJvb3QsIGZpbGVuYW1lKQogICAgaWYgb3MucGF0aC5leGlzdHMoZGVzdCkgYW5kIF9tZDUoZGVzdCkgPT0gaW5mb1tmIk1ENXtzaXplX2ZsYWd9Il06CiAgICAgICAgcmV0dXJuIFRydWUgICMgYWxyZWFkeSBzdGFnZWQvY2FjaGVkIGNvcnJlY3RseQoKICAgIHNlYXJjaF9kaXJzID0gWyIva2FnZ2xlL2lucHV0Il0gKyBbCiAgICAgICAgZCBmb3IgZCBpbiBvcy5lbnZpcm9uLmdldCgiTUVETU5JU1RfTUlSUk9SX0RJUlMiLCAiIikuc3BsaXQoIjoiKSBpZiBkCiAgICBdCiAgICBjYW5kaWRhdGVzID0gW10KICAgIGZvciBkIGluIHNlYXJjaF9kaXJzOgogICAgICAgIGNhbmRpZGF0ZXMuZXh0ZW5kKGdsb2IuZ2xvYihvcy5wYXRoLmpvaW4oZCwgIioqIiwgZmlsZW5hbWUpLCByZWN1cnNpdmU9VHJ1ZSkpCgogICAgZm9yIGNhbmQgaW4gY2FuZGlkYXRlczoKICAgICAgICB0cnk6CiAgICAgICAgICAgIGlmIF9tZDUoY2FuZCkgPT0gaW5mb1tmIk1ENXtzaXplX2ZsYWd9Il06CiAgICAgICAgICAgICAgICBvcy5tYWtlZGlycyhyb290LCBleGlzdF9vaz1UcnVlKQogICAgICAgICAgICAgICAgc2h1dGlsLmNvcHlmaWxlKGNhbmQsIGRlc3QpCiAgICAgICAgICAgICAgICBwcmludChmIltkYXRhXSBzdGFnZWQge2RhdGFzZXR9e3NpemVfZmxhZ30ubnB6IGZyb20gbWlycm9yIHtjYW5kfSAiCiAgICAgICAgICAgICAgICAgICAgICAiKE1ENSB2ZXJpZmllZCkgLS0gc2tpcHBpbmcgWmVub2RvIGRvd25sb2FkIikKICAgICAgICAgICAgICAgIHJldHVybiBUcnVlCiAgICAgICAgZXhjZXB0IE9TRXJyb3I6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICByZXR1cm4gRmFsc2UKCgpkZWYgYnVpbGRfdHJhbnNmb3JtKHNpemUpOgogICAgIiIiVHJhbnNmb3JtIGZvciB0aGUgZ2l2ZW4gcmVzb2x1dGlvbiAobm8gYXVnbWVudGF0aW9uOyBiYXNlbGluZSkuIiIiCiAgICBpZiBzaXplID09IDI4OgogICAgICAgIHJldHVybiBULkNvbXBvc2UoWwogICAgICAgICAgICBULlRvVGVuc29yKCksCiAgICAgICAgICAgIFQuTm9ybWFsaXplKG1lYW49Wy41XSwgc3RkPVsuNV0pLAogICAgICAgIF0pCiAgICBpZiBzaXplID09IDIyNDoKICAgICAgICAjIE5lYXJlc3QtbmVpZ2hib3VyIHNwZWNpZmljYWxseTsgdGhlIDI4LXBpeGVsIHNvdXJjZSBpcyB1cHNhbXBsZWQgaGVyZS4KICAgICAgICByZXR1cm4gVC5Db21wb3NlKFsKICAgICAgICAgICAgVC5SZXNpemUoKDIyNCwgMjI0KSwgaW50ZXJwb2xhdGlvbj1JbWFnZS5ORUFSRVNUKSwKICAgICAgICAgICAgVC5Ub1RlbnNvcigpLAogICAgICAgICAgICBULk5vcm1hbGl6ZShtZWFuPVsuNV0sIHN0ZD1bLjVdKSwKICAgICAgICBdKQogICAgcmFpc2UgVmFsdWVFcnJvcihmInVuc3VwcG9ydGVkIHNpemUge3NpemV9IikKCgpkZWYgZ2V0X2luZm8oZGF0YXNldCk6CiAgICByZXR1cm4gSU5GT1tkYXRhc2V0XQoKCmRlZiBnZXRfZGF0YXNldChkYXRhc2V0LCBzcGxpdCwgc2l6ZSwgcm9vdD1Ob25lLCBkb3dubG9hZD1UcnVlLAogICAgICAgICAgICAgICBkb3dubG9hZF9yZXRyaWVzPTQsIGRvd25sb2FkX2JhY2tvZmZfcz0yMCk6CiAgICAiIiJSZXR1cm4gYSBgYG1lZG1uaXN0YGAgZGF0YXNldCBvYmplY3QgZm9yIGBgc3BsaXRgYCB3aXRoIG91ciB0cmFuc2Zvcm0uCgogICAgRGF0YSBpcyBhbHdheXMgbG9hZGVkIGZyb20gdGhlIDI4LXBpeGVsIGBgLm5wemBgOyBgYHNpemVgYCBvbmx5IGNvbnRyb2xzIHRoZQogICAgdHJhbnNmb3JtICh0aGUgMjI0IHBpcGVsaW5lIHJlc2l6ZXMgaW50ZXJuYWxseSkuCgogICAgWmVub2RvICh0aGUgYGBtZWRtbmlzdGBgIHBhY2thZ2UncyBkb3dubG9hZCBob3N0KSBpbnRlcm1pdHRlbnRseSByZXR1cm5zCiAgICA1MDRzIG9uIGFuIG90aGVyd2lzZS1maW5lIGNvbm5lY3Rpb247IHRoZSBwYWNrYWdlIGl0c2VsZiBkb2Vzbid0IHJldHJ5LCBzbwogICAgYSBzaW5nbGUgYmxpcCBraWxscyBhbiBvdGhlcndpc2UtaGVhbHRoeSB0cmFpbmluZyBydW4uIFJldHJ5IHRoZSB3aG9sZQogICAgY29uc3RydWN0aW9uICh3aGljaCByZS1hdHRlbXB0cyB0aGUgZG93bmxvYWQgaWYgdGhlIC5ucHogc3RpbGwgaXNuJ3QKICAgIGNhY2hlZCkgd2l0aCBiYWNrb2ZmIGJlZm9yZSBnaXZpbmcgdXAuCiAgICAiIiIKICAgIGluZm8gPSBJTkZPW2RhdGFzZXRdCiAgICBEYXRhQ2xhc3MgPSBnZXRhdHRyKG1lZG1uaXN0LCBpbmZvWyJweXRob25fY2xhc3MiXSkKICAgIHRyYW5zZm9ybSA9IGJ1aWxkX3RyYW5zZm9ybShzaXplKQogICAga3dhcmdzID0gZGljdChzcGxpdD1zcGxpdCwgdHJhbnNmb3JtPXRyYW5zZm9ybSwgZG93bmxvYWQ9ZG93bmxvYWQsCiAgICAgICAgICAgICAgICAgIGFzX3JnYj1UcnVlLCBzaXplPTI4KQogICAgaWYgcm9vdCBpcyBub3QgTm9uZToKICAgICAgICBrd2FyZ3NbInJvb3QiXSA9IHJvb3QKCiAgICBpZiBkb3dubG9hZDoKICAgICAgICBfc3RhZ2VfZnJvbV9taXJyb3IoZGF0YXNldCwgcm9vdCBvciBERUZBVUxUX1JPT1QpCgogICAgbGFzdF9lcnIgPSBOb25lCiAgICBmb3IgYXR0ZW1wdCBpbiByYW5nZShkb3dubG9hZF9yZXRyaWVzKToKICAgICAgICB0cnk6CiAgICAgICAgICAgIHJldHVybiBEYXRhQ2xhc3MoKiprd2FyZ3MpCiAgICAgICAgZXhjZXB0IFJ1bnRpbWVFcnJvciBhcyBlOgogICAgICAgICAgICBsYXN0X2VyciA9IGUKICAgICAgICAgICAgaWYgYXR0ZW1wdCA8IGRvd25sb2FkX3JldHJpZXMgLSAxOgogICAgICAgICAgICAgICAgaW1wb3J0IHRpbWUKICAgICAgICAgICAgICAgIHdhaXQgPSBkb3dubG9hZF9iYWNrb2ZmX3MgKiAoYXR0ZW1wdCArIDEpCiAgICAgICAgICAgICAgICBwcmludChmIltkYXRhXSB7ZGF0YXNldH0gZG93bmxvYWQgZmFpbGVkIChhdHRlbXB0IHthdHRlbXB0ICsgMX0vIgogICAgICAgICAgICAgICAgICAgICAgZiJ7ZG93bmxvYWRfcmV0cmllc30pLCByZXRyeWluZyBpbiB7d2FpdH1zOiB7ZX0iKQogICAgICAgICAgICAgICAgdGltZS5zbGVlcCh3YWl0KQogICAgcmFpc2UgbGFzdF9lcnIKCgpkZWYgZ2V0X2xvYWRlcnMoZGF0YXNldCwgc2l6ZSwgYmF0Y2hfc2l6ZT0xMjgsIHJvb3Q9Tm9uZSwgZG93bmxvYWQ9VHJ1ZSwKICAgICAgICAgICAgICAgIG51bV93b3JrZXJzPTIsIHNhbXBsZXI9Tm9uZSwgZXZhbF9iYXRjaF9zaXplPU5vbmUsCiAgICAgICAgICAgICAgICBwaW5fbWVtb3J5PVRydWUpOgogICAgIiIiQnVpbGQgdHJhaW4vdmFsL3Rlc3QgZGF0YWxvYWRlcnMuCgogICAgYGBzYW1wbGVyYGAgKGUuZy4gYSBgYFdlaWdodGVkUmFuZG9tU2FtcGxlcmBgKSByZXBsYWNlcyBzaHVmZmxpbmcgb24gdGhlCiAgICB0cmFpbiBsb2FkZXIgd2hlbiBwcm92aWRlZDsgdXNlZCBieSB0aGUgYmlhcy1taXRpZ2F0aW9uIGV4dGVuc2lvbi4KICAgICIiIgogICAgdHJhaW5fc2V0ID0gZ2V0X2RhdGFzZXQoZGF0YXNldCwgInRyYWluIiwgc2l6ZSwgcm9vdCwgZG93bmxvYWQpCiAgICB2YWxfc2V0ID0gZ2V0X2RhdGFzZXQoZGF0YXNldCwgInZhbCIsIHNpemUsIHJvb3QsIGRvd25sb2FkKQogICAgdGVzdF9zZXQgPSBnZXRfZGF0YXNldChkYXRhc2V0LCAidGVzdCIsIHNpemUsIHJvb3QsIGRvd25sb2FkKQoKICAgIGV2YWxfYnMgPSBldmFsX2JhdGNoX3NpemUgb3IgYmF0Y2hfc2l6ZQogICAgc2h1ZmZsZSA9IHNhbXBsZXIgaXMgTm9uZQogICAgdHJhaW5fbG9hZGVyID0gdG9yY2gudXRpbHMuZGF0YS5EYXRhTG9hZGVyKAogICAgICAgIHRyYWluX3NldCwgYmF0Y2hfc2l6ZT1iYXRjaF9zaXplLCBzaHVmZmxlPXNodWZmbGUsIHNhbXBsZXI9c2FtcGxlciwKICAgICAgICBudW1fd29ya2Vycz1udW1fd29ya2VycywgcGluX21lbW9yeT1waW5fbWVtb3J5LCBkcm9wX2xhc3Q9RmFsc2UpCiAgICB2YWxfbG9hZGVyID0gdG9yY2gudXRpbHMuZGF0YS5EYXRhTG9hZGVyKAogICAgICAgIHZhbF9zZXQsIGJhdGNoX3NpemU9ZXZhbF9icywgc2h1ZmZsZT1GYWxzZSwKICAgICAgICBudW1fd29ya2Vycz1udW1fd29ya2VycywgcGluX21lbW9yeT1waW5fbWVtb3J5KQogICAgdGVzdF9sb2FkZXIgPSB0b3JjaC51dGlscy5kYXRhLkRhdGFMb2FkZXIoCiAgICAgICAgdGVzdF9zZXQsIGJhdGNoX3NpemU9ZXZhbF9icywgc2h1ZmZsZT1GYWxzZSwKICAgICAgICBudW1fd29ya2Vycz1udW1fd29ya2VycywgcGluX21lbW9yeT1waW5fbWVtb3J5KQogICAgcmV0dXJuIHRyYWluX2xvYWRlciwgdmFsX2xvYWRlciwgdGVzdF9sb2FkZXIKCgpkZWYgY2xhc3NfY291bnRzKGRhdGFzZXQsIHJvb3Q9Tm9uZSwgZG93bmxvYWQ9VHJ1ZSk6CiAgICAiIiJQZXItY2xhc3MgdHJhaW5pbmctc2FtcGxlIGNvdW50cyAobXVsdGktY2xhc3MgZGF0YXNldHMpLiIiIgogICAgdHJhaW5fc2V0ID0gZ2V0X2RhdGFzZXQoZGF0YXNldCwgInRyYWluIiwgMjgsIHJvb3QsIGRvd25sb2FkKQogICAgbGFiZWxzID0gbnAuYXNhcnJheSh0cmFpbl9zZXQubGFiZWxzKS5zcXVlZXplKCkuYXN0eXBlKGludCkKICAgIG5fY2xhc3NlcyA9IGxlbihJTkZPW2RhdGFzZXRdWyJsYWJlbCJdKQogICAgY291bnRzID0gbnAuYmluY291bnQobGFiZWxzLCBtaW5sZW5ndGg9bl9jbGFzc2VzKQogICAgcmV0dXJuIGNvdW50cwoKCmRlZiBjbGFzc193ZWlnaHRzKGRhdGFzZXQsIHJvb3Q9Tm9uZSwgZG93bmxvYWQ9VHJ1ZSwgbm9ybWFsaXplPVRydWUpOgogICAgIiIiSW52ZXJzZS1mcmVxdWVuY3kgY2xhc3Mgd2VpZ2h0cyBmb3Igd2VpZ2h0ZWQgQ3Jvc3NFbnRyb3B5TG9zcy4KCiAgICBgYHdfYyDiiJ0gMSAvIGNvdW50X2NgYDsgb3B0aW9uYWxseSBub3JtYWxpc2VkIHRvIG1lYW4gMS4KICAgICIiIgogICAgY291bnRzID0gY2xhc3NfY291bnRzKGRhdGFzZXQsIHJvb3QsIGRvd25sb2FkKS5hc3R5cGUobnAuZmxvYXQ2NCkKICAgIGNvdW50cyA9IG5wLmNsaXAoY291bnRzLCAxLCBOb25lKQogICAgdyA9IDEuMCAvIGNvdW50cwogICAgaWYgbm9ybWFsaXplOgogICAgICAgIHcgPSB3ICogbGVuKHcpIC8gdy5zdW0oKQogICAgcmV0dXJuIHRvcmNoLnRlbnNvcih3LCBkdHlwZT10b3JjaC5mbG9hdDMyKQoKCmRlZiBtYWtlX3dlaWdodGVkX3NhbXBsZXIoZGF0YXNldCwgcm9vdD1Ob25lLCBkb3dubG9hZD1UcnVlKToKICAgICIiIkEgYGBXZWlnaHRlZFJhbmRvbVNhbXBsZXJgYCBnaXZpbmcgZWFjaCBjbGFzcyBlcXVhbCBleHBlY3RlZCBtYXNzLiIiIgogICAgdHJhaW5fc2V0ID0gZ2V0X2RhdGFzZXQoZGF0YXNldCwgInRyYWluIiwgMjgsIHJvb3QsIGRvd25sb2FkKQogICAgbGFiZWxzID0gbnAuYXNhcnJheSh0cmFpbl9zZXQubGFiZWxzKS5zcXVlZXplKCkuYXN0eXBlKGludCkKICAgIGNvdW50cyA9IG5wLmJpbmNvdW50KGxhYmVscywgbWlubGVuZ3RoPWxlbihJTkZPW2RhdGFzZXRdWyJsYWJlbCJdKSkuYXN0eXBlKG5wLmZsb2F0NjQpCiAgICBjb3VudHMgPSBucC5jbGlwKGNvdW50cywgMSwgTm9uZSkKICAgIHNhbXBsZV93ID0gKDEuMCAvIGNvdW50cylbbGFiZWxzXQogICAgc2FtcGxlciA9IHRvcmNoLnV0aWxzLmRhdGEuV2VpZ2h0ZWRSYW5kb21TYW1wbGVyKAogICAgICAgIHdlaWdodHM9dG9yY2gudGVuc29yKHNhbXBsZV93LCBkdHlwZT10b3JjaC5kb3VibGUpLAogICAgICAgIG51bV9zYW1wbGVzPWxlbihzYW1wbGVfdyksIHJlcGxhY2VtZW50PVRydWUpCiAgICByZXR1cm4gc2FtcGxlcgo=',
    'evaluate.py': 'IiIiSW5mZXJlbmNlLCBwcmVkaWN0aW9uIGR1bXBpbmcsIGFuZCBjaGVja3BvaW50IGV2YWx1YXRpb24uIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgb3MKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBwYW5kYXMgYXMgcGQKaW1wb3J0IHRvcmNoCmltcG9ydCB0b3JjaC5ubi5mdW5jdGlvbmFsIGFzIEYKCmZyb20gLiBpbXBvcnQgbWV0cmljcwoKCkB0b3JjaC5ub19ncmFkKCkKZGVmIHByZWRpY3QobW9kZWwsIGxvYWRlciwgdGFzaywgZGV2aWNlLCB1c2VfYW1wPUZhbHNlKToKICAgICIiIlJ1biB0aGUgbW9kZWwgb3ZlciBgYGxvYWRlcmBgOyByZXR1cm4gYGAoeV90cnVlLCB5X3Njb3JlKWBgIG51bXB5IGFycmF5cy4KCiAgICBgYHlfc2NvcmVgYCBpcyBzb2Z0bWF4IHByb2JhYmlsaXRpZXMgKG11bHRpLWNsYXNzKSBvciBzaWdtb2lkcwogICAgKG11bHRpLWxhYmVsKS4gYGB5X3RydWVgYCBpcyBgYChOLCAxKWBgIGludCBmb3IgbXVsdGktY2xhc3MsIGBgKE4sIEMpYGAgZm9yCiAgICBtdWx0aS1sYWJlbCAtLSBtYXRjaGluZyB3aGF0IGBgbWVkbW5pc3QuRXZhbHVhdG9yYGAgZXhwZWN0cy4KICAgICIiIgogICAgbW9kZWwuZXZhbCgpCiAgICBkZXZpY2VfdHlwZSA9ICJjdWRhIiBpZiAiY3VkYSIgaW4gc3RyKGRldmljZSkgZWxzZSAiY3B1IgogICAgc2NvcmVzLCB0cnVlcyA9IFtdLCBbXQogICAgZm9yIHgsIHkgaW4gbG9hZGVyOgogICAgICAgIHggPSB4LnRvKGRldmljZSwgbm9uX2Jsb2NraW5nPVRydWUpCiAgICAgICAgd2l0aCB0b3JjaC5hbXAuYXV0b2Nhc3QoZGV2aWNlX3R5cGUsIGVuYWJsZWQ9dXNlX2FtcCk6CiAgICAgICAgICAgIGxvZ2l0cyA9IG1vZGVsKHgpCiAgICAgICAgbG9naXRzID0gbG9naXRzLmZsb2F0KCkKICAgICAgICBpZiB0YXNrID09ICJtdWx0aS1sYWJlbCwgYmluYXJ5LWNsYXNzIjoKICAgICAgICAgICAgcyA9IHRvcmNoLnNpZ21vaWQobG9naXRzKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHMgPSBGLnNvZnRtYXgobG9naXRzLCBkaW09MSkKICAgICAgICBzY29yZXMuYXBwZW5kKHMuY3B1KCkubnVtcHkoKSkKICAgICAgICB0cnVlcy5hcHBlbmQoeS5udW1weSgpKQogICAgeV9zY29yZSA9IG5wLmNvbmNhdGVuYXRlKHNjb3JlcywgYXhpcz0wKQogICAgeV90cnVlID0gbnAuY29uY2F0ZW5hdGUodHJ1ZXMsIGF4aXM9MCkKICAgIHJldHVybiB5X3RydWUsIHlfc2NvcmUKCgpkZWYgZXZhbHVhdGVfc3BsaXQobW9kZWwsIGxvYWRlciwgdGFzaywgZGV2aWNlLCB1c2VfYW1wPUZhbHNlKToKICAgICIiIlJldHVybiBgYChhdWMsIGFjYywgeV90cnVlLCB5X3Njb3JlKWBgIGZvciBhIHNwbGl0LiIiIgogICAgeV90cnVlLCB5X3Njb3JlID0gcHJlZGljdChtb2RlbCwgbG9hZGVyLCB0YXNrLCBkZXZpY2UsIHVzZV9hbXApCiAgICBhdWMsIGFjYyA9IG1ldHJpY3MuZXZhbHVhdGUoeV90cnVlLCB5X3Njb3JlLCB0YXNrKQogICAgcmV0dXJuIGF1YywgYWNjLCB5X3RydWUsIHlfc2NvcmUKCgpkZWYgcHJlZGljdGlvbl9maWxlbmFtZShmbGFnLCBzcGxpdCwgYXVjLCBhY2MsIHNlZWQsIHNpemU9MjgpOgogICAgIiIiTWVkTU5JU1QtY29udmVudGlvbiBzZWxmLWRlc2NyaWJpbmcgZmlsZW5hbWUuIiIiCiAgICBzaXplX2ZsYWcgPSAiIiBpZiBzaXplID09IDI4IGVsc2UgZiJfe3NpemV9IgogICAgIyBzaXplX2ZsYWcgaXMgZm9sZGVkIGludG8gYGZsYWdgIHBvc2l0aW9uIHBlciBtZWRtbmlzdCBwYXJzZXIgZXhwZWN0YXRpb25zLgogICAgcmV0dXJuIGYie2ZsYWd9e3NpemVfZmxhZ31fe3NwbGl0fV9bQVVDXXthdWM6LjNmfV9bQUNDXXthY2M6LjNmfUBzZWVke3NlZWR9LmNzdiIKCgpkZWYgc2F2ZV9wcmVkaWN0aW9ucyh5X3Njb3JlLCBwYXRoKToKICAgICIiIlNhdmUgc2NvcmVzIGluIHRoZSBtZWRtbmlzdCByZXN1bHQgZm9ybWF0IChpbmRleCwgc2NvcmVfMCwgLi4uKS4iIiIKICAgIG9zLm1ha2VkaXJzKG9zLnBhdGguZGlybmFtZShwYXRoKSBvciAiLiIsIGV4aXN0X29rPVRydWUpCiAgICBwZC5EYXRhRnJhbWUoeV9zY29yZSkudG9fY3N2KHBhdGgsIGhlYWRlcj1Ob25lKQoKCmRlZiBsb2FkX2NoZWNrcG9pbnQocGF0aCwgbW9kZWwsIG9wdGltaXplcj1Ob25lLCBzY2hlZHVsZXI9Tm9uZSwgc2NhbGVyPU5vbmUsCiAgICAgICAgICAgICAgICAgICAgbWFwX2xvY2F0aW9uPSJjcHUiKToKICAgIGNrcHQgPSB0b3JjaC5sb2FkKHBhdGgsIG1hcF9sb2NhdGlvbj1tYXBfbG9jYXRpb24pCiAgICBtb2RlbC5sb2FkX3N0YXRlX2RpY3QoY2twdFsibW9kZWwiXSkKICAgIGlmIG9wdGltaXplciBpcyBub3QgTm9uZSBhbmQgY2twdC5nZXQoIm9wdGltaXplciIpIGlzIG5vdCBOb25lOgogICAgICAgIG9wdGltaXplci5sb2FkX3N0YXRlX2RpY3QoY2twdFsib3B0aW1pemVyIl0pCiAgICBpZiBzY2hlZHVsZXIgaXMgbm90IE5vbmUgYW5kIGNrcHQuZ2V0KCJzY2hlZHVsZXIiKSBpcyBub3QgTm9uZToKICAgICAgICBzY2hlZHVsZXIubG9hZF9zdGF0ZV9kaWN0KGNrcHRbInNjaGVkdWxlciJdKQogICAgaWYgc2NhbGVyIGlzIG5vdCBOb25lIGFuZCBja3B0LmdldCgic2NhbGVyIikgaXMgbm90IE5vbmU6CiAgICAgICAgc2NhbGVyLmxvYWRfc3RhdGVfZGljdChja3B0WyJzY2FsZXIiXSkKICAgIHJldHVybiBja3B0Cg==',
    'extensions.py': '...',
    'figures_replication.py': '...',
    'from_predictions.py': 'IiIiUmVjb21wdXRlIHRoZSBleHRlbnNpb24gYW5hbHlzZXMgZnJvbSBhIHNhdmVkIHByZWRpY3Rpb24gbWF0cml4IOKAlCBubyBHUFUuCgpFdmVyeSBleHRlbnNpb24gdGhhdCBjb25zdW1lcyBvbmx5IGBgKHlfdHJ1ZSwgeV9zY29yZSlgYCDigJQgcGVyLWNsYXNzIFJPQy9QUiwKY29uZnVzaW9uIG1hdHJpeCwgZnJlcXVlbmN5LXZzLXBlcmZvcm1hbmNlLCBjYWxpYnJhdGlvbi9FQ0UsIGNvbmZpZGVuY2Ug4oCUIGNhbiBiZQpyZWdlbmVyYXRlZCBmcm9tIGEgc2NvcmUgQ1NWIHRoYXQgYSAqcHJldmlvdXMqIHJ1biBhbHJlYWR5IGR1bXBlZC4gVGhhdCB0dXJucyBhbgpob3VyIG9mIEdQVSBpbnRvIGEgZmV3IHNlY29uZHMgb2YgQ1BVLCBhbmQgaXQgaXMgaG93IHRoZSBhcmNoaXZlZApgYHJlc3VsdHMvcmVwcm9kdWN0aW9uX2F1dGhvcnNfY29kZS9gYCBydW5zIGdldCB0aGUgc2FtZSB0cmVhdG1lbnQgYXMgbmV3IG9uZXMKd2l0aG91dCByZXRyYWluaW5nIHRoZW0uCgpTY29yZSBDU1ZzIGFyZSBpbiB0aGUgTWVkTU5JU1QgY29udmVudGlvbiB3cml0dGVuIGJ5IGJvdGggdGhlIGF1dGhvcnMnIHNjcmlwdAphbmQgb3VyIDpmdW5jOmBzcmMuZXZhbHVhdGUuc2F2ZV9wcmVkaWN0aW9uc2A6IG5vIGhlYWRlciwgY29sdW1uIDAgaXMgdGhlIHJvdwppbmRleCwgY29sdW1ucyAxLi5DIGFyZSBjbGFzcyBzY29yZXMuCgogICAgcHl0aG9uIC1tIHNyYy5mcm9tX3ByZWRpY3Rpb25zIFxcCiAgICAgICAgLS1zY29yZXMgInJlc3VsdHMvcmVwcm9kdWN0aW9uX2F1dGhvcnNfY29kZS9wZXJzb25CX3Jlc25ldDUwX2Rlcm1hMjI0L291dHB1dC9kZXJtYW1uaXN0LzI2MDcyMF8wODI3MTcvZGVybWFtbmlzdF90ZXN0X1tBVUNdMC45MTJfW0FDQ10wLjczOEBydW4xLmNzdiIgXFwKICAgICAgICAtLWRhdGFzZXQgZGVybWFtbmlzdCAtLXRhZyBwZXJzb25CX3Jlc25ldDUwXzIyNCBcXAogICAgICAgIC0tb3V0IHJlc3VsdHMvcmVwcm9kdWN0aW9uX2F1dGhvcnNfY29kZS9wZXJzb25CX3Jlc25ldDUwX2Rlcm1hMjI0L3JlY29tcHV0ZWQKCk5lZWRzIHRoZSBhbmFseXNpcyBzdGFjayAobnVtcHkvcGFuZGFzL3NrbGVhcm4vc2NpcHkvbWF0cGxvdGxpYi9tZWRtbmlzdCkgYnV0Cm5ldmVyIHRvdWNoZXMgYSBHUFUuIEEgQ1BVLW9ubHkgdG9yY2ggd2hlZWwgaXMgc3VmZmljaWVudC4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGpzb24KaW1wb3J0IG9zCmltcG9ydCByZQoKaW1wb3J0IG51bXB5IGFzIG5wCgojIFRvbGVyYW5jZSBmb3IgdGhlIHNlbGYtY2hlY2sgYWdhaW5zdCB0aGUgQVVDL0FDQyBiYWtlZCBpbnRvIHRoZSBmaWxlbmFtZS4KIyBMb29zZSBlbm91Z2ggZm9yIHRoZSAzLWRlY2ltYWwgcm91bmRpbmcgaW4gdGhlIG5hbWUsIHRpZ2h0IGVub3VnaCB0byBjYXRjaCBhCiMgcm93LW9yZGVyIG1pc21hdGNoICh3aGljaCB3b3VsZCBtb3ZlIEFVQyBieSBmYXIgbW9yZSB0aGFuIHRoaXMpLgpGSUxFTkFNRV9UT0wgPSAxZS0zCgpfRk5BTUVfUkUgPSByZS5jb21waWxlKHIiXFtBVUNcXSg/UDxhdWM+WzAtOS5dKylfXFtBQ0NcXSg/UDxhY2M+WzAtOS5dKykiKQoKCmRlZiBsb2FkX3Njb3JlcyhwYXRoKToKICAgICIiIlJlYWQgYSBNZWRNTklTVC1jb252ZW50aW9uIHNjb3JlIENTViBpbnRvIGFuIGBgKE4sIEMpYGAgZmxvYXQgYXJyYXkuIiIiCiAgICByYXcgPSBucC5sb2FkdHh0KHBhdGgsIGRlbGltaXRlcj0iLCIsIGR0eXBlPWZsb2F0KQogICAgaWYgcmF3Lm5kaW0gIT0gMiBvciByYXcuc2hhcGVbMV0gPCAyOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJ7cGF0aH06IGV4cGVjdGVkIGFuIChOLCAxK0MpIG1hdHJpeCwgZ290IHtyYXcuc2hhcGV9IikKICAgIGluZGV4LCBzY29yZXMgPSByYXdbOiwgMF0sIHJhd1s6LCAxOl0KICAgIGV4cGVjdGVkID0gbnAuYXJhbmdlKGxlbihpbmRleCkpCiAgICBpZiBub3QgbnAuYXJyYXlfZXF1YWwoaW5kZXguYXN0eXBlKGludCksIGV4cGVjdGVkKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKAogICAgICAgICAgICBmIntwYXRofTogY29sdW1uIDAgaXMgbm90IGEgMC4uTi0xIHJvdyBpbmRleCDigJQgdGhlIGZpbGUgbWF5IGhhdmUgYSAiCiAgICAgICAgICAgICJoZWFkZXIgcm93IG9yIGJlIGluIGEgZGlmZmVyZW50IGZvcm1hdCIpCiAgICByZXR1cm4gc2NvcmVzCgoKZGVmIGxvYWRfbGFiZWxzKGRhdGFzZXQsIHNwbGl0PSJ0ZXN0Iiwgcm9vdD1Ob25lLCBkb3dubG9hZD1UcnVlKToKICAgICIiIkdyb3VuZC10cnV0aCBsYWJlbHMgZm9yIGBgc3BsaXRgYCwgaW4gbnB6IG9yZGVyICh0aGUgZHVtcCBvcmRlcikuIiIiCiAgICBmcm9tIC5kYXRhIGltcG9ydCBnZXRfZGF0YXNldAogICAgZHMgPSBnZXRfZGF0YXNldChkYXRhc2V0LCBzcGxpdCwgMjgsIHJvb3Q9cm9vdCwgZG93bmxvYWQ9ZG93bmxvYWQpCiAgICByZXR1cm4gbnAuYXNhcnJheShkcy5sYWJlbHMpCgoKZGVmIG1ldHJpY3NfZnJvbV9maWxlbmFtZShwYXRoKToKICAgICIiIlRoZSBgYFtBVUNdeF9bQUNDXXlgYCBwYWlyIGJha2VkIGludG8gdGhlIGZpbGVuYW1lLCBvciBgYE5vbmVgYC4iIiIKICAgIG0gPSBfRk5BTUVfUkUuc2VhcmNoKG9zLnBhdGguYmFzZW5hbWUocGF0aCkpCiAgICByZXR1cm4gKGZsb2F0KG0uZ3JvdXAoImF1YyIpKSwgZmxvYXQobS5ncm91cCgiYWNjIikpKSBpZiBtIGVsc2UgTm9uZQoKCmRlZiB2ZXJpZnlfYWxpZ25tZW50KHBhdGgsIGF1YywgYWNjLCB0b2w9RklMRU5BTUVfVE9MKToKICAgICIiIkNoZWNrIHJlY29tcHV0ZWQgbWV0cmljcyBhZ2FpbnN0IHRoZSBmaWxlbmFtZSdzIOKAlCBjYXRjaGVzIHJvdyBtaXNhbGlnbm1lbnQuCgogICAgUmV0dXJucyBhIGRpY3QgZGVzY3JpYmluZyB0aGUgY2hlY2sgKGBgc3RhdHVzYGAgaXMgYGBva2BgL2Bgc2tpcHBlZGBgKSwgYW5kCiAgICByYWlzZXMgYGBBc3NlcnRpb25FcnJvcmBgIG9uIGEgcmVhbCBtaXNtYXRjaC4KICAgICIiIgogICAgY2xhaW1lZCA9IG1ldHJpY3NfZnJvbV9maWxlbmFtZShwYXRoKQogICAgaWYgY2xhaW1lZCBpcyBOb25lOgogICAgICAgIHJldHVybiBkaWN0KHN0YXR1cz0ic2tpcHBlZCIsIHJlYXNvbj0ibm8gW0FVQ10vW0FDQ10gaW4gZmlsZW5hbWUiKQogICAgY19hdWMsIGNfYWNjID0gY2xhaW1lZAogICAgZF9hdWMsIGRfYWNjID0gYWJzKGF1YyAtIGNfYXVjKSwgYWJzKGFjYyAtIGNfYWNjKQogICAgYXNzZXJ0IGRfYXVjIDw9IHRvbCBhbmQgZF9hY2MgPD0gdG9sLCAoCiAgICAgICAgZiJyZWNvbXB1dGVkIG1ldHJpY3MgZGlzYWdyZWUgd2l0aCB7b3MucGF0aC5iYXNlbmFtZShwYXRoKX06ICIKICAgICAgICBmIkFVQyB7YXVjOi40Zn0gdnMge2NfYXVjOi4zZn0gKM6Ue2RfYXVjOi40Zn0pLCAiCiAgICAgICAgZiJBQ0Mge2FjYzouNGZ9IHZzIHtjX2FjYzouM2Z9ICjOlHtkX2FjYzouNGZ9KS4gIgogICAgICAgICJNb3N0IGxpa2VseSB0aGUgc2NvcmVzIGFuZCBsYWJlbHMgYXJlIG5vdCBpbiB0aGUgc2FtZSBvcmRlci4iKQogICAgcmV0dXJuIGRpY3Qoc3RhdHVzPSJvayIsIGZpbGVuYW1lX2F1Yz1jX2F1YywgZmlsZW5hbWVfYWNjPWNfYWNjLAogICAgICAgICAgICAgICAgZGVsdGFfYXVjPWRfYXVjLCBkZWx0YV9hY2M9ZF9hY2MpCgoKZGVmIGFuYWx5emUoZGF0YXNldCwgc2NvcmVzX3BhdGgsIG91dF9kaXIsIHNwbGl0PSJ0ZXN0IiwgdGFnPSJyZWNvbXB1dGVkIiwKICAgICAgICAgICAgcm9vdD1Ob25lLCBkb3dubG9hZD1UcnVlLCBmaWd1cmVzPVRydWUpOgogICAgIiIiUnVuIGV2ZXJ5IHByZWRpY3Rpb24tb25seSBleHRlbnNpb24gYW5kIHdyaXRlIENTVnMgKyBmaWd1cmVzLgoKICAgIFJldHVybnMgYSBzdW1tYXJ5IGRpY3QgKGFsc28gd3JpdHRlbiB0byBgYDxvdXRfZGlyPi9zdW1tYXJ5Lmpzb25gYCkuCiAgICAiIiIKICAgIGZyb20gLiBpbXBvcnQgZXh0ZW5zaW9ucyBhcyBleHQKICAgIGZyb20gLiBpbXBvcnQgbWV0cmljcyBhcyBteAogICAgZnJvbSAuZGF0YSBpbXBvcnQgZ2V0X2luZm8sIGdldF9kYXRhc2V0CgogICAgb3MubWFrZWRpcnMob3V0X2RpciwgZXhpc3Rfb2s9VHJ1ZSkKICAgIGZpZ19kaXIgPSBvcy5wYXRoLmpvaW4ob3V0X2RpciwgImZpZ3VyZXMiKQoKICAgIHlfc2NvcmUgPSBsb2FkX3Njb3JlcyhzY29yZXNfcGF0aCkKICAgIHlfdHJ1ZSA9IGxvYWRfbGFiZWxzKGRhdGFzZXQsIHNwbGl0LCByb290PXJvb3QsIGRvd25sb2FkPWRvd25sb2FkKQogICAgaWYgbGVuKHlfdHJ1ZSkgIT0gbGVuKHlfc2NvcmUpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoCiAgICAgICAgICAgIGYie2xlbih5X3Njb3JlKX0gc2NvcmUgcm93cyBidXQge2xlbih5X3RydWUpfSB7c3BsaXR9IGxhYmVscyBmb3IgIgogICAgICAgICAgICBmIntkYXRhc2V0fSDigJQgd3Jvbmcgc3BsaXQgb3Igd3JvbmcgZGF0YXNldD8iKQoKICAgIHRhc2sgPSBnZXRfaW5mbyhkYXRhc2V0KVsidGFzayJdCiAgICBhdWMsIGFjYyA9IG14LmV2YWx1YXRlKHlfdHJ1ZSwgeV9zY29yZSwgdGFzaykKICAgIGFsaWdubWVudCA9IHZlcmlmeV9hbGlnbm1lbnQoc2NvcmVzX3BhdGgsIGF1YywgYWNjKQogICAgcHJpbnQoZiJbe3RhZ31dIEFVQz17YXVjOi40Zn0gQUNDPXthY2M6LjRmfSAgKGFsaWdubWVudCBjaGVjazoge2FsaWdubWVudFsnc3RhdHVzJ119KSIpCgogICAgc3VtbWFyeSA9IGRpY3QoZGF0YXNldD1kYXRhc2V0LCBzcGxpdD1zcGxpdCwgdGFnPXRhZywgdGFzaz10YXNrLAogICAgICAgICAgICAgICAgICAgbj1pbnQobGVuKHlfdHJ1ZSkpLCBhdWM9ZmxvYXQoYXVjKSwgYWNjPWZsb2F0KGFjYyksCiAgICAgICAgICAgICAgICAgICBzY29yZXNfcGF0aD1vcy5wYXRoLnJlbHBhdGgoc2NvcmVzX3BhdGgpLAogICAgICAgICAgICAgICAgICAgYWxpZ25tZW50PWFsaWdubWVudCkKCiAgICAjIFBlci1jbGFzcyB0YWJsZSArIGNvbmZ1c2lvbiBtYXRyaXggKyBwZXItY2xhc3MgUk9DLgogICAgdGFibGUgPSBleHQucGVyX2NsYXNzX2FuYWx5c2lzKGRhdGFzZXQsIHlfdHJ1ZSwgeV9zY29yZSwgb3V0X2RpciwgdGFnPXRhZykKICAgIHByaW50KHRhYmxlLnRvX3N0cmluZyhpbmRleD1GYWxzZSkpCgogICAgIyBDYWxpYnJhdGlvbiAvIEVDRSAod3JpdGVzIGNhbGlicmF0aW9uX2VjZS5jc3YpLgogICAgY2FsID0gZXh0LmNhbGlicmF0aW9uX2FuYWx5c2lzKGRhdGFzZXQsIHlfdHJ1ZSwgeV9zY29yZSwgb3V0X2Rpcj1vdXRfZGlyLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHJvb3Q9cm9vdCwgZG93bmxvYWQ9ZG93bmxvYWQpCiAgICBzdW1tYXJ5WyJlY2UiXSA9IHtrOiBmbG9hdChjYWxba11bImVjZSJdKSBmb3IgayBpbiAoIm92ZXJhbGwiLCAiY29tbW9uIiwgInJhcmUiKX0KICAgIHN1bW1hcnlbInJhcmVfY2xhc3NlcyJdID0gY2FsWyJyYXJlX2NsYXNzZXMiXQoKICAgIGlmIGZpZ3VyZXM6CiAgICAgICAgb3MubWFrZWRpcnMoZmlnX2RpciwgZXhpc3Rfb2s9VHJ1ZSkKICAgICAgICBmcmVxID0gZXh0LmZyZXF1ZW5jeV9wZXJmb3JtYW5jZShkYXRhc2V0LCB0YWJsZSwgZmlnX2RpciwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBzdGVtPWYiZXh0X2ZyZXFfcGVyZl97dGFnfSIsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgcm9vdD1yb290LCBkb3dubG9hZD1kb3dubG9hZCkKICAgICAgICBzdW1tYXJ5WyJmcmVxdWVuY3lfcGVyZm9ybWFuY2UiXSA9IHsKICAgICAgICAgICAgazogZmxvYXQodikgZm9yIGssIHYgaW4gZnJlcS5pdGVtcygpIGlmIGsgIT0gInBuZyJ9CiAgICAgICAgZXh0LnBsb3RfcHJfY3VydmVzKGRhdGFzZXQsIHlfdHJ1ZSwgeV9zY29yZSwgZmlnX2RpciwKICAgICAgICAgICAgICAgICAgICAgICAgICAgc3RlbT1mImV4dF9wcl9jdXJ2ZXNfe3RhZ30iKQogICAgICAgIGV4dC5wbG90X3Blcl9jbGFzc19wZXJmb3JtYW5jZShkYXRhc2V0LCB0YWJsZSwgZmlnX2RpciwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgc3RlbT1mImV4dF9wZXJfY2xhc3NfcGVyZl97dGFnfSIsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHJvb3Q9cm9vdCwgZG93bmxvYWQ9ZG93bmxvYWQpCiAgICAgICAgZXh0LnBsb3RfY2xhc3NfZGlzdHJpYnV0aW9uKGRhdGFzZXQsIGZpZ19kaXIsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHN0ZW09ZiJleHRfY2xhc3NfZGlzdHJpYnV0aW9uX3t0YWd9IiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgcm9vdD1yb290LCBkb3dubG9hZD1kb3dubG9hZCkKICAgICAgICBleHQucGxvdF9jYWxpYnJhdGlvbihkYXRhc2V0LCB5X3RydWUsIHlfc2NvcmUsIGZpZ19kaXIsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgc3RlbT1mImV4dF9jYWxpYnJhdGlvbl97dGFnfSIsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgcm9vdD1yb290LCBkb3dubG9hZD1kb3dubG9hZCkKICAgICAgICAjIFRoZSBtaXNjbGFzc2lmaWNhdGlvbiBnYWxsZXJ5IG5lZWRzIHRoZSBpbWFnZXMgdGhlbXNlbHZlcywgbm90IGp1c3QKICAgICAgICAjIHRoZSBzY29yZXMg4oCUIGJ1dCB0aGV5IGNvbWUgc3RyYWlnaHQgZnJvbSB0aGUgbnB6LCBzbyB0aGlzIGlzIHN0aWxsCiAgICAgICAgIyBHUFUtZnJlZS4KICAgICAgICBpbWFnZXMgPSBnZXRfZGF0YXNldChkYXRhc2V0LCBzcGxpdCwgMjgsIHJvb3Q9cm9vdCwgZG93bmxvYWQ9ZG93bmxvYWQpLmltZ3MKICAgICAgICBleHQucGxvdF9taXNjbGFzc2lmaWVkX2dhbGxlcnkoZGF0YXNldCwgeV90cnVlLCB5X3Njb3JlLCBpbWFnZXMsIGZpZ19kaXIsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHN0ZW09ZiJleHRfbWlzY2xhc3NpZmllZF97dGFnfSIsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHJvb3Q9cm9vdCwgZG93bmxvYWQ9ZG93bmxvYWQpCiAgICAgICAgcHJpbnQoZiJbe3RhZ31dIGZpZ3VyZXMgLT4ge2ZpZ19kaXJ9IikKCiAgICB3aXRoIG9wZW4ob3MucGF0aC5qb2luKG91dF9kaXIsICJzdW1tYXJ5Lmpzb24iKSwgInciKSBhcyBmOgogICAgICAgIGpzb24uZHVtcChzdW1tYXJ5LCBmLCBpbmRlbnQ9MikKICAgIHByaW50KGYiW3t0YWd9XSBhcnRpZmFjdHMgLT4ge291dF9kaXJ9IikKICAgIHJldHVybiBzdW1tYXJ5CgoKZGVmIF9wYXJzZV9hcmdzKGFyZ3Y9Tm9uZSk6CiAgICBwID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoZGVzY3JpcHRpb249X19kb2NfXy5zcGxpdGxpbmVzKClbMF0pCiAgICBwLmFkZF9hcmd1bWVudCgiLS1zY29yZXMiLCByZXF1aXJlZD1UcnVlLCBoZWxwPSJwYXRoIHRvIHRoZSBzY29yZSBDU1YiKQogICAgcC5hZGRfYXJndW1lbnQoIi0tZGF0YXNldCIsIGRlZmF1bHQ9ImRlcm1hbW5pc3QiKQogICAgcC5hZGRfYXJndW1lbnQoIi0tc3BsaXQiLCBkZWZhdWx0PSJ0ZXN0IiwgY2hvaWNlcz1bInRyYWluIiwgInZhbCIsICJ0ZXN0Il0pCiAgICBwLmFkZF9hcmd1bWVudCgiLS10YWciLCBkZWZhdWx0PSJyZWNvbXB1dGVkIiwgaGVscD0ic3VmZml4IGZvciBvdXRwdXQgZmlsZW5hbWVzIikKICAgIHAuYWRkX2FyZ3VtZW50KCItLW91dCIsIHJlcXVpcmVkPVRydWUsIGhlbHA9Im91dHB1dCBkaXJlY3RvcnkiKQogICAgcC5hZGRfYXJndW1lbnQoIi0tcm9vdCIsIGRlZmF1bHQ9Tm9uZSwgaGVscD0ibWVkbW5pc3QgZGF0YSByb290IikKICAgIHAuYWRkX2FyZ3VtZW50KCItLW5vLWRvd25sb2FkIiwgZGVzdD0iZG93bmxvYWQiLCBhY3Rpb249InN0b3JlX2ZhbHNlIikKICAgIHAuYWRkX2FyZ3VtZW50KCItLW5vLWZpZ3VyZXMiLCBkZXN0PSJmaWd1cmVzIiwgYWN0aW9uPSJzdG9yZV9mYWxzZSIpCiAgICByZXR1cm4gcC5wYXJzZV9hcmdzKGFyZ3YpCgoKZGVmIG1haW4oYXJndj1Ob25lKToKICAgIGEgPSBfcGFyc2VfYXJncyhhcmd2KQogICAgYW5hbHl6ZShhLmRhdGFzZXQsIGEuc2NvcmVzLCBhLm91dCwgc3BsaXQ9YS5zcGxpdCwgdGFnPWEudGFnLAogICAgICAgICAgICByb290PWEucm9vdCwgZG93bmxvYWQ9YS5kb3dubG9hZCwgZmlndXJlcz1hLmZpZ3VyZXMpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo=',
    'metrics.py': 'IiIiT3VyIG93biBpbXBsZW1lbnRhdGlvbiBvZiB0aGUgTWVkTU5JU1QgZXZhbHVhdGlvbiBtZXRyaWNzLgoKV2UgcmVpbXBsZW1lbnQgbWFjcm8gb25lLXZzLXJlc3QgUk9DLUFVQyBhbmQgYXJnbWF4IGFjY3VyYWN5IGZyb20gc2NyYXRjaCBhbmQKcHJvdmlkZSA6ZnVuYzpgY2hlY2tfYWdyZWVtZW50YCwgd2hpY2ggYXNzZXJ0cyBvdXIgbnVtYmVycyBtYXRjaApgYG1lZG1uaXN0LkV2YWx1YXRvcmBgICh1c2VkICpvbmx5KiBhcyBhbiBvcmFjbGUpIHRvIHdpdGhpbiBhIHRvbGVyYW5jZS4KCk5vdGhpbmcgaGVyZSBpcyBjb3BpZWQgZnJvbSBgYE1lZE1OSVNUL2V4cGVyaW1lbnRzYGA7IHRoZSBvbmx5IE1lZE1OSVNUIGNvZGUgd2UKdG91Y2ggaXMgdGhlIGBgbWVkbW5pc3RgYCBQeVBJIHBhY2thZ2UncyBgYEV2YWx1YXRvcmBgIGluIHRoZSB2ZXJpZmljYXRpb24gcGF0aC4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgbnVtcHkgYXMgbnAKZnJvbSBza2xlYXJuLm1ldHJpY3MgaW1wb3J0IHJvY19hdWNfc2NvcmUsIGFjY3VyYWN5X3Njb3JlCgoKZGVmIF9zcXVlZXplKHlfdHJ1ZSwgeV9zY29yZSk6CiAgICByZXR1cm4gbnAuYXNhcnJheSh5X3RydWUpLnNxdWVlemUoKSwgbnAuYXNhcnJheSh5X3Njb3JlKS5zcXVlZXplKCkKCgpkZWYgZ2V0QVVDKHlfdHJ1ZSwgeV9zY29yZSwgdGFzayk6CiAgICAiIiJNYWNyby1hdmVyYWdlZCBBVUMuCgogICAgRm9yIG11bHRpLWNsYXNzIHRhc2tzIHRoaXMgaXMgdGhlIG1lYW4gb3ZlciBjbGFzc2VzIG9mIHRoZSBvbmUtdnMtcmVzdAogICAgUk9DLUFVQyAoYGAoeV90cnVlID09IGMpYGAgdnMgYGB5X3Njb3JlWzosIGNdYGApLCB3aGljaCBpcyBleGFjdGx5IHdoYXQKICAgIHNjaWtpdC1sZWFybidzIGBgcm9jX2F1Y19zY29yZShhdmVyYWdlPSdtYWNybycsIG11bHRpX2NsYXNzPSdvdnInKWBgIGFuZAogICAgYGBtZWRtbmlzdC5FdmFsdWF0b3JgYCBjb21wdXRlLiBgYG11bHRpLWxhYmVsLCBiaW5hcnktY2xhc3NgYCBhdmVyYWdlcyB0aGUKICAgIHBlci1sYWJlbCBiaW5hcnkgQVVDOyBgYGJpbmFyeS1jbGFzc2BgIHVzZXMgdGhlIHBvc2l0aXZlLWNsYXNzIHNjb3JlLgogICAgIiIiCiAgICB5X3RydWUsIHlfc2NvcmUgPSBfc3F1ZWV6ZSh5X3RydWUsIHlfc2NvcmUpCgogICAgaWYgdGFzayA9PSAibXVsdGktbGFiZWwsIGJpbmFyeS1jbGFzcyI6CiAgICAgICAgYXVjcyA9IFtyb2NfYXVjX3Njb3JlKHlfdHJ1ZVs6LCBpXSwgeV9zY29yZVs6LCBpXSkgZm9yIGkgaW4gcmFuZ2UoeV9zY29yZS5zaGFwZVsxXSldCiAgICAgICAgcmV0dXJuIGZsb2F0KG5wLm1lYW4oYXVjcykpCgogICAgaWYgdGFzayA9PSAiYmluYXJ5LWNsYXNzIjoKICAgICAgICBpZiB5X3Njb3JlLm5kaW0gPT0gMjoKICAgICAgICAgICAgeV9zY29yZSA9IHlfc2NvcmVbOiwgLTFdCiAgICAgICAgcmV0dXJuIGZsb2F0KHJvY19hdWNfc2NvcmUoeV90cnVlLCB5X3Njb3JlKSkKCiAgICAjIG11bHRpLWNsYXNzIC8gb3JkaW5hbC1yZWdyZXNzaW9uOiBtYWNybyBvbmUtdnMtcmVzdC4KICAgIGF1Y3MgPSBbXQogICAgZm9yIGMgaW4gcmFuZ2UoeV9zY29yZS5zaGFwZVsxXSk6CiAgICAgICAgYXVjcy5hcHBlbmQocm9jX2F1Y19zY29yZSgoeV90cnVlID09IGMpLmFzdHlwZShmbG9hdCksIHlfc2NvcmVbOiwgY10pKQogICAgcmV0dXJuIGZsb2F0KG5wLm1lYW4oYXVjcykpCgoKZGVmIGdldEFDQyh5X3RydWUsIHlfc2NvcmUsIHRhc2ssIHRocmVzaG9sZD0wLjUpOgogICAgIiIiQWNjdXJhY3kuIEFyZ21heCBmb3IgbXVsdGktY2xhc3M7IHBlci1sYWJlbCB0aHJlc2hvbGRpbmcgb3RoZXJ3aXNlLiIiIgogICAgeV90cnVlLCB5X3Njb3JlID0gX3NxdWVlemUoeV90cnVlLCB5X3Njb3JlKQoKICAgIGlmIHRhc2sgPT0gIm11bHRpLWxhYmVsLCBiaW5hcnktY2xhc3MiOgogICAgICAgIHlfcHJlZCA9IHlfc2NvcmUgPiB0aHJlc2hvbGQKICAgICAgICBhY2NzID0gW2FjY3VyYWN5X3Njb3JlKHlfdHJ1ZVs6LCBpXSwgeV9wcmVkWzosIGldKSBmb3IgaSBpbiByYW5nZSh5X3RydWUuc2hhcGVbMV0pXQogICAgICAgIHJldHVybiBmbG9hdChucC5tZWFuKGFjY3MpKQoKICAgIGlmIHRhc2sgPT0gImJpbmFyeS1jbGFzcyI6CiAgICAgICAgaWYgeV9zY29yZS5uZGltID09IDI6CiAgICAgICAgICAgIHlfc2NvcmUgPSB5X3Njb3JlWzosIC0xXQogICAgICAgIHJldHVybiBmbG9hdChhY2N1cmFjeV9zY29yZSh5X3RydWUsIHlfc2NvcmUgPiB0aHJlc2hvbGQpKQoKICAgIHJldHVybiBmbG9hdChhY2N1cmFjeV9zY29yZSh5X3RydWUsIG5wLmFyZ21heCh5X3Njb3JlLCBheGlzPS0xKSkpCgoKZGVmIGV2YWx1YXRlKHlfdHJ1ZSwgeV9zY29yZSwgdGFzayk6CiAgICAiIiJSZXR1cm4gYGAoYXVjLCBhY2MpYGAgZm9yIGEgc2V0IG9mIHByZWRpY3Rpb25zLiIiIgogICAgcmV0dXJuIGdldEFVQyh5X3RydWUsIHlfc2NvcmUsIHRhc2spLCBnZXRBQ0MoeV90cnVlLCB5X3Njb3JlLCB0YXNrKQoKCmRlZiBwZXJfY2xhc3NfbWV0cmljcyh5X3RydWUsIHlfc2NvcmUsIG51bV9jbGFzc2VzLCB5X3ByZWQ9Tm9uZSk6CiAgICAiIiJQZXItY2xhc3MgQVVDIC8gcHJlY2lzaW9uIC8gcmVjYWxsIC8gRjEgKG11bHRpLWNsYXNzIG9ubHkpLgoKICAgIGBgeV9wcmVkYGAgZGVmYXVsdHMgdG8gYXJnbWF4IG92ZXIgYGB5X3Njb3JlYGAgYnV0IGNhbiBiZSBzdXBwbGllZAogICAgZGlyZWN0bHkgKGUuZy4gZnJvbSBwZXItY2xhc3MgdGhyZXNob2xkIHR1bmluZykgc28gcHJlY2lzaW9uL3JlY2FsbC9GMQogICAgcmVmbGVjdCBhIGRpZmZlcmVudCBkZWNpc2lvbiBydWxlIHdoaWxlIEFVQy9BUCBzdGlsbCByZWFkIG9mZiB0aGUKICAgIHRocmVzaG9sZC1mcmVlIHNjb3Jlcy4KCiAgICBSZXR1cm5zIGEgbGlzdCBvZiBkaWN0cywgb25lIHBlciBjbGFzcy4gVXNlZCBieSB0aGUgcGVyLWNsYXNzIGV4dGVuc2lvbi4KICAgICIiIgogICAgZnJvbSBza2xlYXJuLm1ldHJpY3MgaW1wb3J0IHByZWNpc2lvbl9yZWNhbGxfZnNjb3JlX3N1cHBvcnQsIGF2ZXJhZ2VfcHJlY2lzaW9uX3Njb3JlCgogICAgeV90cnVlLCB5X3Njb3JlID0gX3NxdWVlemUoeV90cnVlLCB5X3Njb3JlKQogICAgaWYgeV9wcmVkIGlzIE5vbmU6CiAgICAgICAgeV9wcmVkID0gbnAuYXJnbWF4KHlfc2NvcmUsIGF4aXM9LTEpCiAgICBlbHNlOgogICAgICAgIHlfcHJlZCA9IG5wLmFzYXJyYXkoeV9wcmVkKS5zcXVlZXplKCkKICAgIHByZWNpc2lvbiwgcmVjYWxsLCBmMSwgc3VwcG9ydCA9IHByZWNpc2lvbl9yZWNhbGxfZnNjb3JlX3N1cHBvcnQoCiAgICAgICAgeV90cnVlLCB5X3ByZWQsIGxhYmVscz1saXN0KHJhbmdlKG51bV9jbGFzc2VzKSksIHplcm9fZGl2aXNpb249MAogICAgKQogICAgcm93cyA9IFtdCiAgICBmb3IgYyBpbiByYW5nZShudW1fY2xhc3Nlcyk6CiAgICAgICAgeV9iaW4gPSAoeV90cnVlID09IGMpLmFzdHlwZShmbG9hdCkKICAgICAgICB0cnk6CiAgICAgICAgICAgIGF1Y19jID0gcm9jX2F1Y19zY29yZSh5X2JpbiwgeV9zY29yZVs6LCBjXSkKICAgICAgICBleGNlcHQgVmFsdWVFcnJvcjogICMgYSBjbGFzcyBhYnNlbnQgZnJvbSB0aGlzIHNwbGl0CiAgICAgICAgICAgIGF1Y19jID0gZmxvYXQoIm5hbiIpCiAgICAgICAgdHJ5OgogICAgICAgICAgICAjIFBSLUFVQyAoYXZlcmFnZSBwcmVjaXNpb24pOiB0aGUgaG9uZXN0IHJlYWQgdW5kZXIgaW1iYWxhbmNlLCB3aGVyZQogICAgICAgICAgICAjIFJPQy1BVUMgaXMgb3B0aW1pc3RpYyBmb3IgdGhlIHJhcmUgY2xhc3Nlcy4KICAgICAgICAgICAgYXBfYyA9IGF2ZXJhZ2VfcHJlY2lzaW9uX3Njb3JlKHlfYmluLCB5X3Njb3JlWzosIGNdKQogICAgICAgIGV4Y2VwdCBWYWx1ZUVycm9yOgogICAgICAgICAgICBhcF9jID0gZmxvYXQoIm5hbiIpCiAgICAgICAgcm93cy5hcHBlbmQoCiAgICAgICAgICAgIGRpY3QoCiAgICAgICAgICAgICAgICBjbHM9aW50KGMpLAogICAgICAgICAgICAgICAgYXVjPWZsb2F0KGF1Y19jKSwKICAgICAgICAgICAgICAgIGFwPWZsb2F0KGFwX2MpLAogICAgICAgICAgICAgICAgcHJlY2lzaW9uPWZsb2F0KHByZWNpc2lvbltjXSksCiAgICAgICAgICAgICAgICByZWNhbGw9ZmxvYXQocmVjYWxsW2NdKSwKICAgICAgICAgICAgICAgIGYxPWZsb2F0KGYxW2NdKSwKICAgICAgICAgICAgICAgIHN1cHBvcnQ9aW50KHN1cHBvcnRbY10pLAogICAgICAgICAgICApCiAgICAgICAgKQogICAgcmV0dXJuIHJvd3MKCgpkZWYgY2hlY2tfYWdyZWVtZW50KHlfdHJ1ZSwgeV9zY29yZSwgdGFzaywgZmxhZz1Ob25lLCBzcGxpdD1Ob25lLCBzaXplPTI4LAogICAgICAgICAgICAgICAgICAgIHJvb3Q9Tm9uZSwgdG9sPTFlLTMsIHZlcmJvc2U9VHJ1ZSk6CiAgICAiIiJBc3NlcnQgb3VyIG1ldHJpY3MgYWdyZWUgd2l0aCBgYG1lZG1uaXN0LkV2YWx1YXRvcmBgIHdpdGhpbiBgYHRvbGBgLgoKICAgIFR3byBpbmRlcGVuZGVudCBvcmFjbGUgcGF0aHMgYXJlIGNoZWNrZWQ6CgogICAgMS4gYGBtZWRtbmlzdC5ldmFsdWF0b3IuZ2V0QVVDL2dldEFDQ2BgIG9uIHRoZSBzYW1lIGFycmF5cy4KICAgIDIuIElmIGBgZmxhZ2BgL2Bgc3BsaXRgYCBhcmUgZ2l2ZW4sIGEgcmVhbCBgYG1lZG1uaXN0LkV2YWx1YXRvcmBgIG9iamVjdAogICAgICAgKHdoaWNoIHJlbG9hZHMgbGFiZWxzIGZyb20gdGhlIGBgLm5wemBgIG9uIGRpc2spIGV2YWx1YXRpbmcgYGB5X3Njb3JlYGAuCgogICAgUmFpc2VzIGBgQXNzZXJ0aW9uRXJyb3JgYCBpZiBlaXRoZXIgZHJpZnRzIGJleW9uZCBgYHRvbGBgLgogICAgIiIiCiAgICBmcm9tIG1lZG1uaXN0LmV2YWx1YXRvciBpbXBvcnQgZ2V0QVVDIGFzIHJlZl9nZXRBVUMsIGdldEFDQyBhcyByZWZfZ2V0QUNDCgogICAgb3VyX2F1YyA9IGdldEFVQyh5X3RydWUsIHlfc2NvcmUsIHRhc2spCiAgICBvdXJfYWNjID0gZ2V0QUNDKHlfdHJ1ZSwgeV9zY29yZSwgdGFzaykKCiAgICByZWZfYXVjID0gcmVmX2dldEFVQyhucC5hc2FycmF5KHlfdHJ1ZSksIG5wLmFzYXJyYXkoeV9zY29yZSksIHRhc2spCiAgICByZWZfYWNjID0gcmVmX2dldEFDQyhucC5hc2FycmF5KHlfdHJ1ZSksIG5wLmFzYXJyYXkoeV9zY29yZSksIHRhc2spCgogICAgYXNzZXJ0IGFicyhvdXJfYXVjIC0gcmVmX2F1YykgPCB0b2wsIGYiQVVDIGRyaWZ0OiBvdXJzPXtvdXJfYXVjOi42Zn0gcmVmPXtyZWZfYXVjOi42Zn0iCiAgICBhc3NlcnQgYWJzKG91cl9hY2MgLSByZWZfYWNjKSA8IHRvbCwgZiJBQ0MgZHJpZnQ6IG91cnM9e291cl9hY2M6LjZmfSByZWY9e3JlZl9hY2M6LjZmfSIKCiAgICBldmFsX2F1YyA9IGV2YWxfYWNjID0gTm9uZQogICAgaWYgZmxhZyBpcyBub3QgTm9uZSBhbmQgc3BsaXQgaXMgbm90IE5vbmU6CiAgICAgICAgZnJvbSBtZWRtbmlzdCBpbXBvcnQgRXZhbHVhdG9yCgogICAgICAgIGt3YXJncyA9IHt9IGlmIHJvb3QgaXMgTm9uZSBlbHNlIHsicm9vdCI6IHJvb3R9CiAgICAgICAgZXZhbHVhdG9yID0gRXZhbHVhdG9yKGZsYWcsIHNwbGl0LCBzaXplPXNpemUsICoqa3dhcmdzKQogICAgICAgIG0gPSBldmFsdWF0b3IuZXZhbHVhdGUobnAuYXNhcnJheSh5X3Njb3JlKSkKICAgICAgICBldmFsX2F1YywgZXZhbF9hY2MgPSBmbG9hdChtLkFVQyksIGZsb2F0KG0uQUNDKQogICAgICAgIGFzc2VydCBhYnMob3VyX2F1YyAtIGV2YWxfYXVjKSA8IHRvbCwgZiJBVUMgZHJpZnQgdnMgRXZhbHVhdG9yOiBvdXJzPXtvdXJfYXVjOi42Zn0gcmVmPXtldmFsX2F1YzouNmZ9IgogICAgICAgIGFzc2VydCBhYnMob3VyX2FjYyAtIGV2YWxfYWNjKSA8IHRvbCwgZiJBQ0MgZHJpZnQgdnMgRXZhbHVhdG9yOiBvdXJzPXtvdXJfYWNjOi42Zn0gcmVmPXtldmFsX2FjYzouNmZ9IgoKICAgIGlmIHZlcmJvc2U6CiAgICAgICAgcHJpbnQoZiJbbWV0cmljc10gYWdyZWVtZW50IE9LICBvdXJzPSh7b3VyX2F1YzouNGZ9LHtvdXJfYWNjOi40Zn0pICIKICAgICAgICAgICAgICBmImdldFg9KHtyZWZfYXVjOi40Zn0se3JlZl9hY2M6LjRmfSkiCiAgICAgICAgICAgICAgKyAoZiIgRXZhbHVhdG9yPSh7ZXZhbF9hdWM6LjRmfSx7ZXZhbF9hY2M6LjRmfSkiIGlmIGV2YWxfYXVjIGlzIG5vdCBOb25lIGVsc2UgIiIpKQogICAgcmV0dXJuIGRpY3QoYXVjPW91cl9hdWMsIGFjYz1vdXJfYWNjKQo=',
    'metrics_test.py': 'IiIiU3RhbmRhbG9uZSBhZ3JlZW1lbnQgdGVzdDogb3VyIG1ldHJpY3MgdnMgdGhlIG1lZG1uaXN0IG9yYWNsZS4KClJ1biB3aXRoOiAgcHl0aG9uIC1tIHNyYy5tZXRyaWNzX3Rlc3QKVXNlcyBzeW50aGV0aWMgcHJlZGljdGlvbnMgb25seSAobm8gdG9yY2ggLyBubyBkYXRhc2V0IGRvd25sb2FkIG5lZWRlZCkuCkV4aXRzIG5vbi16ZXJvIGlmIGFueSBtZXRyaWMgZHJpZnRzIGJleW9uZCAxZS02IGZyb20gbWVkbW5pc3QncyBnZXRBVUMvZ2V0QUNDLgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBudW1weSBhcyBucAoKZnJvbSAuIGltcG9ydCBtZXRyaWNzCgoKZGVmIG1haW4oKToKICAgIHJuZyA9IG5wLnJhbmRvbS5SYW5kb21TdGF0ZSgwKQoKICAgICMgbXVsdGktY2xhc3MgKERlcm1hTU5JU1Q9NywgUGF0aE1OSVNUPTkpCiAgICBmb3IgQyBpbiAoNywgOSk6CiAgICAgICAgTiA9IDMwMDAKICAgICAgICB5X3RydWUgPSBybmcucmFuZGludCgwLCBDLCBzaXplPShOLCAxKSkKICAgICAgICBsb2dpdHMgPSBybmcucmFuZG4oTiwgQykgKyBucC5leWUoQylbeV90cnVlLnNxdWVlemUoKV0gKiAxLjUKICAgICAgICBlID0gbnAuZXhwKGxvZ2l0cyAtIGxvZ2l0cy5tYXgoMSwga2VlcGRpbXM9VHJ1ZSkpCiAgICAgICAgeV9zY29yZSA9IGUgLyBlLnN1bSgxLCBrZWVwZGltcz1UcnVlKQogICAgICAgIG1ldHJpY3MuY2hlY2tfYWdyZWVtZW50KHlfdHJ1ZSwgeV9zY29yZSwgIm11bHRpLWNsYXNzIiwgdG9sPTFlLTYpCgogICAgIyBtdWx0aS1sYWJlbCwgYmluYXJ5LWNsYXNzIChDaGVzdE1OSVNUPTE0KQogICAgTCwgTiA9IDE0LCAyMDAwCiAgICB5X3RydWUgPSAocm5nLnJhbmQoTiwgTCkgPiAwLjcpLmFzdHlwZShpbnQpCiAgICB5X3Njb3JlID0gcm5nLnJhbmQoTiwgTCkKICAgIG1ldHJpY3MuY2hlY2tfYWdyZWVtZW50KHlfdHJ1ZSwgeV9zY29yZSwgIm11bHRpLWxhYmVsLCBiaW5hcnktY2xhc3MiLCB0b2w9MWUtNikKCiAgICAjIGJpbmFyeS1jbGFzcwogICAgeV90cnVlID0gcm5nLnJhbmRpbnQoMCwgMiwgc2l6ZT0oTiwgMSkpCiAgICB5X3Njb3JlID0gcm5nLnJhbmQoTiwgMikKICAgIHlfc2NvcmUgPSB5X3Njb3JlIC8geV9zY29yZS5zdW0oMSwga2VlcGRpbXM9VHJ1ZSkKICAgIG1ldHJpY3MuY2hlY2tfYWdyZWVtZW50KHlfdHJ1ZSwgeV9zY29yZSwgImJpbmFyeS1jbGFzcyIsIHRvbD0xZS02KQoKICAgIHByaW50KCJBTEwgTUVUUklDIEFHUkVFTUVOVCBURVNUUyBQQVNTRUQgKHRvbD0xZS02KSIpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo=',
    'models.py': 'IiIiTW9kZWwgZGVmaW5pdGlvbnMsIHJlaW1wbGVtZW50ZWQgZnJvbSBzY3JhdGNoLgoKVHdvIGJhY2tib25lcywgY2hvc2VuIGJ5IGlucHV0IHJlc29sdXRpb24gKHRoaXMgaXMgZGVsaWJlcmF0ZSBhbmQgbWF0Y2hlcyB0aGUKTWVkTU5JU1QgdjIgcHJvdG9jb2wpOgoKKiBgYHNpemUgPT0gMjhgYCAgLT4gYSBDSUZBUi1zdHlsZSBSZXNOZXQ6IDN4MyBzdHJpZGUtMSBzdGVtLCAqbm8qIG1heC1wb29sLAogIGZvdXIgcmVzaWR1YWwgc3RhZ2VzIHdpdGggc3RyaWRlcyBgYFsxLCAyLCAyLCAyXWBgIGFuZCB3aWR0aHMKICBgYFs2NCwgMTI4LCAyNTYsIDUxMl1gYC4gUmVzTmV0LTE4IHVzZXMgYGBCYXNpY0Jsb2NrYGAgd2l0aCBgYFsyLDIsMiwyXWBgOwogIFJlc05ldC01MCB1c2VzIGBgQm90dGxlbmVja2BgIChleHBhbnNpb24gNCkgd2l0aCBgYFszLDQsNiwzXWBgLgoqIGBgc2l6ZSA9PSAyMjRgYCAtPiBgYHRvcmNodmlzaW9uLm1vZGVscy5yZXNuZXQxOC81MGBgIHdpdGggdGhlIHN0YW5kYXJkCiAgSW1hZ2VOZXQgN3g3IHN0cmlkZS0yIHN0ZW0gKyBtYXgtcG9vbC4KCkEgYGB3aWR0aF9tdWx0YGAga25vYiBnaXZlcyB0aGUgbGlnaHR3ZWlnaHQgMC41eCB2YXJpYW50ICh3aWR0aHMKYGBbMzIsNjQsMTI4LDI1Nl1gYCkgdXNlZCBieSB0aGUgZWZmaWNpZW5jeSBleHRlbnNpb24uCgpOb25lIG9mIHRoaXMgaXMgY29waWVkIGZyb20gYGBNZWRNTklTVC9leHBlcmltZW50c2BgIG9yIGBga3VhbmdsaXUvcHl0b3JjaC1jaWZhcmBgOwppdCBpcyB3cml0dGVuIHRvIHRoZSBzcGVjIGFib3ZlLgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCB0b3JjaAppbXBvcnQgdG9yY2gubm4gYXMgbm4KaW1wb3J0IHRvcmNoLm5uLmZ1bmN0aW9uYWwgYXMgRgoKCmRlZiBjb252M3gzKGluX3BsYW5lcywgb3V0X3BsYW5lcywgc3RyaWRlPTEpOgogICAgcmV0dXJuIG5uLkNvbnYyZChpbl9wbGFuZXMsIG91dF9wbGFuZXMsIGtlcm5lbF9zaXplPTMsIHN0cmlkZT1zdHJpZGUsCiAgICAgICAgICAgICAgICAgICAgIHBhZGRpbmc9MSwgYmlhcz1GYWxzZSkKCgpkZWYgY29udjF4MShpbl9wbGFuZXMsIG91dF9wbGFuZXMsIHN0cmlkZT0xKToKICAgIHJldHVybiBubi5Db252MmQoaW5fcGxhbmVzLCBvdXRfcGxhbmVzLCBrZXJuZWxfc2l6ZT0xLCBzdHJpZGU9c3RyaWRlLCBiaWFzPUZhbHNlKQoKCmNsYXNzIEJhc2ljQmxvY2sobm4uTW9kdWxlKToKICAgIGV4cGFuc2lvbiA9IDEKCiAgICBkZWYgX19pbml0X18oc2VsZiwgaW5fcGxhbmVzLCBwbGFuZXMsIHN0cmlkZT0xLCBkb3duc2FtcGxlPU5vbmUpOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYuY29udjEgPSBjb252M3gzKGluX3BsYW5lcywgcGxhbmVzLCBzdHJpZGUpCiAgICAgICAgc2VsZi5ibjEgPSBubi5CYXRjaE5vcm0yZChwbGFuZXMpCiAgICAgICAgc2VsZi5jb252MiA9IGNvbnYzeDMocGxhbmVzLCBwbGFuZXMpCiAgICAgICAgc2VsZi5ibjIgPSBubi5CYXRjaE5vcm0yZChwbGFuZXMpCiAgICAgICAgc2VsZi5kb3duc2FtcGxlID0gZG93bnNhbXBsZQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIHgpOgogICAgICAgIGlkZW50aXR5ID0geAogICAgICAgIG91dCA9IEYucmVsdShzZWxmLmJuMShzZWxmLmNvbnYxKHgpKSwgaW5wbGFjZT1UcnVlKQogICAgICAgIG91dCA9IHNlbGYuYm4yKHNlbGYuY29udjIob3V0KSkKICAgICAgICBpZiBzZWxmLmRvd25zYW1wbGUgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIGlkZW50aXR5ID0gc2VsZi5kb3duc2FtcGxlKHgpCiAgICAgICAgcmV0dXJuIEYucmVsdShvdXQgKyBpZGVudGl0eSwgaW5wbGFjZT1UcnVlKQoKCmNsYXNzIEJvdHRsZW5lY2sobm4uTW9kdWxlKToKICAgIGV4cGFuc2lvbiA9IDQKCiAgICBkZWYgX19pbml0X18oc2VsZiwgaW5fcGxhbmVzLCBwbGFuZXMsIHN0cmlkZT0xLCBkb3duc2FtcGxlPU5vbmUpOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYuY29udjEgPSBjb252MXgxKGluX3BsYW5lcywgcGxhbmVzKQogICAgICAgIHNlbGYuYm4xID0gbm4uQmF0Y2hOb3JtMmQocGxhbmVzKQogICAgICAgIHNlbGYuY29udjIgPSBjb252M3gzKHBsYW5lcywgcGxhbmVzLCBzdHJpZGUpCiAgICAgICAgc2VsZi5ibjIgPSBubi5CYXRjaE5vcm0yZChwbGFuZXMpCiAgICAgICAgc2VsZi5jb252MyA9IGNvbnYxeDEocGxhbmVzLCBwbGFuZXMgKiBzZWxmLmV4cGFuc2lvbikKICAgICAgICBzZWxmLmJuMyA9IG5uLkJhdGNoTm9ybTJkKHBsYW5lcyAqIHNlbGYuZXhwYW5zaW9uKQogICAgICAgIHNlbGYuZG93bnNhbXBsZSA9IGRvd25zYW1wbGUKCiAgICBkZWYgZm9yd2FyZChzZWxmLCB4KToKICAgICAgICBpZGVudGl0eSA9IHgKICAgICAgICBvdXQgPSBGLnJlbHUoc2VsZi5ibjEoc2VsZi5jb252MSh4KSksIGlucGxhY2U9VHJ1ZSkKICAgICAgICBvdXQgPSBGLnJlbHUoc2VsZi5ibjIoc2VsZi5jb252MihvdXQpKSwgaW5wbGFjZT1UcnVlKQogICAgICAgIG91dCA9IHNlbGYuYm4zKHNlbGYuY29udjMob3V0KSkKICAgICAgICBpZiBzZWxmLmRvd25zYW1wbGUgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIGlkZW50aXR5ID0gc2VsZi5kb3duc2FtcGxlKHgpCiAgICAgICAgcmV0dXJuIEYucmVsdShvdXQgKyBpZGVudGl0eSwgaW5wbGFjZT1UcnVlKQoKCmNsYXNzIENpZmFyUmVzTmV0KG5uLk1vZHVsZSk6CiAgICAiIiJDSUZBUi1zdHlsZSBSZXNOZXQgZm9yIDI4eDI4IGlucHV0cyAoM3gzIHN0ZW0sIG5vIG1heC1wb29sKS4iIiIKCiAgICBkZWYgX19pbml0X18oc2VsZiwgYmxvY2ssIGxheWVycywgbnVtX2NsYXNzZXMsIGluX2NoYW5uZWxzPTMsIHdpZHRoX211bHQ9MS4wKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICB3aWR0aHMgPSBbaW50KHcgKiB3aWR0aF9tdWx0KSBmb3IgdyBpbiAoNjQsIDEyOCwgMjU2LCA1MTIpXQogICAgICAgIHNlbGYuaW5fcGxhbmVzID0gd2lkdGhzWzBdCgogICAgICAgIHNlbGYuY29udjEgPSBubi5Db252MmQoaW5fY2hhbm5lbHMsIHdpZHRoc1swXSwga2VybmVsX3NpemU9Mywgc3RyaWRlPTEsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBwYWRkaW5nPTEsIGJpYXM9RmFsc2UpCiAgICAgICAgc2VsZi5ibjEgPSBubi5CYXRjaE5vcm0yZCh3aWR0aHNbMF0pCgogICAgICAgIHNlbGYubGF5ZXIxID0gc2VsZi5fbWFrZV9sYXllcihibG9jaywgd2lkdGhzWzBdLCBsYXllcnNbMF0sIHN0cmlkZT0xKQogICAgICAgIHNlbGYubGF5ZXIyID0gc2VsZi5fbWFrZV9sYXllcihibG9jaywgd2lkdGhzWzFdLCBsYXllcnNbMV0sIHN0cmlkZT0yKQogICAgICAgIHNlbGYubGF5ZXIzID0gc2VsZi5fbWFrZV9sYXllcihibG9jaywgd2lkdGhzWzJdLCBsYXllcnNbMl0sIHN0cmlkZT0yKQogICAgICAgIHNlbGYubGF5ZXI0ID0gc2VsZi5fbWFrZV9sYXllcihibG9jaywgd2lkdGhzWzNdLCBsYXllcnNbM10sIHN0cmlkZT0yKQoKICAgICAgICBzZWxmLmF2Z3Bvb2wgPSBubi5BZGFwdGl2ZUF2Z1Bvb2wyZCgoMSwgMSkpCiAgICAgICAgc2VsZi5mYyA9IG5uLkxpbmVhcih3aWR0aHNbM10gKiBibG9jay5leHBhbnNpb24sIG51bV9jbGFzc2VzKQoKICAgICAgICBmb3IgbSBpbiBzZWxmLm1vZHVsZXMoKToKICAgICAgICAgICAgaWYgaXNpbnN0YW5jZShtLCBubi5Db252MmQpOgogICAgICAgICAgICAgICAgbm4uaW5pdC5rYWltaW5nX25vcm1hbF8obS53ZWlnaHQsIG1vZGU9ImZhbl9vdXQiLCBub25saW5lYXJpdHk9InJlbHUiKQogICAgICAgICAgICBlbGlmIGlzaW5zdGFuY2UobSwgbm4uQmF0Y2hOb3JtMmQpOgogICAgICAgICAgICAgICAgbm4uaW5pdC5jb25zdGFudF8obS53ZWlnaHQsIDEpCiAgICAgICAgICAgICAgICBubi5pbml0LmNvbnN0YW50XyhtLmJpYXMsIDApCgogICAgZGVmIF9tYWtlX2xheWVyKHNlbGYsIGJsb2NrLCBwbGFuZXMsIGJsb2Nrcywgc3RyaWRlKToKICAgICAgICBkb3duc2FtcGxlID0gTm9uZQogICAgICAgIGlmIHN0cmlkZSAhPSAxIG9yIHNlbGYuaW5fcGxhbmVzICE9IHBsYW5lcyAqIGJsb2NrLmV4cGFuc2lvbjoKICAgICAgICAgICAgZG93bnNhbXBsZSA9IG5uLlNlcXVlbnRpYWwoCiAgICAgICAgICAgICAgICBjb252MXgxKHNlbGYuaW5fcGxhbmVzLCBwbGFuZXMgKiBibG9jay5leHBhbnNpb24sIHN0cmlkZSksCiAgICAgICAgICAgICAgICBubi5CYXRjaE5vcm0yZChwbGFuZXMgKiBibG9jay5leHBhbnNpb24pLAogICAgICAgICAgICApCiAgICAgICAgbGF5ZXJzID0gW2Jsb2NrKHNlbGYuaW5fcGxhbmVzLCBwbGFuZXMsIHN0cmlkZSwgZG93bnNhbXBsZSldCiAgICAgICAgc2VsZi5pbl9wbGFuZXMgPSBwbGFuZXMgKiBibG9jay5leHBhbnNpb24KICAgICAgICBmb3IgXyBpbiByYW5nZSgxLCBibG9ja3MpOgogICAgICAgICAgICBsYXllcnMuYXBwZW5kKGJsb2NrKHNlbGYuaW5fcGxhbmVzLCBwbGFuZXMpKQogICAgICAgIHJldHVybiBubi5TZXF1ZW50aWFsKCpsYXllcnMpCgogICAgZGVmIGZvcndhcmQoc2VsZiwgeCk6CiAgICAgICAgeCA9IEYucmVsdShzZWxmLmJuMShzZWxmLmNvbnYxKHgpKSwgaW5wbGFjZT1UcnVlKQogICAgICAgIHggPSBzZWxmLmxheWVyMSh4KQogICAgICAgIHggPSBzZWxmLmxheWVyMih4KQogICAgICAgIHggPSBzZWxmLmxheWVyMyh4KQogICAgICAgIHggPSBzZWxmLmxheWVyNCh4KQogICAgICAgIHggPSBzZWxmLmF2Z3Bvb2woeCkKICAgICAgICB4ID0gdG9yY2guZmxhdHRlbih4LCAxKQogICAgICAgIHJldHVybiBzZWxmLmZjKHgpCgoKZGVmIGJ1aWxkX21vZGVsKG1vZGVsX25hbWUsIHNpemUsIG51bV9jbGFzc2VzLCBpbl9jaGFubmVscz0zLCB3aWR0aF9tdWx0PTEuMCk6CiAgICAiIiJCdWlsZCBhIG1vZGVsIGZvciB0aGUgZ2l2ZW4gYXJjaGl0ZWN0dXJlIC8gcmVzb2x1dGlvbi4KCiAgICBBcmdzOgogICAgICAgIG1vZGVsX25hbWU6IGBgInJlc25ldDE4ImBgIG9yIGBgInJlc25ldDUwImBgLgogICAgICAgIHNpemU6IDI4IChDSUZBUi1zdHlsZSkgb3IgMjI0ICh0b3JjaHZpc2lvbiBJbWFnZU5ldCBiYWNrYm9uZSkuCiAgICAgICAgbnVtX2NsYXNzZXM6IG51bWJlciBvZiBvdXRwdXQgbG9naXRzLgogICAgICAgIGluX2NoYW5uZWxzOiBpbnB1dCBjaGFubmVscyAoMyBmb3IgdGhlIFJHQiBNZWRNTklTVCBkYXRhc2V0cyB3ZSB1c2UpLgogICAgICAgIHdpZHRoX211bHQ6IGNoYW5uZWwgbXVsdGlwbGllcjsgMC41IGdpdmVzIHRoZSBsaWdodHdlaWdodCB2YXJpYW50LgogICAgICAgICAgICAgICAgICAgIE9ubHkgc3VwcG9ydGVkIGF0IHNpemUgMjguCiAgICAiIiIKICAgIG1vZGVsX25hbWUgPSBtb2RlbF9uYW1lLmxvd2VyKCkKICAgIGlmIG1vZGVsX25hbWUgbm90IGluICgicmVzbmV0MTgiLCAicmVzbmV0NTAiKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYidW5rbm93biBtb2RlbCB7bW9kZWxfbmFtZSFyfSIpCgogICAgaWYgc2l6ZSA9PSAyODoKICAgICAgICBpZiBtb2RlbF9uYW1lID09ICJyZXNuZXQxOCI6CiAgICAgICAgICAgIHJldHVybiBDaWZhclJlc05ldChCYXNpY0Jsb2NrLCBbMiwgMiwgMiwgMl0sIG51bV9jbGFzc2VzLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgaW5fY2hhbm5lbHM9aW5fY2hhbm5lbHMsIHdpZHRoX211bHQ9d2lkdGhfbXVsdCkKICAgICAgICByZXR1cm4gQ2lmYXJSZXNOZXQoQm90dGxlbmVjaywgWzMsIDQsIDYsIDNdLCBudW1fY2xhc3NlcywKICAgICAgICAgICAgICAgICAgICAgICAgICAgaW5fY2hhbm5lbHM9aW5fY2hhbm5lbHMsIHdpZHRoX211bHQ9d2lkdGhfbXVsdCkKCiAgICBpZiBzaXplID09IDIyNDoKICAgICAgICBpZiB3aWR0aF9tdWx0ICE9IDEuMDoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigid2lkdGhfbXVsdCBpcyBvbmx5IHN1cHBvcnRlZCBmb3IgdGhlIHNpemUtMjggQ0lGQVIgYmFja2JvbmUiKQogICAgICAgIGlmIGluX2NoYW5uZWxzICE9IDM6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInRvcmNodmlzaW9uIHJlc25ldCBiYWNrYm9uZSBleHBlY3RzIDMgaW5wdXQgY2hhbm5lbHMiKQogICAgICAgIGltcG9ydCB0b3JjaHZpc2lvbi5tb2RlbHMgYXMgdHZtCgogICAgICAgIHRyeTogICMgbmV3ZXIgdG9yY2h2aXNpb24KICAgICAgICAgICAgY3RvciA9IGdldGF0dHIodHZtLCBtb2RlbF9uYW1lKQogICAgICAgICAgICByZXR1cm4gY3Rvcih3ZWlnaHRzPU5vbmUsIG51bV9jbGFzc2VzPW51bV9jbGFzc2VzKQogICAgICAgIGV4Y2VwdCBUeXBlRXJyb3I6ICAjIG9sZGVyIHRvcmNodmlzaW9uIHdpdGhvdXQgYHdlaWdodHM9YAogICAgICAgICAgICBjdG9yID0gZ2V0YXR0cih0dm0sIG1vZGVsX25hbWUpCiAgICAgICAgICAgIHJldHVybiBjdG9yKHByZXRyYWluZWQ9RmFsc2UsIG51bV9jbGFzc2VzPW51bV9jbGFzc2VzKQoKICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJ1bnN1cHBvcnRlZCBzaXplIHtzaXplfTsgZXhwZWN0ZWQgMjggb3IgMjI0IikKCgpkZWYgY291bnRfcGFyYW1ldGVycyhtb2RlbCk6CiAgICByZXR1cm4gc3VtKHAubnVtZWwoKSBmb3IgcCBpbiBtb2RlbC5wYXJhbWV0ZXJzKCkgaWYgcC5yZXF1aXJlc19ncmFkKQoKCmRlZiBtb2RlbF9zaXplX21iKG1vZGVsKToKICAgICIiIk9uLWRpc2sgc2l6ZSBpbiBNQiBvZiB0aGUgbW9kZWwncyBmbG9hdDMyIHN0YXRlIGRpY3QuIiIiCiAgICBuX2J5dGVzID0gc3VtKHAubnVtZWwoKSAqIHAuZWxlbWVudF9zaXplKCkgZm9yIHAgaW4gbW9kZWwucGFyYW1ldGVycygpKQogICAgbl9ieXRlcyArPSBzdW0oYi5udW1lbCgpICogYi5lbGVtZW50X3NpemUoKSBmb3IgYiBpbiBtb2RlbC5idWZmZXJzKCkpCiAgICByZXR1cm4gbl9ieXRlcyAvICgxMDI0ICoqIDIpCg==',
    'plotting.py': 'IiIiU2hhcmVkIHBsb3R0aW5nIHN0eWxlIGZvciBldmVyeSBmaWd1cmUgaW4gdGhlIHBhcGVyLgoKT25lIHBsYWNlIGZvciB0aGUgY29sb3JibGluZC1zYWZlIHBhbGV0dGUsIGNsZWFuIG1hdHBsb3RsaWIgZGVmYXVsdHMgKG5vIHRvcC8KcmlnaHQgc3BpbmVzLCByZWFkYWJsZSBmb250cyksIGFuZCBzaW5nbGUtY29sdW1uIHNpemluZyAofjMuMyBpbiB3aWRlKS4gRXZlcnkKZmlndXJlIHNjcmlwdCBpbXBvcnRzIGZyb20gaGVyZSBhbmQgc2F2ZXMgdGhyb3VnaCA6ZnVuYzpgc2F2ZWZpZ2AsIHdoaWNoIHdyaXRlcwpib3RoIGEgdmVjdG9yIFBERiBhbmQgYSAzMDAtZHBpIFBORyBpbnRvIGBgcmVwb3J0L2ZpZ3VyZXMvYGAuIFJlZ2VuZXJhdGluZwpmaWd1cmVzIG5ldmVyIHJlLXJ1bnMgdHJhaW5pbmcg4oCUIHRoZSBmaWd1cmUgc2NyaXB0cyByZWFkIG9ubHkgZnJvbSBgYHJlc3VsdHMvYGAuCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IG9zCgppbXBvcnQgbWF0cGxvdGxpYgptYXRwbG90bGliLnVzZSgiQWdnIikKaW1wb3J0IG1hdHBsb3RsaWIucHlwbG90IGFzIHBsdAoKCiMgT2thYmUtSXRvIGNvbG9yYmxpbmQtc2FmZSBxdWFsaXRhdGl2ZSBwYWxldHRlICg4IGh1ZXMsIGRldXRlcmFub3BpYS1zYWZlKS4KUEFMRVRURSA9IFsKICAgICIjMDA3MkIyIiwgICMgYmx1ZQogICAgIiNFNjlGMDAiLCAgIyBvcmFuZ2UKICAgICIjMDA5RTczIiwgICMgZ3JlZW4KICAgICIjRDU1RTAwIiwgICMgdmVybWlsbGlvbgogICAgIiNDQzc5QTciLCAgIyByZWRkaXNoIHB1cnBsZQogICAgIiM1NkI0RTkiLCAgIyBza3kgYmx1ZQogICAgIiNGMEU0NDIiLCAgIyB5ZWxsb3cKICAgICIjMDAwMDAwIiwgICMgYmxhY2sKXQoKIyBEaXZlcmdpbmcgY29sb3JtYXAgZm9yIHNpZ25lZCAob3VycyAtIHBhcGVyKSBkZWx0YXMsIGNlbnRlcmVkIGF0IHplcm8uCkRJVkVSR0lOR19DTUFQID0gIlJkQnVfciIKCiMgU2luZ2xlLWNvbHVtbiBmaWd1cmUgd2lkdGggKGluY2hlcykgZm9yIGEgdHdvLWNvbHVtbiBwYXBlci4KQ09MX1dJRFRIID0gMy4zCkdPTERFTiA9IDAuNjIgICMgaGVpZ2h0OndpZHRoIHJhdGlvIHVzZWQgZm9yIGRlZmF1bHQgc2luZ2xlLXBhbmVsIGZpZ3VyZXMKCgpkZWYgc2V0X3N0eWxlKCk6CiAgICAiIiJJbnN0YWxsIHRoZSBzaGFyZWQgcmNQYXJhbXMuIElkZW1wb3RlbnQ7IGNhbGwgb25jZSBiZWZvcmUgcGxvdHRpbmcuIiIiCiAgICBwbHQucmNQYXJhbXMudXBkYXRlKHsKICAgICAgICAiZmlndXJlLmRwaSI6IDExMCwKICAgICAgICAic2F2ZWZpZy5kcGkiOiAzMDAsCiAgICAgICAgInNhdmVmaWcuYmJveCI6ICJ0aWdodCIsCiAgICAgICAgImZvbnQuc2l6ZSI6IDgsCiAgICAgICAgImF4ZXMudGl0bGVzaXplIjogOSwKICAgICAgICAiYXhlcy5sYWJlbHNpemUiOiA4LAogICAgICAgICJsZWdlbmQuZm9udHNpemUiOiA3LAogICAgICAgICJ4dGljay5sYWJlbHNpemUiOiA3LAogICAgICAgICJ5dGljay5sYWJlbHNpemUiOiA3LAogICAgICAgICJheGVzLmxpbmV3aWR0aCI6IDAuOCwKICAgICAgICAiYXhlcy5zcGluZXMudG9wIjogRmFsc2UsCiAgICAgICAgImF4ZXMuc3BpbmVzLnJpZ2h0IjogRmFsc2UsCiAgICAgICAgImF4ZXMuZ3JpZCI6IFRydWUsCiAgICAgICAgImdyaWQuYWxwaGEiOiAwLjI1LAogICAgICAgICJncmlkLmxpbmV3aWR0aCI6IDAuNiwKICAgICAgICAibGluZXMubGluZXdpZHRoIjogMS41LAogICAgICAgICJsZWdlbmQuZnJhbWVvbiI6IEZhbHNlLAogICAgICAgICJmaWd1cmUuYXV0b2xheW91dCI6IEZhbHNlLAogICAgfSkKICAgICMgQ3ljbGUgdGhlIGNvbG9yYmxpbmQtc2FmZSBwYWxldHRlIGJ5IGRlZmF1bHQuCiAgICBwbHQucmNQYXJhbXNbImF4ZXMucHJvcF9jeWNsZSJdID0gcGx0LmN5Y2xlcihjb2xvcj1QQUxFVFRFKQoKCmRlZiBuZXdfZmlnKHdpZHRoPUNPTF9XSURUSCwgaGVpZ2h0PU5vbmUsIG5jb2xzPTEsIG5yb3dzPTEsICoqa3cpOgogICAgIiIiUmV0dXJuIGBgKGZpZywgYXgpYGAgc2l6ZWQgZm9yIHRoZSBwYXBlci4gYGBoZWlnaHRgYCBkZWZhdWx0cyB0byBnb2xkZW4uIiIiCiAgICBpZiBoZWlnaHQgaXMgTm9uZToKICAgICAgICBoZWlnaHQgPSB3aWR0aCAqIEdPTERFTgogICAgcmV0dXJuIHBsdC5zdWJwbG90cyhucm93cywgbmNvbHMsIGZpZ3NpemU9KHdpZHRoICogbmNvbHMsIGhlaWdodCAqIG5yb3dzKSwgKiprdykKCgpkZWYgZGVzcGluZShheCk6CiAgICAiIiJSZW1vdmUgdGhlIHRvcC9yaWdodCBzcGluZXMgb24gYSBzaW5nbGUgQXhlcyAocmNQYXJhbXMgaGFuZGxlcyBtb3N0KS4iIiIKICAgIGF4LnNwaW5lc1sidG9wIl0uc2V0X3Zpc2libGUoRmFsc2UpCiAgICBheC5zcGluZXNbInJpZ2h0Il0uc2V0X3Zpc2libGUoRmFsc2UpCgoKZGVmIHNhdmVmaWcoZmlnLCBuYW1lLCByZXBvcnRfZGlyPSJyZXBvcnQiKToKICAgICIiIlNhdmUgYGBmaWdgYCBhcyBib3RoIFBERiBhbmQgMzAwLWRwaSBQTkcgdW5kZXIgYGByZXBvcnRfZGlyL2ZpZ3VyZXMvYGAuCgogICAgUmV0dXJucyB0aGUgUE5HIHBhdGggKGhhbmR5IGZvciBub3RlYm9vayBkaXNwbGF5KS4gYGBuYW1lYGAgaXMgYSBzdGVtIHdpdGgKICAgIG5vIGV4dGVuc2lvbiwgZS5nLiBgYCJmaWcxX3JlcHJvZHVjdGlvbiJgYC4KICAgICIiIgogICAgZmlnX2RpciA9IG9zLnBhdGguam9pbihyZXBvcnRfZGlyLCAiZmlndXJlcyIpCiAgICBvcy5tYWtlZGlycyhmaWdfZGlyLCBleGlzdF9vaz1UcnVlKQogICAgcGRmID0gb3MucGF0aC5qb2luKGZpZ19kaXIsIGYie25hbWV9LnBkZiIpCiAgICBwbmcgPSBvcy5wYXRoLmpvaW4oZmlnX2RpciwgZiJ7bmFtZX0ucG5nIikKICAgIGZpZy5zYXZlZmlnKHBkZikKICAgIGZpZy5zYXZlZmlnKHBuZywgZHBpPTMwMCkKICAgIHBsdC5jbG9zZShmaWcpCiAgICByZXR1cm4gcG5nCg==',
    'reference.py': 'IiIiUGFwZXIgcmVmZXJlbmNlIG51bWJlcnMgKE1lZE1OSVNUIHYyLCBZYW5nIGV0IGFsLiAyMDIzLCBUYWJsZSAzKS4KCktlcHQgZGVwZW5kZW5jeS1mcmVlIHNvIGFnZ3JlZ2F0aW9uIGNhbiBpbXBvcnQgaXQgd2l0aG91dCB0b3JjaC4KRm9ybWF0OiAoZGF0YXNldCwgbW9kZWwsIHNpemUpIC0+IChBVUMsIEFDQykuCiIiIgoKUkVGRVJFTkNFID0gewogICAgIyBEZXJtYU1OSVNUIOKAlCB0aGUgcHJpbWFyeSwgaW4tZGVwdGggZm91ci1jb25maWcgbWF0cml4IChSMTgvUjUwIHggMjgvMjI0KS4KICAgICgiZGVybWFtbmlzdCIsICJyZXNuZXQxOCIsIDI4KTogKDAuOTE3LCAwLjczNSksCiAgICAoImRlcm1hbW5pc3QiLCAicmVzbmV0MTgiLCAyMjQpOiAoMC45MjAsIDAuNzU0KSwKICAgICgiZGVybWFtbmlzdCIsICJyZXNuZXQ1MCIsIDI4KTogKDAuOTEzLCAwLjczNSksCiAgICAoImRlcm1hbW5pc3QiLCAicmVzbmV0NTAiLCAyMjQpOiAoMC45MTIsIDAuNzMxKSwKICAgICMgVGhlIG90aGVyIGVsZXZlbiBNZWRNTklTVDJEIGRhdGFzZXRzIOKAlCBSZXNOZXQtMTggQCAyOCBvbmx5IChvdXIgc3dlZXApLgogICAgKCJwYXRobW5pc3QiLCAicmVzbmV0MTgiLCAyOCk6ICgwLjk4MywgMC45MDcpLAogICAgKCJjaGVzdG1uaXN0IiwgInJlc25ldDE4IiwgMjgpOiAoMC43NjgsIDAuOTQ3KSwKICAgICgib2N0bW5pc3QiLCAicmVzbmV0MTgiLCAyOCk6ICgwLjk0MywgMC43NDMpLAogICAgKCJwbmV1bW9uaWFtbmlzdCIsICJyZXNuZXQxOCIsIDI4KTogKDAuOTQ0LCAwLjg1NCksCiAgICAoInJldGluYW1uaXN0IiwgInJlc25ldDE4IiwgMjgpOiAoMC43MTcsIDAuNTI0KSwKICAgICgiYnJlYXN0bW5pc3QiLCAicmVzbmV0MTgiLCAyOCk6ICgwLjkwMSwgMC44NjMpLAogICAgKCJibG9vZG1uaXN0IiwgInJlc25ldDE4IiwgMjgpOiAoMC45OTgsIDAuOTU4KSwKICAgICgidGlzc3VlbW5pc3QiLCAicmVzbmV0MTgiLCAyOCk6ICgwLjkzMCwgMC42NzYpLAogICAgKCJvcmdhbmFtbmlzdCIsICJyZXNuZXQxOCIsIDI4KTogKDAuOTk3LCAwLjkzNSksCiAgICAoIm9yZ2FuY21uaXN0IiwgInJlc25ldDE4IiwgMjgpOiAoMC45OTIsIDAuOTAwKSwKICAgICgib3JnYW5zbW5pc3QiLCAicmVzbmV0MTgiLCAyOCk6ICgwLjk3MiwgMC43ODIpLAogICAgIyBFeHRyYSBSMTgvUjUwIEAgMjI0LzI4IFBhdGhNTklTVCByZWZlcmVuY2VzIChrZXB0IGZvciBvcHRpb25hbCBkZXB0aCBydW5zKS4KICAgICgicGF0aG1uaXN0IiwgInJlc25ldDE4IiwgMjI0KTogKDAuOTg5LCAwLjkwOSksCiAgICAoInBhdGhtbmlzdCIsICJyZXNuZXQ1MCIsIDI4KTogKDAuOTkwLCAwLjkxMSksCiAgICAoInBhdGhtbmlzdCIsICJyZXNuZXQ1MCIsIDIyNCk6ICgwLjk4OSwgMC44OTIpLAp9CgojIENvbnZlbmllbmNlOiB0aGUgdHdlbHZlIE1lZE1OSVNUMkQgZGF0YXNldHMgaW4gdGhlIHNwZWMncyBjYW5vbmljYWwgb3JkZXIsCiMgZWFjaCB3aXRoIHRoZSBSZXNOZXQtMTggQCAyOCByZWZlcmVuY2UgdGhlIHJlcGxpY2F0aW9uIHN3ZWVwIHRhcmdldHMuCkRBVEFTRVRTXzJEID0gWwogICAgInBhdGhtbmlzdCIsICJjaGVzdG1uaXN0IiwgImRlcm1hbW5pc3QiLCAib2N0bW5pc3QiLAogICAgInBuZXVtb25pYW1uaXN0IiwgInJldGluYW1uaXN0IiwgImJyZWFzdG1uaXN0IiwgImJsb29kbW5pc3QiLAogICAgInRpc3N1ZW1uaXN0IiwgIm9yZ2FuYW1uaXN0IiwgIm9yZ2FuY21uaXN0IiwgIm9yZ2Fuc21uaXN0IiwKXQo=',
    'reproduction_arm.py': 'IiIiVGhlICpyZXByb2R1Y3Rpb24qIGFybTogcnVucyBleGVjdXRlZCB3aXRoIHRoZSBvcmlnaW5hbCBhdXRob3JzJyBvd24gY29kZS4KClRoaXMgc3R1ZHkgaGFzIHR3byBhcm1zLCBhbmQgdGhleSBtdXN0IG5ldmVyIGJlIHBvb2xlZCBpbnRvIG9uZSB0YWJsZToKCiogKipSZXBsaWNhdGlvbioqIChgYHNyYy9gYCwgOm1vZDpgc3JjLmFnZ3JlZ2F0ZWApIOKAlCBvdXIgaW5kZXBlbmRlbnQKICByZWltcGxlbWVudGF0aW9uLiBOb3RoaW5nIGZyb20gYGBNZWRNTklTVC9leHBlcmltZW50c2BgIGlzIHVzZWQuCiogKipSZXByb2R1Y3Rpb24qKiAodGhpcyBtb2R1bGUpIOKAlCB0aGUgYXV0aG9ycycgYGB0cmFpbl9hbmRfZXZhbF9weXRvcmNoLnB5YGAKICBmcm9tIGBgTWVkTU5JU1QvZXhwZXJpbWVudHNgYCwgcnVuIHVubW9kaWZpZWQgb24gS2FnZ2xlLiBJdCBhbnN3ZXJzIGEKICBkaWZmZXJlbnQgcXVlc3Rpb246IGRvZXMgdGhlIGF1dGhvcnMnIHJlbGVhc2VkIGNvZGUsIG9uIHRoZSBhdXRob3JzJyByZWxlYXNlZAogIGRhdGEsIHN0aWxsIHByb2R1Y2UgdGhlIHB1Ymxpc2hlZCBudW1iZXJzPwoKQm90aCBhcmUgbGVnaXRpbWF0ZSBhbmQgUmVTY2llbmNlIEMgY2FyZXMgYWJvdXQgdGhlIGRpc3RpbmN0aW9uLCBzbyB0aGUKcHJvdmVuYW5jZSBvZiBldmVyeSBudW1iZXIgYmVsb3cgaXMgcmVjb3JkZWQgZXhwbGljaXRseSByYXRoZXIgdGhhbiBpbmZlcnJlZC4KClRoZSBydW5zIGhlcmUgYXJlICoqc2luZ2xlLXNlZWQqKiAodGhleSBwcmVkYXRlIHRoZSBzZWVkZWQgbWF0cml4KSwgc28gbm8gwrFzdGQKaXMgcmVwb3J0ZWQg4oCUIHN0YXRlZCByYXRoZXIgdGhhbiBwYXBlcmVkIG92ZXIuCgogICAgcHl0aG9uIC1tIHNyYy5yZXByb2R1Y3Rpb25fYXJtICAgICAgICAgICAgIyAtPiByZXBvcnQvcmVwcm9kdWN0aW9uX2FybS57Y3N2LG1kfQoKU3RkbGliLW9ubHksIHNvIGl0IHJ1bnMgd2l0aG91dCB0b3JjaC4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgY3N2CmltcG9ydCBvcwoKZnJvbSAucmVmZXJlbmNlIGltcG9ydCBSRUZFUkVOQ0UKCiMgU2FtZSB0b2xlcmFuY2VzIGFzIHRoZSByZXBsaWNhdGlvbiBhcm0sIHNvIHRoZSB0d28gdGFibGVzIGFyZSByZWFkIHRoZSBzYW1lIHdheS4KQVVDX1RPTCA9IDAuMDIKQUNDX1RPTCA9IDAuMDMKClJFU1VMVFNfU1VCRElSID0gb3MucGF0aC5qb2luKCJyZXN1bHRzIiwgInJlcHJvZHVjdGlvbl9hdXRob3JzX2NvZGUiKQoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gIwojIE1hbmlmZXN0IG9mIGV2ZXJ5dGhpbmcgdGhhdCB3YXMgYWN0dWFsbHkgcnVuLCB3aXRoIGZ1bGwgcHJvdmVuYW5jZS4KIwojIGBgc2l6ZWBgIGlzIDIyNCBmb3IgdGhlIHJ1bnMgdGhhdCBwYXNzZWQgYGAtLXJlc2l6ZWBgOiB0aGUgYXV0aG9ycycgc2NyaXB0CiMgdXBzYW1wbGVzIHRoZSAyOC1waXhlbCBucHogdG8gMjI0LCB3aGljaCBpcyB0aGUgc2V0dGluZyBwYXBlciBUYWJsZSAzIHJlcG9ydHMKIyBpbiBpdHMgMjI0IGNvbHVtbi4gR2V0dGluZyB0aGlzIHdyb25nIHNpbGVudGx5IGNvbXBhcmVzIGFnYWluc3QgdGhlIHdyb25nCiMgcmVmZXJlbmNlIHJvdyAoc2VlIE5PVEUgb24gcGVyc29uQSBiZWxvdykuCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tICMKUlVOUyA9IFsKICAgIGRpY3QoCiAgICAgICAgbGFiZWw9InBlcnNvbkEiLAogICAgICAgIGRhdGFzZXQ9ImRlcm1hbW5pc3QiLCBtb2RlbD0icmVzbmV0MTgiLCBzaXplPTIyNCwgc2VlZHM9MSwKICAgICAgICBhdWM9MC45MjEyLCBhY2M9MC43NDAxLAogICAgICAgIG5vdGVib29rPSJub3RlYm9va3MvcmVwcm9kdWN0aW9uX2F1dGhvcnNfY29kZS9wZXJzb25BX3Jlc25ldDE4X2Rlcm1hbW5pc3QuaXB5bmIiLAogICAgICAgIGNvZGU9Ik1lZE1OSVNUL2V4cGVyaW1lbnRzIEAgdHJhaW5fYW5kX2V2YWxfcHl0b3JjaC5weSAodW5tb2RpZmllZCkiLAogICAgICAgIGNvbW1hbmQ9KCJweXRob24gdHJhaW5fYW5kX2V2YWxfcHl0b3JjaC5weSAtLWRhdGFfZmxhZyBkZXJtYW1uaXN0ICIKICAgICAgICAgICAgICAgICAiLS1udW1fZXBvY2hzIDEwMCAtLWJhdGNoX3NpemUgMTI4IC0tcmVzaXplIC0tbW9kZWxfZmxhZyByZXNuZXQxOCAiCiAgICAgICAgICAgICAgICAgIi0tcnVuIHJ1bjEiKSwKICAgICAgICByZXN1bHRzPW9zLnBhdGguam9pbihSRVNVTFRTX1NVQkRJUiwgInBlcnNvbkFfcmVzbmV0MThfZGVybWEyMjQiKSwKICAgICAgICBwcmVkaWN0aW9ucz1Ob25lLAogICAgICAgICMgTk9URTogdGhlIGFyY2hpdmVkIHJlcHJvZHVjdGlvbl9zdW1tYXJ5X3BlcnNvbkEuY3N2IGNvbXBhcmVzIHRoaXMgcnVuCiAgICAgICAgIyBhZ2FpbnN0IEFVQyAwLjkxNywgd2hpY2ggaXMgdGhlIHBhcGVyJ3MgKjI4LXBpeGVsKiBSZXNOZXQtMTggcm93LiBUaGUKICAgICAgICAjIHJ1biB1c2VkIC0tcmVzaXplLCBzbyAyMjQgKEFVQyAwLjkyMCAvIEFDQyAwLjc1NCkgaXMgdGhlIGNvcnJlY3QKICAgICAgICAjIHJlZmVyZW5jZS4gQ29ycmVjdGVkIGhlcmU7IHRoZSBhcmNoaXZlZCBDU1YgaXMgbGVmdCB1bnRvdWNoZWQgYXMgYQogICAgICAgICMgcmVjb3JkIG9mIHdoYXQgd2FzIG9yaWdpbmFsbHkgY29tcHV0ZWQuCiAgICAgICAgbm90ZT0icmVmZXJlbmNlIGNvcnJlY3RlZCBmcm9tIHRoZSAyOHB4IHJvdyB0byB0aGUgMjI0cHggcm93IChydW4gdXNlZCAtLXJlc2l6ZSkiLAogICAgKSwKICAgIGRpY3QoCiAgICAgICAgbGFiZWw9InBlcnNvbkIiLAogICAgICAgIGRhdGFzZXQ9ImRlcm1hbW5pc3QiLCBtb2RlbD0icmVzbmV0NTAiLCBzaXplPTIyNCwgc2VlZHM9MSwKICAgICAgICBhdWM9MC45MTE5LCBhY2M9MC43Mzc3LAogICAgICAgIG5vdGVib29rPSJub3RlYm9va3MvcmVwcm9kdWN0aW9uX2F1dGhvcnNfY29kZS9wZXJzb25CX3Jlc25ldDUwX2Rlcm1hbW5pc3QuaXB5bmIiLAogICAgICAgIGNvZGU9Ik1lZE1OSVNUL2V4cGVyaW1lbnRzIEAgdHJhaW5fYW5kX2V2YWxfcHl0b3JjaC5weSAodW5tb2RpZmllZCkiLAogICAgICAgIGNvbW1hbmQ9KCJweXRob24gdHJhaW5fYW5kX2V2YWxfcHl0b3JjaC5weSAtLWRhdGFfZmxhZyBkZXJtYW1uaXN0ICIKICAgICAgICAgICAgICAgICAiLS1udW1fZXBvY2hzIDEwMCAtLWJhdGNoX3NpemUgMTI4IC0tcmVzaXplIC0tbW9kZWxfZmxhZyByZXNuZXQ1MCAiCiAgICAgICAgICAgICAgICAgIi0tcnVuIHJ1bjEiKSwKICAgICAgICByZXN1bHRzPW9zLnBhdGguam9pbihSRVNVTFRTX1NVQkRJUiwgInBlcnNvbkJfcmVzbmV0NTBfZGVybWEyMjQiKSwKICAgICAgICBwcmVkaWN0aW9ucz1vcy5wYXRoLmpvaW4oCiAgICAgICAgICAgIFJFU1VMVFNfU1VCRElSLCAicGVyc29uQl9yZXNuZXQ1MF9kZXJtYTIyNCIsICJvdXRwdXQiLCAiZGVybWFtbmlzdCIsCiAgICAgICAgICAgICIyNjA3MjBfMDgyNzE3IiwKICAgICAgICAgICAgImRlcm1hbW5pc3RfdGVzdF9bQVVDXTAuOTEyX1tBQ0NdMC43MzhAcnVuMS5jc3YiKSwKICAgICAgICBub3RlPSJmdWxsIHRlc3Qgc29mdG1heCBtYXRyaXggcmV0YWluZWQgLT4gcHJlZGljdGlvbi1vbmx5IGV4dGVuc2lvbnMgIgogICAgICAgICAgICAgInJlY29tcHV0YWJsZSB2aWEgc3JjLmZyb21fcHJlZGljdGlvbnMgKG5vIEdQVSkiLAogICAgKSwKXQoKCmRlZiByb3dzKCk6CiAgICAiIiJPbmUgZGljdCBwZXIgcmVwcm9kdWN0aW9uIHJ1biwgd2l0aCByZWZlcmVuY2UgZGVsdGFzIGFuZCB0b2xlcmFuY2UgZmxhZy4iIiIKICAgIG91dCA9IFtdCiAgICBmb3IgciBpbiBSVU5TOgogICAgICAgIGtleSA9IChyWyJkYXRhc2V0Il0sIHJbIm1vZGVsIl0sIHJbInNpemUiXSkKICAgICAgICByZWYgPSBSRUZFUkVOQ0UuZ2V0KGtleSkKICAgICAgICByZWZfYXVjLCByZWZfYWNjID0gcmVmIGlmIHJlZiBlbHNlIChOb25lLCBOb25lKQogICAgICAgIGRfYXVjID0gclsiYXVjIl0gLSByZWZfYXVjIGlmIHJlZiBlbHNlIE5vbmUKICAgICAgICBkX2FjYyA9IHJbImFjYyJdIC0gcmVmX2FjYyBpZiByZWYgZWxzZSBOb25lCiAgICAgICAgZmxhZyA9ICIiCiAgICAgICAgaWYgcmVmIGFuZCAoYWJzKGRfYXVjKSA+IEFVQ19UT0wgb3IgYWJzKGRfYWNjKSA+IEFDQ19UT0wpOgogICAgICAgICAgICBmbGFnID0gIk9VVF9PRl9UT0wiCiAgICAgICAgb3V0LmFwcGVuZChkaWN0KAogICAgICAgICAgICBsYWJlbD1yWyJsYWJlbCJdLCBkYXRhc2V0PXJbImRhdGFzZXQiXSwgbW9kZWw9clsibW9kZWwiXSwKICAgICAgICAgICAgc2l6ZT1yWyJzaXplIl0sIG5fc2VlZHM9clsic2VlZHMiXSwKICAgICAgICAgICAgYXVjPXJbImF1YyJdLCBhY2M9clsiYWNjIl0sCiAgICAgICAgICAgIHJlZl9hdWM9cmVmX2F1YywgcmVmX2FjYz1yZWZfYWNjLAogICAgICAgICAgICBkZWx0YV9hdWM9ZF9hdWMsIGRlbHRhX2FjYz1kX2FjYywgZmxhZz1mbGFnLAogICAgICAgICAgICBjb2RlPXJbImNvZGUiXSwgbm90ZWJvb2s9clsibm90ZWJvb2siXSwgcmVzdWx0cz1yWyJyZXN1bHRzIl0sCiAgICAgICAgICAgIHByZWRpY3Rpb25zPXJbInByZWRpY3Rpb25zIl0gb3IgIiIsIG5vdGU9ci5nZXQoIm5vdGUiLCAiIiksCiAgICAgICAgKSkKICAgIHJldHVybiBvdXQKCgpkZWYgbWlzc2luZ19hcnRpZmFjdHMocmVwb19yb290PSIuIik6CiAgICAiIiJQYXRocyBpbiB0aGUgbWFuaWZlc3QgdGhhdCBhcmUgbm90IG9uIGRpc2sg4oCUIGtlZXBzIHRoZSB0YWJsZSBob25lc3QuIiIiCiAgICBnb25lID0gW10KICAgIGZvciByIGluIFJVTlM6CiAgICAgICAgZm9yIGtleSBpbiAoIm5vdGVib29rIiwgInJlc3VsdHMiLCAicHJlZGljdGlvbnMiKToKICAgICAgICAgICAgcGF0aCA9IHIuZ2V0KGtleSkKICAgICAgICAgICAgaWYgcGF0aCBhbmQgbm90IG9zLnBhdGguZXhpc3RzKG9zLnBhdGguam9pbihyZXBvX3Jvb3QsIHBhdGgpKToKICAgICAgICAgICAgICAgIGdvbmUuYXBwZW5kKChyWyJsYWJlbCJdLCBrZXksIHBhdGgpKQogICAgcmV0dXJuIGdvbmUKCgpkZWYgX2ZtdCh4LCBuZD0zKToKICAgIHJldHVybiAiIiBpZiB4IGlzIE5vbmUgZWxzZSBmInt4Oi57bmR9Zn0iCgoKZGVmIF93cml0ZV9jc3YoZGF0YSwgcGF0aCk6CiAgICBmaWVsZHMgPSBbImxhYmVsIiwgImRhdGFzZXQiLCAibW9kZWwiLCAic2l6ZSIsICJuX3NlZWRzIiwgImF1YyIsICJhY2MiLAogICAgICAgICAgICAgICJyZWZfYXVjIiwgInJlZl9hY2MiLCAiZGVsdGFfYXVjIiwgImRlbHRhX2FjYyIsICJmbGFnIiwgImNvZGUiLAogICAgICAgICAgICAgICJub3RlYm9vayIsICJyZXN1bHRzIiwgInByZWRpY3Rpb25zIiwgIm5vdGUiXQogICAgd2l0aCBvcGVuKHBhdGgsICJ3IiwgbmV3bGluZT0iIikgYXMgZjoKICAgICAgICB3ID0gY3N2LkRpY3RXcml0ZXIoZiwgZmllbGRuYW1lcz1maWVsZHMpCiAgICAgICAgdy53cml0ZWhlYWRlcigpCiAgICAgICAgdy53cml0ZXJvd3MoZGF0YSkKCgpkZWYgX3dyaXRlX21kKGRhdGEsIHBhdGgsIGdvbmUpOgogICAgbGluZXMgPSBbCiAgICAgICAgIiMgUmVwcm9kdWN0aW9uIGFybSDigJQgdGhlIGF1dGhvcnMnIG93biBjb2RlIiwKICAgICAgICAiIiwKICAgICAgICAiUnVucyBleGVjdXRlZCB3aXRoIGB0cmFpbl9hbmRfZXZhbF9weXRvcmNoLnB5YCBmcm9tICIKICAgICAgICAiW01lZE1OSVNUL2V4cGVyaW1lbnRzXShodHRwczovL2dpdGh1Yi5jb20vTWVkTU5JU1QvZXhwZXJpbWVudHMpLCAiCiAgICAgICAgInVubW9kaWZpZWQuIFRoaXMgaXMgKipub3QqKiB0aGUgcmVwbGljYXRpb24gYXJtOyBmb3Igb3VyIGluZGVwZW5kZW50ICIKICAgICAgICAicmVpbXBsZW1lbnRhdGlvbiBzZWUgYGNvbXBhcmlzb24ubWRgLCBnZW5lcmF0ZWQgYnkgYHNyYy5hZ2dyZWdhdGVgLiIsCiAgICAgICAgIiIsCiAgICAgICAgImAtLXJlc2l6ZWAgdXBzYW1wbGVzIHRoZSAyOC1waXhlbCBkYXRhIHRvIDIyNCwgc28gdGhlc2UgY29tcGFyZSAiCiAgICAgICAgImFnYWluc3QgdGhlIHBhcGVyJ3MgKioyMjQqKiBjb2x1bW4gb2YgVGFibGUgMy4iLAogICAgICAgICIiLAogICAgICAgIGYiVG9sZXJhbmNlOiB8zpRBVUN8IOKJpCB7QVVDX1RPTH0sIHzOlEFDQ3wg4omkIHtBQ0NfVE9MfS4gIgogICAgICAgICJEZWx0YSA9IG91cnMg4oiSIHBhcGVyLiAqKlNpbmdsZSBzZWVkIHBlciBjb25maWcg4oCUIG5vIHZhcmlhbmNlIGlzICIKICAgICAgICAiY2xhaW1lZC4qKiIsCiAgICAgICAgIiIsCiAgICAgICAgInwgUnVuIHwgRGF0YXNldCB8IE1vZGVsIHwgU2l6ZSB8IFNlZWRzIHwgT3VyIEFVQyB8IFBhcGVyIEFVQyB8IM6UQVVDIHwgIgogICAgICAgICJPdXIgQUNDIHwgUGFwZXIgQUNDIHwgzpRBQ0MgfCBGbGFnIHwiLAogICAgICAgICJ8LS0tfC0tLXwtLS18LS0tfC0tLXwtLS18LS0tfC0tLXwtLS18LS0tfC0tLXwtLS18IiwKICAgIF0KICAgIGZvciByIGluIGRhdGE6CiAgICAgICAgbGluZXMuYXBwZW5kKAogICAgICAgICAgICBmInwge3JbJ2xhYmVsJ119IHwge3JbJ2RhdGFzZXQnXX0gfCB7clsnbW9kZWwnXX0gfCB7clsnc2l6ZSddfSB8ICIKICAgICAgICAgICAgZiJ7clsnbl9zZWVkcyddfSB8IHtfZm10KHJbJ2F1YyddLCA0KX0gfCB7X2ZtdChyWydyZWZfYXVjJ10pfSB8ICIKICAgICAgICAgICAgZiJ7X2ZtdChyWydkZWx0YV9hdWMnXSwgNCl9IHwge19mbXQoclsnYWNjJ10sIDQpfSB8ICIKICAgICAgICAgICAgZiJ7X2ZtdChyWydyZWZfYWNjJ10pfSB8IHtfZm10KHJbJ2RlbHRhX2FjYyddLCA0KX0gfCB7clsnZmxhZyddfSB8IikKCiAgICBsaW5lcyArPSBbIiIsICIjIyBQcm92ZW5hbmNlIiwgIiJdCiAgICBmb3IgciBpbiBkYXRhOgogICAgICAgIGxpbmVzLmFwcGVuZChmIioqe3JbJ2xhYmVsJ119Kiog4oCUIGB7clsnY29kZSddfWAiKQogICAgICAgIGxpbmVzLmFwcGVuZChmIi0gbm90ZWJvb2s6IGB7clsnbm90ZWJvb2snXX1gIikKICAgICAgICBsaW5lcy5hcHBlbmQoZiItIGFydGlmYWN0czogYHtyWydyZXN1bHRzJ119YCIpCiAgICAgICAgaWYgclsicHJlZGljdGlvbnMiXToKICAgICAgICAgICAgbGluZXMuYXBwZW5kKGYiLSB0ZXN0IHNjb3JlIG1hdHJpeDogYHtyWydwcmVkaWN0aW9ucyddfWAiKQogICAgICAgIGlmIHJbIm5vdGUiXToKICAgICAgICAgICAgbGluZXMuYXBwZW5kKGYiLSBub3RlOiB7clsnbm90ZSddfSIpCiAgICAgICAgbGluZXMuYXBwZW5kKCIiKQoKICAgIGlmIGdvbmU6CiAgICAgICAgbGluZXMgKz0gWyIjIyBNaXNzaW5nIGFydGlmYWN0cyIsICIiXQogICAgICAgIGZvciBsYWJlbCwga2V5LCBwYXRoIGluIGdvbmU6CiAgICAgICAgICAgIGxpbmVzLmFwcGVuZChmIi0gKip7bGFiZWx9Kioge2tleX06IGB7cGF0aH1gIG5vdCBmb3VuZCIpCiAgICAgICAgbGluZXMuYXBwZW5kKCIiKQoKICAgIHdpdGggb3BlbihwYXRoLCAidyIpIGFzIGY6CiAgICAgICAgZi53cml0ZSgiXG4iLmpvaW4obGluZXMpKQoKCmRlZiBidWlsZChyZXBvcnRfZGlyPSJyZXBvcnQiLCByZXBvX3Jvb3Q9Ii4iKToKICAgIG9zLm1ha2VkaXJzKHJlcG9ydF9kaXIsIGV4aXN0X29rPVRydWUpCiAgICBkYXRhID0gcm93cygpCiAgICBnb25lID0gbWlzc2luZ19hcnRpZmFjdHMocmVwb19yb290KQogICAgX3dyaXRlX2NzdihkYXRhLCBvcy5wYXRoLmpvaW4ocmVwb3J0X2RpciwgInJlcHJvZHVjdGlvbl9hcm0uY3N2IikpCiAgICBfd3JpdGVfbWQoZGF0YSwgb3MucGF0aC5qb2luKHJlcG9ydF9kaXIsICJyZXByb2R1Y3Rpb25fYXJtLm1kIiksIGdvbmUpCiAgICByZXR1cm4gZGF0YSwgZ29uZQoKCmRlZiBtYWluKGFyZ3Y9Tm9uZSk6CiAgICBpbXBvcnQgc3lzCiAgICByZXBvcnRfZGlyID0gYXJndlswXSBpZiBhcmd2IGVsc2UgInJlcG9ydCIKICAgIGRhdGEsIGdvbmUgPSBidWlsZChyZXBvcnRfZGlyKQogICAgZm9yIHIgaW4gZGF0YToKICAgICAgICBwcmludChmIntyWydsYWJlbCddOj44fSAge3JbJ2RhdGFzZXQnXX0ge3JbJ21vZGVsJ119IEB7clsnc2l6ZSddfSAgIgogICAgICAgICAgICAgIGYiQVVDIHtyWydhdWMnXTouNGZ9ICjOlHtyWydkZWx0YV9hdWMnXTorLjRmfSkgICIKICAgICAgICAgICAgICBmIkFDQyB7clsnYWNjJ106LjRmfSAozpR7clsnZGVsdGFfYWNjJ106Ky40Zn0pICB7clsnZmxhZyddfSIpCiAgICBmb3IgbGFiZWwsIGtleSwgcGF0aCBpbiBnb25lOgogICAgICAgIHByaW50KGYiV0FSTklORyAge2xhYmVsfTogbWlzc2luZyB7a2V5fSAtPiB7cGF0aH0iLCBmaWxlPXN5cy5zdGRlcnIpCiAgICBwcmludChmIlxud3JvdGUge3JlcG9ydF9kaXJ9L3JlcHJvZHVjdGlvbl9hcm0ue3tjc3YsbWR9fSIpCiAgICByZXR1cm4gMAoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBpbXBvcnQgc3lzCiAgICBzeXMuZXhpdChtYWluKHN5cy5hcmd2WzE6XSkpCg==',
    'run.py': 'IiIiQ29uZmlnLWRyaXZlbiBlbnRyeSBwb2ludCBmb3IgYSBzaW5nbGUgdHJhaW5pbmcgcnVuLgoKVXNhZ2UgKENMSSk6CiAgICBweXRob24gLW0gc3JjLnJ1biAtLWRhdGFzZXQgZGVybWFtbmlzdCAtLW1vZGVsIHJlc25ldDE4IC0tc2l6ZSAyOCAtLXNlZWQgMCBcCiAgICAgICAgLS1lcG9jaHMgMTAwIC0tcmVzdWx0cy1kaXIgcmVzdWx0cwoKT3IgaW1wb3J0IDpmdW5jOmBydW5fY29uZmlnYCBmcm9tIGEgbm90ZWJvb2suCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IG9zCmltcG9ydCBpbwppbXBvcnQgc3lzCmltcG9ydCBqc29uCmltcG9ydCB0aW1lCmltcG9ydCBhcmdwYXJzZQppbXBvcnQgcGxhdGZvcm0KZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzLCBhc2RpY3QsIGZpZWxkCgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHRvcmNoCgpmcm9tIC4gaW1wb3J0IGRhdGEgYXMgZGF0YW1vZApmcm9tIC4gaW1wb3J0IG1ldHJpY3MgYXMgbWV0cmljc21vZApmcm9tIC5tb2RlbHMgaW1wb3J0IGJ1aWxkX21vZGVsLCBjb3VudF9wYXJhbWV0ZXJzLCBtb2RlbF9zaXplX21iCmZyb20gLnRyYWluIGltcG9ydCBzZXRfc2VlZCwgcnVuX3RyYWluaW5nCmZyb20gLmV2YWx1YXRlIGltcG9ydCBwcmVkaWN0LCBzYXZlX3ByZWRpY3Rpb25zLCBwcmVkaWN0aW9uX2ZpbGVuYW1lCmZyb20gLnJlZmVyZW5jZSBpbXBvcnQgUkVGRVJFTkNFCgoKQGRhdGFjbGFzcwpjbGFzcyBSdW5Db25maWc6CiAgICBkYXRhc2V0OiBzdHIgPSAiZGVybWFtbmlzdCIKICAgIG1vZGVsOiBzdHIgPSAicmVzbmV0MTgiCiAgICBzaXplOiBpbnQgPSAyOAogICAgc2VlZDogaW50ID0gMAogICAgZXBvY2hzOiBpbnQgPSAxMDAKICAgIGJhdGNoX3NpemU6IGludCA9IDEyOAogICAgbHI6IGZsb2F0ID0gMWUtMwogICAgYW1wOiBib29sID0gRmFsc2UKICAgIGRldGVybWluaXN0aWM6IGJvb2wgPSBUcnVlCiAgICB3aWR0aF9tdWx0OiBmbG9hdCA9IDEuMAogICAgIyBiaWFzLW1pdGlnYXRpb24gZmxhZ3MgKGV4dGVuc2lvbnMpCiAgICB3ZWlnaHRlZF9zYW1wbGVyOiBib29sID0gRmFsc2UKICAgIHdlaWdodGVkX2xvc3M6IGJvb2wgPSBGYWxzZQogICAgbnVtX3dvcmtlcnM6IGludCA9IDIKICAgIGNrcHRfZXZlcnk6IGludCA9IDUKICAgIHJlc3VtZTogYm9vbCA9IFRydWUKICAgIGRvd25sb2FkOiBib29sID0gVHJ1ZQogICAgcm9vdDogc3RyID0gTm9uZQogICAgcmVzdWx0c19kaXI6IHN0ciA9ICJyZXN1bHRzIgogICAgdGFnOiBzdHIgPSAiIiAgIyBleHRyYSBsYWJlbCBmb2xkZWQgaW50byB0aGUgcnVuIGRpcmVjdG9yeSBuYW1lCgogICAgZGVmIGZsYWcoc2VsZik6CiAgICAgICAgcmV0dXJuIHNlbGYuZGF0YXNldAoKICAgIGRlZiBydW5fbmFtZShzZWxmKToKICAgICAgICBwYXJ0cyA9IFtzZWxmLmRhdGFzZXQsIHNlbGYubW9kZWwsIGYic3tzZWxmLnNpemV9IiwgZiJzZWVke3NlbGYuc2VlZH0iXQogICAgICAgIGlmIHNlbGYud2lkdGhfbXVsdCAhPSAxLjA6CiAgICAgICAgICAgIHBhcnRzLmFwcGVuZChmInd7c2VsZi53aWR0aF9tdWx0fSIpCiAgICAgICAgaWYgc2VsZi53ZWlnaHRlZF9zYW1wbGVyOgogICAgICAgICAgICBwYXJ0cy5hcHBlbmQoIndzYW1wbGVyIikKICAgICAgICBpZiBzZWxmLndlaWdodGVkX2xvc3M6CiAgICAgICAgICAgIHBhcnRzLmFwcGVuZCgid2xvc3MiKQogICAgICAgIGlmIHNlbGYudGFnOgogICAgICAgICAgICBwYXJ0cy5hcHBlbmQoc2VsZi50YWcpCiAgICAgICAgcmV0dXJuICJfIi5qb2luKHBhcnRzKQoKCmRlZiBfdmVyc2lvbnMoKToKICAgIGltcG9ydCBza2xlYXJuCiAgICBpbXBvcnQgdG9yY2h2aXNpb24KICAgIGltcG9ydCBtZWRtbmlzdAogICAgcmV0dXJuIGRpY3QoCiAgICAgICAgcHl0aG9uPXBsYXRmb3JtLnB5dGhvbl92ZXJzaW9uKCksCiAgICAgICAgdG9yY2g9dG9yY2guX192ZXJzaW9uX18sCiAgICAgICAgdG9yY2h2aXNpb249dG9yY2h2aXNpb24uX192ZXJzaW9uX18sCiAgICAgICAgbWVkbW5pc3Q9bWVkbW5pc3QuX192ZXJzaW9uX18sCiAgICAgICAgbnVtcHk9bnAuX192ZXJzaW9uX18sCiAgICAgICAgc2tsZWFybj1za2xlYXJuLl9fdmVyc2lvbl9fLAogICAgICAgIGN1ZGE9dG9yY2gudmVyc2lvbi5jdWRhLAogICAgICAgIGdwdT10b3JjaC5jdWRhLmdldF9kZXZpY2VfbmFtZSgwKSBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpCiAgICAgICAgICAgIGVsc2UgKCJtcHMiIGlmIHRvcmNoLmJhY2tlbmRzLm1wcy5pc19hdmFpbGFibGUoKSBlbHNlICJjcHUiKSwKICAgICkKCgpkZWYgX3NlbGVjdF9kZXZpY2UoKToKICAgIGlmIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCk6CiAgICAgICAgcmV0dXJuICJjdWRhIgogICAgaWYgdG9yY2guYmFja2VuZHMubXBzLmlzX2F2YWlsYWJsZSgpOgogICAgICAgIHJldHVybiAibXBzIgogICAgcmV0dXJuICJjcHUiCgoKZGVmIHJ1bl9jb25maWcoY2ZnOiBSdW5Db25maWcsIGxvZ19mbj1wcmludCwgdmVyaWZ5X21ldHJpY3M9VHJ1ZSk6CiAgICAiIiJFeGVjdXRlIG9uZSBydW4gZW5kLXRvLWVuZCBhbmQgd3JpdGUgYXJ0aWZhY3RzLiBSZXR1cm5zIGEgcmVzdWx0IGRpY3QuIiIiCiAgICBkZXZpY2UgPSBfc2VsZWN0X2RldmljZSgpCiAgICBpbmZvID0gZGF0YW1vZC5nZXRfaW5mbyhjZmcuZGF0YXNldCkKICAgIHRhc2sgPSBpbmZvWyJ0YXNrIl0KICAgIG5fY2xhc3NlcyA9IGxlbihpbmZvWyJsYWJlbCJdKQoKICAgIHVzZWRfZGV0ID0gc2V0X3NlZWQoY2ZnLnNlZWQsIGRldGVybWluaXN0aWM9Y2ZnLmRldGVybWluaXN0aWMpCiAgICBydW5fZGlyID0gb3MucGF0aC5qb2luKGNmZy5yZXN1bHRzX2RpciwgY2ZnLnJ1bl9uYW1lKCkpCiAgICBvcy5tYWtlZGlycyhydW5fZGlyLCBleGlzdF9vaz1UcnVlKQogICAgbG9nX2ZuKGYiPT09IHtjZmcucnVuX25hbWUoKX0gfCB0YXNrPXt0YXNrfSBjbGFzc2VzPXtuX2NsYXNzZXN9IGRldmljZT17ZGV2aWNlfSA9PT0iKQoKICAgIHNhbXBsZXIgPSBkYXRhbW9kLm1ha2Vfd2VpZ2h0ZWRfc2FtcGxlcihjZmcuZGF0YXNldCwgY2ZnLnJvb3QsIGNmZy5kb3dubG9hZCkgXAogICAgICAgIGlmIGNmZy53ZWlnaHRlZF9zYW1wbGVyIGVsc2UgTm9uZQogICAgbG9hZGVycyA9IGRhdGFtb2QuZ2V0X2xvYWRlcnMoCiAgICAgICAgY2ZnLmRhdGFzZXQsIGNmZy5zaXplLCBiYXRjaF9zaXplPWNmZy5iYXRjaF9zaXplLCByb290PWNmZy5yb290LAogICAgICAgIGRvd25sb2FkPWNmZy5kb3dubG9hZCwgbnVtX3dvcmtlcnM9Y2ZnLm51bV93b3JrZXJzLCBzYW1wbGVyPXNhbXBsZXIsCiAgICAgICAgcGluX21lbW9yeT0oZGV2aWNlID09ICJjdWRhIikpCgogICAgY2xhc3Nfd2VpZ2h0ID0gZGF0YW1vZC5jbGFzc193ZWlnaHRzKGNmZy5kYXRhc2V0LCBjZmcucm9vdCwgY2ZnLmRvd25sb2FkKSBcCiAgICAgICAgaWYgY2ZnLndlaWdodGVkX2xvc3MgZWxzZSBOb25lCgogICAgbW9kZWwgPSBidWlsZF9tb2RlbChjZmcubW9kZWwsIGNmZy5zaXplLCBuX2NsYXNzZXMsIGluX2NoYW5uZWxzPTMsCiAgICAgICAgICAgICAgICAgICAgICAgIHdpZHRoX211bHQ9Y2ZnLndpZHRoX211bHQpCiAgICBuX3BhcmFtcyA9IGNvdW50X3BhcmFtZXRlcnMobW9kZWwpCiAgICBzaXplX21iID0gbW9kZWxfc2l6ZV9tYihtb2RlbCkKICAgIGxvZ19mbihmIm1vZGVsIHBhcmFtcz17bl9wYXJhbXM6LH0gc2l6ZT17c2l6ZV9tYjouMmZ9IE1CIikKCiAgICBpZiBkZXZpY2UgPT0gImN1ZGEiOgogICAgICAgIHRvcmNoLmN1ZGEucmVzZXRfcGVha19tZW1vcnlfc3RhdHMoKQogICAgdDAgPSB0aW1lLnRpbWUoKQogICAgcmVzdWx0ID0gcnVuX3RyYWluaW5nKAogICAgICAgIG1vZGVsLCBsb2FkZXJzLCB0YXNrLCBlcG9jaHM9Y2ZnLmVwb2NocywgbHI9Y2ZnLmxyLCBkZXZpY2U9ZGV2aWNlLAogICAgICAgIGNsYXNzX3dlaWdodD1jbGFzc193ZWlnaHQsIHVzZV9hbXA9Y2ZnLmFtcCwgZGV0ZXJtaW5pc3RpYz1jZmcuZGV0ZXJtaW5pc3RpYywKICAgICAgICBzZWVkPWNmZy5zZWVkLCBja3B0X2Rpcj1ydW5fZGlyLCBja3B0X2V2ZXJ5PWNmZy5ja3B0X2V2ZXJ5LAogICAgICAgIHJlc3VtZT1jZmcucmVzdW1lLCBsb2dfZm49bG9nX2ZuKQogICAgd2FsbCA9IHRpbWUudGltZSgpIC0gdDAKICAgIHBlYWtfZ3B1X21lbV9tYiA9ICh0b3JjaC5jdWRhLm1heF9tZW1vcnlfYWxsb2NhdGVkKCkgLyAoMTAyNCAqKiAyKQogICAgICAgICAgICAgICAgICAgICAgIGlmIGRldmljZSA9PSAiY3VkYSIgZWxzZSBOb25lKQoKICAgICMgU2F2ZSB0ZXN0IHByZWRpY3Rpb25zIGluIHRoZSBtZWRtbmlzdC1jb252ZW50aW9uIGZpbGVuYW1lLgogICAgcHJlZF9uYW1lID0gcHJlZGljdGlvbl9maWxlbmFtZShjZmcuZmxhZygpLCAidGVzdCIsIHJlc3VsdFsidGVzdF9hdWMiXSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgcmVzdWx0WyJ0ZXN0X2FjYyJdLCBjZmcuc2VlZCwgc2l6ZT1jZmcuc2l6ZSkKICAgIHNhdmVfcHJlZGljdGlvbnMocmVzdWx0WyJ0ZXN0X3lfc2NvcmUiXSwgb3MucGF0aC5qb2luKHJ1bl9kaXIsIHByZWRfbmFtZSkpCgogICAgIyBPcHRpb25hbCBvcmFjbGUgYWdyZWVtZW50IGNoZWNrIG9uIHRoZSByZWFsIHRlc3QgcHJlZGljdGlvbnMuCiAgICBhZ3JlZW1lbnQgPSBOb25lCiAgICBpZiB2ZXJpZnlfbWV0cmljczoKICAgICAgICB0cnk6CiAgICAgICAgICAgIGFncmVlbWVudCA9IG1ldHJpY3Ntb2QuY2hlY2tfYWdyZWVtZW50KAogICAgICAgICAgICAgICAgcmVzdWx0WyJ0ZXN0X3lfdHJ1ZSJdLCByZXN1bHRbInRlc3RfeV9zY29yZSJdLCB0YXNrLAogICAgICAgICAgICAgICAgZmxhZz1jZmcuZmxhZygpLCBzcGxpdD0idGVzdCIsIHNpemU9MjgsIHJvb3Q9Y2ZnLnJvb3QsCiAgICAgICAgICAgICAgICB0b2w9MWUtMywgdmVyYm9zZT1GYWxzZSkKICAgICAgICAgICAgbG9nX2ZuKGYiW3ZlcmlmeV0gbWV0cmljcyBhZ3JlZSB3aXRoIG1lZG1uaXN0LkV2YWx1YXRvciAoYXVjPXthZ3JlZW1lbnRbJ2F1YyddOi40Zn0pIikKICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6ICAjIG5ldmVyIGxldCB2ZXJpZmljYXRpb24ga2lsbCBhIGNvbXBsZXRlZCBydW4KICAgICAgICAgICAgbG9nX2ZuKGYiW3ZlcmlmeV0gV0FSTklORyBhZ3JlZW1lbnQgY2hlY2sgc2tpcHBlZDoge2V9IikKCiAgICByZWYgPSBSRUZFUkVOQ0UuZ2V0KChjZmcuZGF0YXNldCwgY2ZnLm1vZGVsLCBjZmcuc2l6ZSkpCiAgICBydW5fanNvbiA9IGRpY3QoCiAgICAgICAgY29uZmlnPWFzZGljdChjZmcpLAogICAgICAgIHRhc2s9dGFzaywgbl9jbGFzc2VzPW5fY2xhc3NlcywKICAgICAgICBuX3BhcmFtcz1uX3BhcmFtcywgbW9kZWxfc2l6ZV9tYj1zaXplX21iLAogICAgICAgIGJlc3RfZXBvY2g9cmVzdWx0WyJiZXN0X2Vwb2NoIl0sIGJlc3RfdmFsX2F1Yz1yZXN1bHRbImJlc3RfdmFsX2F1YyJdLAogICAgICAgIHRyYWluX2F1Yz1yZXN1bHRbInRyYWluX2F1YyJdLCB0cmFpbl9hY2M9cmVzdWx0WyJ0cmFpbl9hY2MiXSwKICAgICAgICB2YWxfYXVjPXJlc3VsdFsidmFsX2F1YyJdLCB2YWxfYWNjPXJlc3VsdFsidmFsX2FjYyJdLAogICAgICAgIHRlc3RfYXVjPXJlc3VsdFsidGVzdF9hdWMiXSwgdGVzdF9hY2M9cmVzdWx0WyJ0ZXN0X2FjYyJdLAogICAgICAgIHJlZmVyZW5jZV9hdWM9KHJlZlswXSBpZiByZWYgZWxzZSBOb25lKSwKICAgICAgICByZWZlcmVuY2VfYWNjPShyZWZbMV0gaWYgcmVmIGVsc2UgTm9uZSksCiAgICAgICAgZGVsdGFfYXVjPShyZXN1bHRbInRlc3RfYXVjIl0gLSByZWZbMF0gaWYgcmVmIGVsc2UgTm9uZSksCiAgICAgICAgZGVsdGFfYWNjPShyZXN1bHRbInRlc3RfYWNjIl0gLSByZWZbMV0gaWYgcmVmIGVsc2UgTm9uZSksCiAgICAgICAgd2FsbF9jbG9ja19zPXdhbGwsCiAgICAgICAgcGVha19ncHVfbWVtX21iPXBlYWtfZ3B1X21lbV9tYiwKICAgICAgICBkZXRlcm1pbmlzdGljX2N1ZG5uPXVzZWRfZGV0LAogICAgICAgIGFtcD1jZmcuYW1wLAogICAgICAgIHNlZWQ9Y2ZnLnNlZWQsCiAgICAgICAgcHJlZGljdGlvbl9maWxlPXByZWRfbmFtZSwKICAgICAgICBoaXN0b3J5PXJlc3VsdFsiaGlzdG9yeSJdLAogICAgICAgIGFncmVlbWVudD1hZ3JlZW1lbnQsCiAgICAgICAgdmVyc2lvbnM9X3ZlcnNpb25zKCksCiAgICApCiAgICB3aXRoIG9wZW4ob3MucGF0aC5qb2luKHJ1bl9kaXIsICJydW4uanNvbiIpLCAidyIpIGFzIGY6CiAgICAgICAganNvbi5kdW1wKHJ1bl9qc29uLCBmLCBpbmRlbnQ9MikKCiAgICBsb2dfZm4oZiJET05FIHtjZmcucnVuX25hbWUoKX0gdGVzdF9hdWM9e3Jlc3VsdFsndGVzdF9hdWMnXTouNGZ9ICIKICAgICAgICAgICBmInRlc3RfYWNjPXtyZXN1bHRbJ3Rlc3RfYWNjJ106LjRmfSAiCiAgICAgICAgICAgKyAoZiIocGFwZXIge3JlZlswXTouM2Z9L3tyZWZbMV06LjNmfSkiIGlmIHJlZiBlbHNlICIiKQogICAgICAgICAgICsgZiIgW3t3YWxsLzYwOi4xZn0gbWluXSIpCiAgICByZXR1cm4gcnVuX2pzb24KCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSAjCiMgUnVuLW1hdHJpeCBlbnVtZXJhdGlvbiAodGllcnMpCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tICMKCiMgVGhlIGVsZXZlbiBub24tRGVybWEgTWVkTU5JU1QyRCBkYXRhc2V0cyBhcmUgc3BsaXQgYnkgZGF0YSBzY2FsZS4gVGhlIHNtYWxsIC8KIyBtZWRpdW0gb25lcyBnZXQgdGhyZWUgc2VlZHM7IHRoZSBmb3VyIGxhcmdlIG9uZXMgZ2V0IGEgc2luZ2xlIHNlZWQgKHNlZWQgMCksCiMgYmVjYXVzZSBhIHNpbmdsZSBydW4gaXMgc3RhYmxlIGF0IHRoYXQgZGF0YSBzY2FsZSBhbmQgdGhyZWUgd291bGQgdHJpcGxlIHRoZQojIGNvc3QuIFRoaXMgc3BsaXQgaXMgYSBjb21wdXRlIGRlY2lzaW9uLCBkaXNjbG9zZWQgaW4gdGhlIHdyaXRldXAuClNNQUxMX01FRElVTV9EQVRBU0VUUyA9ICgKICAgICJyZXRpbmFtbmlzdCIsICJicmVhc3RtbmlzdCIsICJwbmV1bW9uaWFtbmlzdCIsICJibG9vZG1uaXN0IiwKICAgICJvcmdhbmFtbmlzdCIsICJvcmdhbmNtbmlzdCIsICJvcmdhbnNtbmlzdCIsCikKTEFSR0VfREFUQVNFVFMgPSAoInRpc3N1ZW1uaXN0IiwgIm9jdG1uaXN0IiwgInBhdGhtbmlzdCIsICJjaGVzdG1uaXN0IikKCgpkZWYgcnVuX21hdHJpeChzZWVkcz0oMCwgMSwgMiksIG9jdG1uaXN0X3NlY29uZF9zZWVkPUZhbHNlKToKICAgICIiIlJldHVybiB0aGUgdGllcmVkIGxpc3Qgb2YgYGAodGllciwgZGF0YXNldCwgbW9kZWwsIHNpemUsIHNlZWQpYGAgdHVwbGVzLgoKICAgIFRoaXMgaXMgdGhlIGNvbXB1dGUtbWluaW1hbCBzbGljZSBvZiBNZWRNTklTVCB2MiBUYWJsZSAzIHRoZSByZXBsaWNhdGlvbgogICAgdGFyZ2V0czoKCiAgICAqICoqVGllciAxKiog4oCUIERlcm1hTU5JU1QgKHByaW1hcnkpLCBSZXNOZXQtMTgvNTAgeCBzaXplcyAyOC8yMjQsIGFsbAogICAgICBgYHNlZWRzYGAuIFR3ZWx2ZSBydW5zIGF0IHRocmVlIHNlZWRzLgogICAgKiAqKlRpZXIgMioqIOKAlCB0aGUgc21hbGwvbWVkaXVtIGRhdGFzZXRzIGF0IFJlc05ldC0xOCBAIDI4LCBhbGwgYGBzZWVkc2BgLgogICAgKiAqKlRpZXIgMyoqIOKAlCB0aGUgZm91ciBsYXJnZSBkYXRhc2V0cyBhdCBSZXNOZXQtMTggQCAyOCwgKipzZWVkIDAgb25seSoqCiAgICAgIChzaW5nbGUtc2VlZCBmb3IgY29tcHV0ZSByZWFzb25zOyBzdGFibGUgYXQgdGhhdCBkYXRhIHNjYWxlKS4KCiAgICBgYG9jdG1uaXN0X3NlY29uZF9zZWVkPVRydWVgYCBhZGRzIGEgYGBzZWVkIDFgYCBPQ1RNTklTVCBydW46IGl0J3MgVGllcgogICAgMydzIG9ubHkgdG9sZXJhbmNlIG1pc3MgKGRlbHRhIGFjY3VyYWN5ID0gLTAuMDQ4KSwgc28gYSBzaW5nbGUgc2VlZAogICAgY2FuJ3QgZGlzdGluZ3Vpc2ggYSBzeXN0ZW1hdGljIGdhcCBmcm9tIHNpbmdsZS1zZWVkIG5vaXNlLiBMZWZ0IG9mZiBieQogICAgZGVmYXVsdCAofjIuMTUgR1BVLWhvdXJzIG9uIGEgUDEwMCwgYmFzZWQgb24gdGhlIHNlZWQtMCBydW4ncyByZWNvcmRlZAogICAgd2FsbC1jbG9jaykgLS0gZW5hYmxlIGl0IGV4cGxpY2l0bHkgb25jZSBHUFUgYnVkZ2V0IGFsbG93cy4KICAgICIiIgogICAgbWF0cml4ID0gW10KICAgICMgVGllciAxOiBEZXJtYU1OSVNULCBhbGwgNCBtb2RlbCB4IHNpemUsIGFsbCBzZWVkcy4KICAgIGZvciBtb2RlbCBpbiAoInJlc25ldDE4IiwgInJlc25ldDUwIik6CiAgICAgICAgZm9yIHNpemUgaW4gKDI4LCAyMjQpOgogICAgICAgICAgICBmb3IgcyBpbiBzZWVkczoKICAgICAgICAgICAgICAgIG1hdHJpeC5hcHBlbmQoKDEsICJkZXJtYW1uaXN0IiwgbW9kZWwsIHNpemUsIHMpKQogICAgIyBUaWVyIDI6IHNtYWxsL21lZGl1bSBkYXRhc2V0cywgUmVzTmV0LTE4IEAgMjgsIGFsbCBzZWVkcy4KICAgIGZvciBkYXRhc2V0IGluIFNNQUxMX01FRElVTV9EQVRBU0VUUzoKICAgICAgICBmb3IgcyBpbiBzZWVkczoKICAgICAgICAgICAgbWF0cml4LmFwcGVuZCgoMiwgZGF0YXNldCwgInJlc25ldDE4IiwgMjgsIHMpKQogICAgIyBUaWVyIDM6IGxhcmdlIGRhdGFzZXRzLCBSZXNOZXQtMTggQCAyOCwgc2luZ2xlIHNlZWQgKHNlZWQgMCksIGV4Y2VwdAogICAgIyBPQ1RNTklTVCdzIG9wdC1pbiBzZWNvbmQgc2VlZCAoc2VlIGRvY3N0cmluZykuCiAgICBmb3IgZGF0YXNldCBpbiBMQVJHRV9EQVRBU0VUUzoKICAgICAgICBtYXRyaXguYXBwZW5kKCgzLCBkYXRhc2V0LCAicmVzbmV0MTgiLCAyOCwgMCkpCiAgICAgICAgaWYgZGF0YXNldCA9PSAib2N0bW5pc3QiIGFuZCBvY3RtbmlzdF9zZWNvbmRfc2VlZDoKICAgICAgICAgICAgbWF0cml4LmFwcGVuZCgoMywgZGF0YXNldCwgInJlc25ldDE4IiwgMjgsIDEpKQogICAgcmV0dXJuIG1hdHJpeAoKCmRlZiBfcGFyc2VfYXJncyhhcmd2PU5vbmUpOgogICAgcCA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKGRlc2NyaXB0aW9uPSJNZWRNTklTVCB2MiByZXBsaWNhdGlvbjogc2luZ2xlIHJ1biIpCiAgICBwLmFkZF9hcmd1bWVudCgiLS1kYXRhc2V0IiwgZGVmYXVsdD0iZGVybWFtbmlzdCIpCiAgICBwLmFkZF9hcmd1bWVudCgiLS1tb2RlbCIsIGRlZmF1bHQ9InJlc25ldDE4IiwgY2hvaWNlcz1bInJlc25ldDE4IiwgInJlc25ldDUwIl0pCiAgICBwLmFkZF9hcmd1bWVudCgiLS1zaXplIiwgdHlwZT1pbnQsIGRlZmF1bHQ9MjgsIGNob2ljZXM9WzI4LCAyMjRdKQogICAgcC5hZGRfYXJndW1lbnQoIi0tc2VlZCIsIHR5cGU9aW50LCBkZWZhdWx0PTApCiAgICBwLmFkZF9hcmd1bWVudCgiLS1lcG9jaHMiLCB0eXBlPWludCwgZGVmYXVsdD0xMDApCiAgICBwLmFkZF9hcmd1bWVudCgiLS1iYXRjaC1zaXplIiwgdHlwZT1pbnQsIGRlZmF1bHQ9MTI4KQogICAgcC5hZGRfYXJndW1lbnQoIi0tbHIiLCB0eXBlPWZsb2F0LCBkZWZhdWx0PTFlLTMpCiAgICBwLmFkZF9hcmd1bWVudCgiLS1hbXAiLCBhY3Rpb249InN0b3JlX3RydWUiKQogICAgcC5hZGRfYXJndW1lbnQoIi0tbm8tZGV0ZXJtaW5pc3RpYyIsIGRlc3Q9ImRldGVybWluaXN0aWMiLCBhY3Rpb249InN0b3JlX2ZhbHNlIikKICAgIHAuYWRkX2FyZ3VtZW50KCItLXdpZHRoLW11bHQiLCB0eXBlPWZsb2F0LCBkZWZhdWx0PTEuMCkKICAgIHAuYWRkX2FyZ3VtZW50KCItLXdlaWdodGVkLXNhbXBsZXIiLCBhY3Rpb249InN0b3JlX3RydWUiKQogICAgcC5hZGRfYXJndW1lbnQoIi0td2VpZ2h0ZWQtbG9zcyIsIGFjdGlvbj0ic3RvcmVfdHJ1ZSIpCiAgICBwLmFkZF9hcmd1bWVudCgiLS1udW0td29ya2VycyIsIHR5cGU9aW50LCBkZWZhdWx0PTIpCiAgICBwLmFkZF9hcmd1bWVudCgiLS1ja3B0LWV2ZXJ5IiwgdHlwZT1pbnQsIGRlZmF1bHQ9NSkKICAgIHAuYWRkX2FyZ3VtZW50KCItLW5vLXJlc3VtZSIsIGRlc3Q9InJlc3VtZSIsIGFjdGlvbj0ic3RvcmVfZmFsc2UiKQogICAgcC5hZGRfYXJndW1lbnQoIi0tbm8tZG93bmxvYWQiLCBkZXN0PSJkb3dubG9hZCIsIGFjdGlvbj0ic3RvcmVfZmFsc2UiKQogICAgcC5hZGRfYXJndW1lbnQoIi0tcm9vdCIsIGRlZmF1bHQ9Tm9uZSkKICAgIHAuYWRkX2FyZ3VtZW50KCItLXJlc3VsdHMtZGlyIiwgZGVmYXVsdD0icmVzdWx0cyIpCiAgICBwLmFkZF9hcmd1bWVudCgiLS10YWciLCBkZWZhdWx0PSIiKQogICAgcmV0dXJuIHAucGFyc2VfYXJncyhhcmd2KQoKCmRlZiBtYWluKGFyZ3Y9Tm9uZSk6CiAgICBhcmdzID0gX3BhcnNlX2FyZ3MoYXJndikKICAgIGNmZyA9IFJ1bkNvbmZpZygqKnZhcnMoYXJncykpCiAgICBydW5fY29uZmlnKGNmZykKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg==',
    'train.py': 'IiIiVHJhaW5pbmcgbG9vcDogc2VlZGluZywgYmVzdC12YWwtQVVDIGNoZWNrcG9pbnRpbmcsIHJlc3VtZSwgb3B0aW9uYWwgQU1QLgoKUHJvdG9jb2wgKE1lZE1OSVNUIHYyIGJhc2VsaW5lKToKICAgIEFkYW0obHI9MWUtMyksIE11bHRpU3RlcExSKGdhbW1hPTAuMSwgbWlsZXN0b25lcz1bMC41RSwgMC43NUVdKSwKICAgIGJhdGNoIDEyOCwgMTAwIGVwb2NocywgQ3Jvc3NFbnRyb3B5TG9zcyAoQkNFV2l0aExvZ2l0cyBmb3IgbXVsdGktbGFiZWwpLAogICAgbm8gYXVnbWVudGF0aW9uLiBNb2RlbCBzZWxlY3Rpb24gPSBjaGVja3BvaW50IHdpdGggdGhlIGhpZ2hlc3QgdmFsaWRhdGlvbgogICAgbWFjcm8tQVVDIHNlZW4gc28gZmFyOyB0aGF0IGNoZWNrcG9pbnQncyB0ZXN0IEFVQy9BQ0MgaXMgcmVwb3J0ZWQuCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IG9zCmltcG9ydCB0aW1lCmltcG9ydCByYW5kb20KaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCB0b3JjaAppbXBvcnQgdG9yY2gubm4gYXMgbm4KCmZyb20gLmV2YWx1YXRlIGltcG9ydCBldmFsdWF0ZV9zcGxpdCwgbG9hZF9jaGVja3BvaW50CgoKZGVmIHNldF9zZWVkKHNlZWQsIGRldGVybWluaXN0aWM9VHJ1ZSk6CiAgICAiIiJTZWVkIHB5dGhvbiAvIG51bXB5IC8gdG9yY2ggLyBDVURBLiBSZXR1cm5zIHdoZXRoZXIgY3Vkbm4gaXMgZGV0ZXJtaW5pc3RpYy4iIiIKICAgIHJhbmRvbS5zZWVkKHNlZWQpCiAgICBucC5yYW5kb20uc2VlZChzZWVkKQogICAgdG9yY2gubWFudWFsX3NlZWQoc2VlZCkKICAgIHRvcmNoLmN1ZGEubWFudWFsX3NlZWRfYWxsKHNlZWQpCiAgICBpZiBkZXRlcm1pbmlzdGljOgogICAgICAgIHRvcmNoLmJhY2tlbmRzLmN1ZG5uLmRldGVybWluaXN0aWMgPSBUcnVlCiAgICAgICAgdG9yY2guYmFja2VuZHMuY3Vkbm4uYmVuY2htYXJrID0gRmFsc2UKICAgIGVsc2U6CiAgICAgICAgdG9yY2guYmFja2VuZHMuY3Vkbm4uZGV0ZXJtaW5pc3RpYyA9IEZhbHNlCiAgICAgICAgdG9yY2guYmFja2VuZHMuY3Vkbm4uYmVuY2htYXJrID0gVHJ1ZQogICAgcmV0dXJuIGRldGVybWluaXN0aWMKCgpkZWYgbWFrZV9jcml0ZXJpb24odGFzaywgY2xhc3Nfd2VpZ2h0PU5vbmUsIGRldmljZT1Ob25lKToKICAgIGlmIHRhc2sgPT0gIm11bHRpLWxhYmVsLCBiaW5hcnktY2xhc3MiOgogICAgICAgIHJldHVybiBubi5CQ0VXaXRoTG9naXRzTG9zcygpCiAgICB3ID0gY2xhc3Nfd2VpZ2h0LnRvKGRldmljZSkgaWYgKGNsYXNzX3dlaWdodCBpcyBub3QgTm9uZSBhbmQgZGV2aWNlIGlzIG5vdCBOb25lKSBlbHNlIGNsYXNzX3dlaWdodAogICAgcmV0dXJuIG5uLkNyb3NzRW50cm9weUxvc3Mod2VpZ2h0PXcpCgoKZGVmIF90YXJnZXRzX2Zvcl9sb3NzKHksIHRhc2ssIGRldmljZSk6CiAgICBpZiB0YXNrID09ICJtdWx0aS1sYWJlbCwgYmluYXJ5LWNsYXNzIjoKICAgICAgICByZXR1cm4geS50byh0b3JjaC5mbG9hdDMyKS50byhkZXZpY2UpCiAgICByZXR1cm4geS5zcXVlZXplKGRpbT0xKS5sb25nKCkudG8oZGV2aWNlKSBpZiB5Lm5kaW0gPT0gMiBlbHNlIHkubG9uZygpLnRvKGRldmljZSkKCgpkZWYgdHJhaW5fb25lX2Vwb2NoKG1vZGVsLCBsb2FkZXIsIGNyaXRlcmlvbiwgb3B0aW1pemVyLCB0YXNrLCBkZXZpY2UsCiAgICAgICAgICAgICAgICAgICAgc2NhbGVyPU5vbmUsIHVzZV9hbXA9RmFsc2UpOgogICAgbW9kZWwudHJhaW4oKQogICAgZGV2aWNlX3R5cGUgPSAiY3VkYSIgaWYgImN1ZGEiIGluIHN0cihkZXZpY2UpIGVsc2UgImNwdSIKICAgIHJ1bm5pbmcsIG5faW1ncywgdDAgPSAwLjAsIDAsIHRpbWUudGltZSgpCiAgICBmb3IgeCwgeSBpbiBsb2FkZXI6CiAgICAgICAgeCA9IHgudG8oZGV2aWNlLCBub25fYmxvY2tpbmc9VHJ1ZSkKICAgICAgICB0YXJnZXQgPSBfdGFyZ2V0c19mb3JfbG9zcyh5LCB0YXNrLCBkZXZpY2UpCiAgICAgICAgb3B0aW1pemVyLnplcm9fZ3JhZChzZXRfdG9fbm9uZT1UcnVlKQogICAgICAgIHdpdGggdG9yY2guYW1wLmF1dG9jYXN0KGRldmljZV90eXBlLCBlbmFibGVkPXVzZV9hbXApOgogICAgICAgICAgICBsb2dpdHMgPSBtb2RlbCh4KQogICAgICAgICAgICBsb3NzID0gY3JpdGVyaW9uKGxvZ2l0cywgdGFyZ2V0KQogICAgICAgIGlmIHVzZV9hbXA6CiAgICAgICAgICAgIHNjYWxlci5zY2FsZShsb3NzKS5iYWNrd2FyZCgpCiAgICAgICAgICAgIHNjYWxlci5zdGVwKG9wdGltaXplcikKICAgICAgICAgICAgc2NhbGVyLnVwZGF0ZSgpCiAgICAgICAgZWxzZToKICAgICAgICAgICAgbG9zcy5iYWNrd2FyZCgpCiAgICAgICAgICAgIG9wdGltaXplci5zdGVwKCkKICAgICAgICBydW5uaW5nICs9IGxvc3MuaXRlbSgpICogeC5zaXplKDApCiAgICAgICAgbl9pbWdzICs9IHguc2l6ZSgwKQogICAgZHQgPSB0aW1lLnRpbWUoKSAtIHQwCiAgICByZXR1cm4gcnVubmluZyAvIG1heChuX2ltZ3MsIDEpLCBuX2ltZ3MgLyBtYXgoZHQsIDFlLTkpLCBkdAoKCmRlZiBydW5fdHJhaW5pbmcobW9kZWwsIGxvYWRlcnMsIHRhc2ssICosIGVwb2Nocz0xMDAsIGxyPTFlLTMsIGRldmljZT0iY3VkYSIsCiAgICAgICAgICAgICAgICAgY2xhc3Nfd2VpZ2h0PU5vbmUsIHVzZV9hbXA9RmFsc2UsIGRldGVybWluaXN0aWM9VHJ1ZSwgc2VlZD0wLAogICAgICAgICAgICAgICAgIGNrcHRfZGlyPU5vbmUsIGNrcHRfZXZlcnk9NSwgcmVzdW1lPVRydWUsIGxvZ19mbj1wcmludCk6CiAgICAiIiJUcmFpbiwgc2VsZWN0aW5nIHRoZSBiZXN0LXZhbC1BVUMgY2hlY2twb2ludC4gUmVzdW1hYmxlLgoKICAgIFJldHVybnMgYSBoaXN0b3J5L3Jlc3VsdCBkaWN0LiBXcml0ZXMgYGBiZXN0X21vZGVsLnB0aGBgIGFuZCBwZXJpb2RpYwogICAgYGBsYXN0LnB0aGBgIGludG8gYGBja3B0X2RpcmBgIHNvIGEgcnVuIGNhbiBjb250aW51ZSBpbiBhIGZyZXNoIEthZ2dsZQogICAgc2Vzc2lvbiB2aWEgYGByZXN1bWU9VHJ1ZWBgLgogICAgIiIiCiAgICB0cmFpbl9sb2FkZXIsIHZhbF9sb2FkZXIsIHRlc3RfbG9hZGVyID0gbG9hZGVycwogICAgbW9kZWwgPSBtb2RlbC50byhkZXZpY2UpCgogICAgY3JpdGVyaW9uID0gbWFrZV9jcml0ZXJpb24odGFzaywgY2xhc3Nfd2VpZ2h0PWNsYXNzX3dlaWdodCwgZGV2aWNlPWRldmljZSkKICAgIG9wdGltaXplciA9IHRvcmNoLm9wdGltLkFkYW0obW9kZWwucGFyYW1ldGVycygpLCBscj1scikKICAgIG1pbGVzdG9uZXMgPSBbaW50KDAuNSAqIGVwb2NocyksIGludCgwLjc1ICogZXBvY2hzKV0KICAgIHNjaGVkdWxlciA9IHRvcmNoLm9wdGltLmxyX3NjaGVkdWxlci5NdWx0aVN0ZXBMUihvcHRpbWl6ZXIsIG1pbGVzdG9uZXM9bWlsZXN0b25lcywgZ2FtbWE9MC4xKQogICAgc2NhbGVyID0gdG9yY2guYW1wLkdyYWRTY2FsZXIoImN1ZGEiIGlmICJjdWRhIiBpbiBzdHIoZGV2aWNlKSBlbHNlICJjcHUiLCBlbmFibGVkPXVzZV9hbXApCgogICAgc3RhcnRfZXBvY2ggPSAwCiAgICBiZXN0X3ZhbF9hdWMgPSAtMS4wCiAgICBiZXN0X2Vwb2NoID0gLTEKICAgIGhpc3RvcnkgPSBbXQoKICAgIGxhc3RfcGF0aCA9IG9zLnBhdGguam9pbihja3B0X2RpciwgImxhc3QucHRoIikgaWYgY2twdF9kaXIgZWxzZSBOb25lCiAgICBiZXN0X3BhdGggPSBvcy5wYXRoLmpvaW4oY2twdF9kaXIsICJiZXN0X21vZGVsLnB0aCIpIGlmIGNrcHRfZGlyIGVsc2UgTm9uZQogICAgaWYgY2twdF9kaXI6CiAgICAgICAgb3MubWFrZWRpcnMoY2twdF9kaXIsIGV4aXN0X29rPVRydWUpCgogICAgaWYgcmVzdW1lIGFuZCBsYXN0X3BhdGggYW5kIG9zLnBhdGguZXhpc3RzKGxhc3RfcGF0aCk6CiAgICAgICAgY2twdCA9IGxvYWRfY2hlY2twb2ludChsYXN0X3BhdGgsIG1vZGVsLCBvcHRpbWl6ZXIsIHNjaGVkdWxlciwgc2NhbGVyLCBtYXBfbG9jYXRpb249ZGV2aWNlKQogICAgICAgIHN0YXJ0X2Vwb2NoID0gY2twdFsiZXBvY2giXSArIDEKICAgICAgICBiZXN0X3ZhbF9hdWMgPSBja3B0LmdldCgiYmVzdF92YWxfYXVjIiwgLTEuMCkKICAgICAgICBiZXN0X2Vwb2NoID0gY2twdC5nZXQoImJlc3RfZXBvY2giLCAtMSkKICAgICAgICBoaXN0b3J5ID0gY2twdC5nZXQoImhpc3RvcnkiLCBbXSkKICAgICAgICBsb2dfZm4oZiJbcmVzdW1lXSBjb250aW51aW5nIGZyb20gZXBvY2gge3N0YXJ0X2Vwb2NofSAoYmVzdF92YWxfYXVjPXtiZXN0X3ZhbF9hdWM6LjRmfSkiKQoKICAgIGZvciBlcG9jaCBpbiByYW5nZShzdGFydF9lcG9jaCwgZXBvY2hzKToKICAgICAgICBsb3NzLCBpcHMsIGR0ID0gdHJhaW5fb25lX2Vwb2NoKAogICAgICAgICAgICBtb2RlbCwgdHJhaW5fbG9hZGVyLCBjcml0ZXJpb24sIG9wdGltaXplciwgdGFzaywgZGV2aWNlLCBzY2FsZXIsIHVzZV9hbXApCiAgICAgICAgc2NoZWR1bGVyLnN0ZXAoKQoKICAgICAgICB2YWxfYXVjLCB2YWxfYWNjLCBfLCBfID0gZXZhbHVhdGVfc3BsaXQobW9kZWwsIHZhbF9sb2FkZXIsIHRhc2ssIGRldmljZSwgdXNlX2FtcCkKICAgICAgICBpc19iZXN0ID0gdmFsX2F1YyA+IGJlc3RfdmFsX2F1YwogICAgICAgIGlmIGlzX2Jlc3Q6CiAgICAgICAgICAgIGJlc3RfdmFsX2F1YywgYmVzdF9lcG9jaCA9IHZhbF9hdWMsIGVwb2NoCiAgICAgICAgICAgIGlmIGJlc3RfcGF0aDoKICAgICAgICAgICAgICAgIHRvcmNoLnNhdmUoeyJtb2RlbCI6IG1vZGVsLnN0YXRlX2RpY3QoKSwgImVwb2NoIjogZXBvY2gsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAidmFsX2F1YyI6IHZhbF9hdWMsICJ2YWxfYWNjIjogdmFsX2FjY30sIGJlc3RfcGF0aCkKCiAgICAgICAgaGlzdG9yeS5hcHBlbmQoZGljdChlcG9jaD1lcG9jaCwgdHJhaW5fbG9zcz1sb3NzLCB2YWxfYXVjPXZhbF9hdWMsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICB2YWxfYWNjPXZhbF9hY2MsIGltZ3NfcGVyX3NlYz1pcHMsIGVwb2NoX3RpbWVfcz1kdCwKICAgICAgICAgICAgICAgICAgICAgICAgICAgIGxyPW9wdGltaXplci5wYXJhbV9ncm91cHNbMF1bImxyIl0pKQogICAgICAgIGxvZ19mbihmIltle2Vwb2NoOjAzZH1dIGxvc3M9e2xvc3M6LjRmfSB2YWxfYXVjPXt2YWxfYXVjOi40Zn0gIgogICAgICAgICAgICAgICBmInZhbF9hY2M9e3ZhbF9hY2M6LjRmfSB7aXBzOi4wZn0gaW1nL3Mge2R0Oi4xZn1zIgogICAgICAgICAgICAgICArICgiICAqYmVzdCoiIGlmIGlzX2Jlc3QgZWxzZSAiIikpCgogICAgICAgIGlmIGxhc3RfcGF0aCBhbmQgKChlcG9jaCArIDEpICUgY2twdF9ldmVyeSA9PSAwIG9yIGVwb2NoID09IGVwb2NocyAtIDEpOgogICAgICAgICAgICB0b3JjaC5zYXZlKHsibW9kZWwiOiBtb2RlbC5zdGF0ZV9kaWN0KCksICJvcHRpbWl6ZXIiOiBvcHRpbWl6ZXIuc3RhdGVfZGljdCgpLAogICAgICAgICAgICAgICAgICAgICAgICAic2NoZWR1bGVyIjogc2NoZWR1bGVyLnN0YXRlX2RpY3QoKSwgInNjYWxlciI6IHNjYWxlci5zdGF0ZV9kaWN0KCksCiAgICAgICAgICAgICAgICAgICAgICAgICJlcG9jaCI6IGVwb2NoLCAiYmVzdF92YWxfYXVjIjogYmVzdF92YWxfYXVjLAogICAgICAgICAgICAgICAgICAgICAgICAiYmVzdF9lcG9jaCI6IGJlc3RfZXBvY2gsICJoaXN0b3J5IjogaGlzdG9yeX0sIGxhc3RfcGF0aCkKCiAgICAjIFJlbG9hZCBiZXN0IGNoZWNrcG9pbnQgZm9yIGZpbmFsIHJlcG9ydGluZy4KICAgIGlmIGJlc3RfcGF0aCBhbmQgb3MucGF0aC5leGlzdHMoYmVzdF9wYXRoKToKICAgICAgICBsb2FkX2NoZWNrcG9pbnQoYmVzdF9wYXRoLCBtb2RlbCwgbWFwX2xvY2F0aW9uPWRldmljZSkKCiAgICB0cmFpbl9hdWMsIHRyYWluX2FjYywgXywgXyA9IGV2YWx1YXRlX3NwbGl0KG1vZGVsLCB0cmFpbl9sb2FkZXIsIHRhc2ssIGRldmljZSwgdXNlX2FtcCkKICAgIHZhbF9hdWMsIHZhbF9hY2MsIF8sIF8gPSBldmFsdWF0ZV9zcGxpdChtb2RlbCwgdmFsX2xvYWRlciwgdGFzaywgZGV2aWNlLCB1c2VfYW1wKQogICAgdGVzdF9hdWMsIHRlc3RfYWNjLCB5X3RydWUsIHlfc2NvcmUgPSBldmFsdWF0ZV9zcGxpdChtb2RlbCwgdGVzdF9sb2FkZXIsIHRhc2ssIGRldmljZSwgdXNlX2FtcCkKCiAgICByZXR1cm4gZGljdCgKICAgICAgICBiZXN0X2Vwb2NoPWJlc3RfZXBvY2gsIGJlc3RfdmFsX2F1Yz1iZXN0X3ZhbF9hdWMsCiAgICAgICAgdHJhaW5fYXVjPXRyYWluX2F1YywgdHJhaW5fYWNjPXRyYWluX2FjYywKICAgICAgICB2YWxfYXVjPXZhbF9hdWMsIHZhbF9hY2M9dmFsX2FjYywKICAgICAgICB0ZXN0X2F1Yz10ZXN0X2F1YywgdGVzdF9hY2M9dGVzdF9hY2MsCiAgICAgICAgaGlzdG9yeT1oaXN0b3J5LCB0ZXN0X3lfdHJ1ZT15X3RydWUsIHRlc3RfeV9zY29yZT15X3Njb3JlLAogICAgKQo=',
}

import base64, os, importlib, sys
os.makedirs("src", exist_ok=True)
for _name, _b64 in SRC_B64.items():
    with open(os.path.join("src", _name), "wb") as _f:
        _f.write(base64.b64decode(_b64))
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
for _m in [k for k in list(sys.modules) if k == "src" or k.startswith("src.")]:
    del sys.modules[_m]
import src
print("wrote src package:", sorted(SRC_B64))
import torch, torchvision
print("torch", torch.__version__, "| torchvision", torchvision.__version__,
      "| cuda", torch.cuda.is_available(),
      "|", (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"))


## 2. Metric agreement test (our AUC/ACC vs `medmnist.Evaluator`, tol 1e-6)

In [ ]:
from src import metrics_test
metrics_test.main()   # raises if our metrics drift from the medmnist oracle


## 3. CONFIG — edit this cell each session

`MODE` selects what runs. In `baselines` mode, pick the `TIERS` and a
`MAX_MINUTES` budget you can complete before Kaggle's cap.

In [ ]:
import os

# "all" runs baselines -> extensions -> report in one session, which is what you
# want once the bulk of the matrix is already done: a single Save & Run All
# produces every table and figure. Use the individual modes when a phase is too
# big to finish inside Kaggle's ~12h session cap.
MODE = "extensions"         # scoped run: no existing checkpoints reachable this
                             # session (teammate's prior output is private), so
                             # this trains only what the extensions section needs
                             # -- not the full baseline matrix ("all"/"baselines").

# SMOKE=True gives a 3-epoch, seed-0, DermaMNIST-@28-only pass in ~2 minutes.
# It exercises the whole path (download -> model -> train loop -> metric oracle
# -> run.json) before any quota goes into 100-epoch runs, and writes to a
# separate *_smoke directory so it can never be mistaken for a real result.
SMOKE = False

# --- baselines-mode knobs ---
# TIERS is a subset of [1, 2, 3]:
#   T1 = DermaMNIST, R18/R50 (x sizes in SIZES), all SEEDS  -> 6 runs at SIZES=[28]
#   T2 = 7 small/medium datasets, R18 @ 28, all SEEDS       -> 21 runs
#   T3 = 4 large datasets (Tissue/OCT/Path/Chest), R18 @ 28 -> 4 runs, seed 0 only
#
# Session 1 completed T1 + T2 (27 runs) and started T3. TIERS=[3] finishes the
# large datasets; anything already carrying a full run.json is skipped, so this
# is safe to re-run and costs nothing for work already done.
#
# SIZES filters by input size. [28] is deliberate: the 224 configs cost ~25
# GPU-h alone, and DermaMNIST @224 is already covered by the reproduction arm
# (see report/reproduction_arm.md). Set [28, 224] only to replicate 224 too.
TIERS = [3]
SIZES = [28]
SEEDS = [0]                 # single-seed for this scoped run (no checkpoints to
                             # reuse); adds ~2h of GPU time per seed if raised to
                             # [0, 1, 2] later. T3 always uses seed 0 regardless.
OCTMNIST_SECOND_SEED = False  # True adds OCTMNIST seed 1 (~2.15 GPU-h on a P100);
                               # it's Tier 3's only tolerance miss, but left off by
                               # default to conserve GPU quota -- flip on deliberately.
EPOCHS = 100
USE_AMP = False             # True ~halves 224 runtime; flagged in run.json if used
DETERMINISTIC = True
MAX_MINUTES = 480           # stop STARTING new baseline runs after this many minutes

# --- persistence for resume across sessions ---
WORK_RESULTS = "/kaggle/working/results" if os.path.isdir("/kaggle/working") else "results"

# Previous session's committed output, attached as an input dataset. Everything
# in it is copied into WORK_RESULTS so finished runs are skipped and mid-run
# last.pth checkpoints continue. Set to None for a from-scratch session.
PREV_RESULTS = None  # teammate's prior output (wenhaolu49/notebookc458262781) is
                      # private and not attachable this session -- from scratch.

os.makedirs(WORK_RESULTS, exist_ok=True)
if PREV_RESULTS:
    # Fail loudly: a wrong path here silently retrains everything from scratch.
    assert os.path.isdir(PREV_RESULTS), (
        f"PREV_RESULTS not found: {PREV_RESULTS}\n"
        "Attach the previous session's output (+ Add Input -> Your Work -> "
        "Notebook Output) and copy the exact path from the input panel.")
    import shutil
    for name in sorted(os.listdir(PREV_RESULTS)):
        src_d = os.path.join(PREV_RESULTS, name)
        dst_d = os.path.join(WORK_RESULTS, name)
        if os.path.isdir(src_d) and not os.path.exists(dst_d):
            shutil.copytree(src_d, dst_d)
    print("seeded resume state from", PREV_RESULTS)

# --- smoke override: applied last so it wins over everything above ---
if SMOKE:
    TIERS, SIZES, SEEDS, EPOCHS = [1], [28], [0], 3
    WORK_RESULTS = WORK_RESULTS + "_smoke"
    os.makedirs(WORK_RESULTS, exist_ok=True)
    print("*** SMOKE TEST - 3 epochs, seed 0, DermaMNIST @28 only. "
          "Set SMOKE=False for real runs. ***")

# --- inventory: what carried over, what is still missing ---
import json as _json, glob as _glob
_complete, _partial = [], []
for _rj in sorted(_glob.glob(os.path.join(WORK_RESULTS, "*", "run.json"))):
    _n = len(_json.load(open(_rj)).get("history", []))
    (_complete if _n >= EPOCHS else _partial).append(
        (os.path.basename(os.path.dirname(_rj)), _n))
_dirs = {os.path.basename(d) for d in _glob.glob(os.path.join(WORK_RESULTS, "*"))
         if os.path.isdir(d)}
_nojson = sorted(_dirs - {n for n, _ in _complete + _partial})

print("MODE =", MODE, "| tiers", TIERS, "| sizes", SIZES, "| seeds", SEEDS,
      "| epochs", EPOCHS)
print("results dir =", WORK_RESULTS)
print(f"carried over: {len(_complete)} complete, {len(_partial)} partial, "
      f"{len(_nojson)} started-but-unfinished")
for _n, _e in _partial: print(f"   PARTIAL  {_n}  ({_e}/{EPOCHS} epochs -> resumes)")
for _n in _nojson:      print(f"   NO JSON  {_n}  (resumes from last.pth)")


## 4. Baselines — train the tiered run matrix (resumable)

In [ ]:
import time, json
from src.run import RunConfig, run_config, run_matrix

if MODE in ("baselines", "all"):
    matrix = [m for m in run_matrix(seeds=tuple(SEEDS), octmnist_second_seed=OCTMNIST_SECOND_SEED) if m[0] in TIERS]
    matrix = [m for m in matrix if m[3] in SIZES]
    print(f"{len(matrix)} runs selected | tiers {TIERS} | sizes {SIZES} | seeds {SEEDS}")
    for t, ds, mdl, sz, sd in matrix:
        print(f"   T{t}  {ds:16s} {mdl:8s} @{sz:<4d} seed{sd}")
    t_start = time.time()
    for tier, dataset, model, size, seed in matrix:
        cfg = RunConfig(dataset=dataset, model=model, size=size, seed=seed,
                        epochs=EPOCHS, amp=USE_AMP, deterministic=DETERMINISTIC,
                        results_dir=WORK_RESULTS)
        rj = os.path.join(WORK_RESULTS, cfg.run_name(), "run.json")
        if os.path.exists(rj):
            d = json.load(open(rj))
            if len(d.get("history", [])) >= EPOCHS:
                print("skip (done):", cfg.run_name()); continue
        if (time.time() - t_start) / 60 > MAX_MINUTES:
            print("MAX_MINUTES reached; stopping before", cfg.run_name(),
                  "(re-run next session to continue)"); break
        run_config(cfg)
    print("baselines pass complete for this session.")
else:
    print("skipped (MODE is {!r}, not baselines/all)".format(MODE))


## 5. Extensions — equity study on DermaMNIST (R18 @ 28)

Reuses the Prompt-1 models / loaders / metrics / training loop unchanged (only
the sampler, loss weights, or channel widths differ) and keeps everything at
28×28. Produces, from saved artifacts:

- **Per-class breakdown** with ROC-AUC **and PR-AUC (average precision)**,
  precision/recall/F1, carrying ±std across the three DermaMNIST seeds.
- **Frequency vs performance** (Pearson/Spearman): the core equity claim.
- **Bias mitigation** — `WeightedRandomSampler` and inverse-frequency weighted
  loss, 3 seeds each; per-class + aggregate tradeoff.
- **Lightweight 0.5× variant** — params, MB, latency, rare-vs-common degradation.
- **Corruption robustness** (inference-only): Gaussian noise, JPEG, brightness.
- **Confidence & calibration** (inference-only): reliability, ECE, per-class confidence.

CSVs land under `results/extension/`; figures under `results/extension/figures/`
(PDF + 300-dpi PNG) so they survive Kaggle's session split and the report cell
copies them into `report/figures/`. Variant runs carry flags/tags so they never
pollute the baseline comparison table.

In [ ]:
import numpy as np, json, csv, os
from src.run import RunConfig, run_config
from src import extensions as ext, evaluate as ev, data as dm, models as M, metrics as mx
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def _run_dir(cfg):
    return os.path.join(WORK_RESULTS, cfg.run_name())

def _done(cfg):
    return os.path.exists(os.path.join(_run_dir(cfg), "best_model.pth"))

def _ensure(cfg):
    if not _done(cfg):
        run_config(cfg)
    return cfg

def load_model(cfg):
    info = dm.get_info(cfg.dataset); nC = len(info["label"])
    model = M.build_model(cfg.model, cfg.size, nC, width_mult=cfg.width_mult).to(DEVICE)
    ev.load_checkpoint(os.path.join(_run_dir(cfg), "best_model.pth"), model, map_location=DEVICE)
    return model

def get_preds(cfg, model=None):
    info = dm.get_info(cfg.dataset)
    if model is None:
        model = load_model(cfg)
    _, _, test_loader = dm.get_loaders(cfg.dataset, cfg.size, batch_size=128,
                                       pin_memory=(DEVICE == "cuda"))
    return ev.predict(model, test_loader, info["task"], DEVICE)

# Widths swept for the compression study. 0.5x is the headline point; 0.75x
# and 0.25x turn the single-width anecdote into a curve and show whether
# dermatofibroma degrades gradually or falls off a cliff (reviewer request).
LIGHT_WIDTHS = (0.75, 0.5, 0.25)

if MODE in ("extensions", "all"):
    EXT_OUT = os.path.join(WORK_RESULTS, "extension"); os.makedirs(EXT_OUT, exist_ok=True)
    common = dict(dataset="dermamnist", model="resnet18", size=28,
                  epochs=EPOCHS, amp=USE_AMP, results_dir=WORK_RESULTS)

    # --- ensure the runs we need (baseline seeds reuse Tier 1; variants are new) ---
    base_cfgs  = [RunConfig(**common, seed=s) for s in SEEDS]
    wsamp_cfgs = [RunConfig(**common, seed=s, weighted_sampler=True) for s in SEEDS]
    wloss_cfgs = [RunConfig(**common, seed=s, weighted_loss=True) for s in SEEDS]
    light_cfgs_by_width = {
        w: [RunConfig(**common, seed=s, width_mult=w, tag=f"light{w}") for s in SEEDS]
        for w in LIGHT_WIDTHS
    }
    for group in (base_cfgs, wsamp_cfgs, wloss_cfgs, *light_cfgs_by_width.values()):
        for c in group:
            _ensure(c)

    base_done = [c for c in base_cfgs if _done(c)]
    base_preds = [get_preds(c) for c in base_done]              # list of (yt, ys)
    yt, ys = base_preds[0]                                      # seed-0 baseline

    # (1) per-class breakdown with ROC-AUC + PR-AUC, ±std across seeds
    ms = ext.per_class_multiseed("dermamnist", base_preds)
    ms.to_csv(os.path.join(EXT_OUT, "perclass_multiseed.csv"), index=False)
    ext.per_class_analysis("dermamnist", yt, ys, EXT_OUT, tag="baseline")  # confusion + ROC + CSV
    ext.plot_pr_curves("dermamnist", yt, ys, EXT_OUT)
    ext.plot_per_class_performance("dermamnist", ms, EXT_OUT)
    ext.plot_class_distribution("dermamnist", EXT_OUT)
    print("per-class (baseline, mean over %d seeds):" % len(base_preds))
    print(ms.to_string(index=False))

    # (2) frequency vs performance
    freq = ext.frequency_performance("dermamnist", ms, EXT_OUT)
    print("\nfreq vs recall: Pearson r=%.3f  Spearman rho=%.3f  (AUC Pearson=%.3f)"
          % (freq["pearson_recall"], freq["spearman_recall"], freq["pearson_auc"]))

    # (3) bias mitigation: per-class (multiseed) + aggregate tradeoff
    wsamp_preds = [get_preds(c) for c in wsamp_cfgs if _done(c)]
    wloss_preds = [get_preds(c) for c in wloss_cfgs if _done(c)]
    variants = {"baseline": (yt, ys),
                "weighted_sampler": wsamp_preds[0] if wsamp_preds else (yt, ys),
                "weighted_loss": wloss_preds[0] if wloss_preds else (yt, ys)}
    summary, _ = ext.bias_comparison("dermamnist", variants, EXT_OUT)
    mit_tables = {"baseline": ms,
                  "weighted_sampler": ext.per_class_multiseed("dermamnist", wsamp_preds) if wsamp_preds else ms,
                  "weighted_loss": ext.per_class_multiseed("dermamnist", wloss_preds) if wloss_preds else ms}
    ext.plot_mitigation("dermamnist", mit_tables, EXT_OUT)
    print("\nbias mitigation summary (equity vs accuracy tradeoff):")
    print(summary.to_string(index=False))

    # (3b) threshold-tuned baseline control: per-class thresholds tuned on the
    # validation split (F1-maximizing), evaluated on the same test predictions
    # used everywhere else. This isolates how much of (3)'s equity gain is a
    # threshold shift vs. a change in what the model learned (reviewer request).
    _, val_loader, _ = dm.get_loaders("dermamnist", 28, batch_size=128,
                                      pin_memory=(DEVICE == "cuda"))
    base_model = load_model(base_done[0])
    yt_val, ys_val = ev.predict(base_model, val_loader, dm.get_info("dermamnist")["task"], DEVICE)
    thresholds = ext.tune_thresholds(yt_val, ys_val)
    yp_tuned = ext.apply_thresholds(ys, thresholds)
    tuned_table = ext.per_class_table_from_preds("dermamnist", yt, yp_tuned, ys)
    tuned_table.to_csv(os.path.join(EXT_OUT, "perclass_threshold_tuned.csv"), index=False)
    tuned_summary = ext.aggregate_from_table(tuned_table)
    print("\nthreshold-tuned baseline (same model, no retraining):")
    print(tuned_summary)

    # (4) lightweight variant sweep: profile, efficiency tradeoff, rare-vs-common
    # degradation for each width in LIGHT_WIDTHS (0.75x/0.5x/0.25x), not just 0.5x.
    base_tbl = ext.per_class_table("dermamnist", yt, ys)
    info = dm.get_info("dermamnist"); nC = len(info["label"])
    prof_full = ext.profile_model(M.build_model("resnet18", 28, nC, width_mult=1.0), DEVICE, size=28)
    profiles = {"resnet18_full": prof_full}
    eff_metrics = {"resnet18_full": mx.evaluate(yt, ys, info["task"])}
    light_preds_by_width, light_tbl_by_width, corr_by_width = {}, {}, {}
    for w in LIGHT_WIDTHS:
        light_done = [c for c in light_cfgs_by_width[w] if _done(c)]
        ylt, yls = get_preds(light_done[0]) if light_done else (yt, ys)
        light_preds_by_width[w] = (ylt, yls)
        light_tbl = ext.per_class_table("dermamnist", ylt, yls)
        light_tbl_by_width[w] = light_tbl
        merged, corr = ext.rare_vs_common_degradation(
            "dermamnist", base_tbl, light_tbl, EXT_OUT, stem=f"lightweight_degradation_w{w}")
        corr_by_width[w] = corr
        prof_w = ext.profile_model(M.build_model("resnet18", 28, nC, width_mult=w), DEVICE, size=28)
        name = f"resnet18_{w}x"
        profiles[name] = prof_w
        eff_metrics[name] = mx.evaluate(ylt, yls, info["task"])
    ext.plot_efficiency(profiles, eff_metrics, EXT_OUT)
    with open(os.path.join(EXT_OUT, "model_size_inference.csv"), "w", newline="") as f:
        w_ = csv.writer(f)
        w_.writerow(["variant", "n_params", "size_mb", "latency_ms_per_image", "test_auc", "test_acc",
                     "freq_vs_f1drop_corr"])
        w_.writerow(["resnet18_full", prof_full["n_params"], f"{prof_full['size_mb']:.3f}",
                     f"{prof_full['latency_ms_per_image']:.4f}",
                     f"{eff_metrics['resnet18_full'][0]:.4f}", f"{eff_metrics['resnet18_full'][1]:.4f}", ""])
        for w in LIGHT_WIDTHS:
            name = f"resnet18_{w}x"
            a, b = eff_metrics[name]
            w_.writerow([name, profiles[name]["n_params"], f"{profiles[name]['size_mb']:.3f}",
                         f"{profiles[name]['latency_ms_per_image']:.4f}", f"{a:.4f}", f"{b:.4f}",
                         f"{corr_by_width[w]:.3f}"])

    # keep the 0.5x variables the pre-sweep report text below reads from
    prof_light = profiles["resnet18_0.5x"]
    corr = corr_by_width[0.5]

    # (5) corruption robustness (inference-only) on the baseline model
    test_imgs = dm.get_dataset("dermamnist", "test", 28).imgs   # (N,28,28,3) uint8, aligned with yt
    rob = ext.robustness_eval(base_model, "dermamnist", test_imgs, yt, device=DEVICE)
    rob.to_csv(os.path.join(EXT_OUT, "robustness.csv"), index=False)
    ext.plot_robustness(rob, EXT_OUT)

    # (6) confidence & calibration (inference-only)
    ext.calibration_analysis("dermamnist", yt, ys, out_dir=EXT_OUT)  # writes calibration_ece.csv
    ext.plot_calibration("dermamnist", yt, ys, EXT_OUT)

    # misclassification gallery (rare-class errors)
    ext.plot_misclassified_gallery("dermamnist", yt, ys, test_imgs, EXT_OUT)

    # short equity writeup (unchanged content/shape from before the reviewer
    # follow-up: the sweep above only adds supporting CSVs/figure points, it
    # does not rewrite what gets said in the manuscript)
    REPORT = "/kaggle/working/report" if os.path.isdir("/kaggle/working") else "report"
    os.makedirs(REPORT, exist_ok=True)
    with open(os.path.join(REPORT, "extension.md"), "w") as f:
        f.write("# DermaMNIST equity & extension study\n\n")
        f.write("ResNet-18 @ 28, %d baseline seed(s). Kept separate from the replication's comparison.md.\n\n" % len(base_preds))
        f.write("## Frequency vs performance\n")
        f.write("- Pearson r (count vs recall) = %.3f; Spearman rho = %.3f; Pearson (count vs AUC) = %.3f.\n\n" % (
            freq["pearson_recall"], freq["spearman_recall"], freq["pearson_auc"]))
        f.write("## Bias mitigation (equity vs accuracy tradeoff)\n\n")
        try:
            f.write(summary.to_markdown(index=False) + "\n\n")
        except Exception:  # tabulate not present -> plain text fallback
            f.write("```\n" + summary.to_string(index=False) + "\n```\n\n")
        f.write("## Lightweight variant\n")
        f.write("- params %.2fM -> %.2fM; latency %.3f -> %.3f ms/img; freq~F1-drop corr = %.3f.\n" % (
            prof_full["n_params"]/1e6, prof_light["n_params"]/1e6,
            prof_full["latency_ms_per_image"], prof_light["latency_ms_per_image"], corr))
    print("\nextension artifacts ->", EXT_OUT, "| figures ->", os.path.join(EXT_OUT, "figures"))
else:
    print("skipped (MODE is {!r}, not extensions/all)".format(MODE))


## 6. Report — aggregate table + every figure for the paper

In [ ]:
if MODE in ("report", "all"):
    from src import aggregate as agg
    from src import figures_replication as figrep
    from IPython.display import Image, display
    import shutil, glob

    REPORT = "/kaggle/working/report" if os.path.isdir("/kaggle/working") else "report"
    FIGDIR = os.path.join(REPORT, "figures")
    os.makedirs(FIGDIR, exist_ok=True)

    # 1. Aggregate every baseline run -> comparison.{csv,md} vs paper Table 3.
    rows = agg.aggregate(WORK_RESULTS, REPORT)
    print(open(os.path.join(REPORT, "comparison.md")).read())

    # 2. ReScience figures, generated from results/ alone into report/figures/
    #    (PDF + 300-dpi PNG each). Never re-runs training.
    fig_pngs = figrep.generate_all(WORK_RESULTS, REPORT)

    # 3. Copy the extension figures/CSVs (from results/extension) into the report.
    EXT_OUT = os.path.join(WORK_RESULTS, "extension")
    ext_pngs = []
    if os.path.isdir(EXT_OUT):
        for p in glob.glob(os.path.join(EXT_OUT, "figures", "*.png")) + glob.glob(os.path.join(EXT_OUT, "*.png")):
            dst = os.path.join(FIGDIR, os.path.basename(p)); shutil.copy(p, dst); ext_pngs.append(dst)
        for p in glob.glob(os.path.join(EXT_OUT, "figures", "*.pdf")):
            shutil.copy(p, os.path.join(FIGDIR, os.path.basename(p)))
        for p in glob.glob(os.path.join(EXT_OUT, "*.csv")):
            shutil.copy(p, os.path.join(REPORT, os.path.basename(p)))

    print("\nfigures written to", FIGDIR)
    for p in fig_pngs + sorted(set(ext_pngs)):
        print("---", os.path.basename(p))
        display(Image(p))
else:
    print("skipped (MODE is {!r}, not report/all)".format(MODE))

## Outputs

Everything lands under `/kaggle/working/` and is saved with the notebook version:

- `results/<run_name>/` — `run.json` (config + metrics + pinned versions + seed +
  wall-clock + **peak GPU memory**), `best_model.pth`, `last.pth`, and the
  medmnist-convention predictions CSV, e.g.
  `dermamnist_test_[AUC]0.917_[ACC]0.735@seed0.csv`.
- `results/extension/` — equity CSVs (`perclass_multiseed.csv`, `bias_summary.csv`,
  `model_size_inference.csv`, `robustness.csv`, `calibration_ece.csv`,
  `lightweight_degradation.csv`) and `figures/` (PDF + 300-dpi PNG).
- `report/comparison.{csv,md}` — our mean±std vs paper Table 3 (all twelve R18@28
  datasets + the DermaMNIST four-config matrix), with signed deltas and flags.
- `report/extension.md` — equity findings + mitigation tradeoff, kept separate.
- `report/figures/` — every figure as **PDF + 300-dpi PNG**:
  - *Replication:* `fig1_reproduction` (ours-vs-paper AUC/ACC scatter + Derma 4-config bars),
    `fig2_training_curves` (LR-decay markers), `fig3_seed_variance`,
    `fig4_delta_heatmap`, `fig5_compute_footprint` (epoch time / peak GPU memory).
  - *Equity:* `ext_class_distribution`, `ext_per_class_perf`, `ext_freq_perf`,
    `confusion_baseline`, `roc_baseline`, `ext_pr_curves`, `bias_perclass_f1`,
    `ext_mitigation`, `ext_efficiency`, `ext_robustness`, `ext_calibration`,
    `ext_misclassified`.

To continue training next session, attach this notebook's output as input data
and set `PREV_RESULTS` in the CONFIG cell.